In [1]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [2]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

In [3]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [4]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


##_______________________________________________________________ "" ____________________________________________________________

##_______________________________________________________________ "" ____________________________________________________________

In [5]:
# ============================================================
# DEEP DIVE — WHATSAPP PERFORMANCE IN DPD 30-45
#
# QUESTION:
# After surviving until DPD 30-45, does another WhatsApp
# attempt still show economic value?
#
# Grain: 1 row = 1 historical WhatsApp send
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. PREPARE DATA
# ============================================================

wa_3045 = wa.copy()

wa_3045["sent_at"] = pd.to_datetime(
    wa_3045["sent_at"]
)

wa_3045["days_past_due"] = pd.to_numeric(
    wa_3045["days_past_due"],
    errors="coerce"
)

wa_3045["amount_paid_brl"] = pd.to_numeric(
    wa_3045["amount_paid_brl"],
    errors="coerce"
).fillna(0)

wa_3045["outstanding_balance_brl"] = pd.to_numeric(
    wa_3045["outstanding_balance_brl"],
    errors="coerce"
)


# ============================================================
# 2. KEEP ONLY DPD 30-45
# ============================================================

wa_3045 = (
    wa_3045.loc[
        wa_3045["days_past_due"].between(
            30,
            45
        )
    ]
    .copy()
)


# ============================================================
# 3. BASIC RESPONSE VARIABLES
# ============================================================

wa_3045["paid_72h"] = (
    wa_3045["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

wa_3045["payment_amount_72h"] = (
    wa_3045["amount_paid_brl"]
)

wa_3045["recovery_per_message"] = (
    wa_3045["payment_amount_72h"]
)


# ============================================================
# 4. HIGH-LEVEL PERFORMANCE
# ============================================================

n_messages = len(wa_3045)

n_customers = (
    wa_3045["customer_id"]
    .nunique()
)

n_payers = (
    wa_3045.loc[
        wa_3045["paid_72h"],
        "customer_id"
    ]
    .nunique()
)

total_recovery = (
    wa_3045["payment_amount_72h"]
    .sum()
)


print("=" * 110)
print("HISTORICAL WHATSAPP — DPD 30-45")
print("=" * 110)

print(
    f"Messages                     : "
    f"{n_messages:,}"
)

print(
    f"Unique customers             : "
    f"{n_customers:,}"
)

print(
    f"Customers paying within 72h  : "
    f"{n_payers:,}"
)

print(
    f"Payment events within 72h    : "
    f"{wa_3045['paid_72h'].sum():,}"
)

print(
    f"Payment response rate        : "
    f"{wa_3045['paid_72h'].mean():.2%}"
)

print(
    f"Observed recovery            : "
    f"R$ {total_recovery:,.2f}"
)

print(
    f"Recovery / message           : "
    f"R$ {total_recovery / n_messages:,.2f}"
)

print(
    f"WhatsApp cost                : "
    f"R$ {n_messages:,.2f}"
)

print(
    f"Observed recovery - WA cost  : "
    f"R$ {total_recovery - n_messages:,.2f}"
)


# ============================================================
# 5. PERFORMANCE BY EXACT DPD
# ============================================================

by_exact_dpd = (
    wa_3045
    .groupby(
        "days_past_due"
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_prior_msgs=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


by_exact_dpd["recovery_per_message"] = (
    by_exact_dpd["recovery_brl"]
    /
    by_exact_dpd["messages"]
)

by_exact_dpd["cost_brl"] = (
    by_exact_dpd["messages"]
)

by_exact_dpd["observed_net_brl"] = (
    by_exact_dpd["recovery_brl"]
    -
    by_exact_dpd["cost_brl"]
)


print("\n" + "=" * 110)
print("PERFORMANCE BY EXACT DPD")
print("=" * 110)

display(
    by_exact_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.2f}",
        "cost_brl": "R$ {:,.2f}",
        "observed_net_brl": "R$ {:,.2f}"
    })
)


# ============================================================
# 6. SUB-BUCKETS INSIDE 30-45
#
# We want to know whether the signal decays as we approach DPD45.
# ============================================================

wa_3045["dpd_subbucket"] = pd.cut(
    wa_3045["days_past_due"],
    bins=[
        29,
        34,
        39,
        45
    ],
    labels=[
        "30-34",
        "35-39",
        "40-45"
    ]
)


by_subbucket = (
    wa_3045
    .groupby(
        "dpd_subbucket",
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_prior_msgs=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


by_subbucket["recovery_per_message"] = (
    by_subbucket["recovery_brl"]
    /
    by_subbucket["messages"]
)


print("\n" + "=" * 110)
print("PERFORMANCE WITHIN DPD 30-45")
print("=" * 110)

display(
    by_subbucket.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.2f}"
    })
)


# ============================================================
# 7. PRESSURE — NUMBER OF MESSAGES IN LAST 14 DAYS
# ============================================================

wa_3045["pressure_bucket"] = pd.cut(
    wa_3045["n_msgs_last_14d"],
    bins=[
        -1,
        0,
        2,
        4,
        6,
        np.inf
    ],
    labels=[
        "0",
        "1-2",
        "3-4",
        "5-6",
        "7+"
    ]
)


by_pressure = (
    wa_3045
    .groupby(
        "pressure_bucket",
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        ),

        avg_dpd=(
            "days_past_due",
            "mean"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        )
    )
    .reset_index()
)


by_pressure["recovery_per_message"] = (
    by_pressure["recovery_brl"]
    /
    by_pressure["messages"]
)


print("\n" + "=" * 110)
print("PERFORMANCE BY CONTACT PRESSURE — DPD 30-45")
print("=" * 110)

display(
    by_pressure.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_dpd": "{:.1f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# 8. DPD × PRESSURE
#
# Critical view:
# Does timing interact with previous contact saturation?
# ============================================================

dpd_pressure = (
    wa_3045
    .groupby(
        [
            "dpd_subbucket",
            "pressure_bucket"
        ],
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        )
    )
    .reset_index()
)


dpd_pressure["recovery_per_message"] = (
    dpd_pressure["recovery_brl"]
    /
    dpd_pressure["messages"]
)


print("\n" + "=" * 110)
print("DPD × CONTACT PRESSURE")
print("=" * 110)

display(
    dpd_pressure.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}"
    })
)


# ============================================================
# 9. TEMPLATE × DPD
# ============================================================

template_dpd = (
    wa_3045
    .groupby(
        [
            "dpd_subbucket",
            "template"
        ],
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        ),

        avg_prior_msgs=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


template_dpd["recovery_per_message"] = (
    template_dpd["recovery_brl"]
    /
    template_dpd["messages"]
)


print("\n" + "=" * 110)
print("TEMPLATE × DPD — 30-45")
print("=" * 110)

display(
    template_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.2f}"
    })
)


# ============================================================
# 10. INTERACTION × DPD
# ============================================================

interaction_dpd = (
    wa_3045
    .groupby(
        [
            "dpd_subbucket",
            "interaction"
        ],
        observed=True
    )
    .agg(
        messages=(
            "customer_id",
            "size"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "payment_amount_72h",
            "sum"
        )
    )
    .reset_index()
)


interaction_dpd["recovery_per_message"] = (
    interaction_dpd["recovery_brl"]
    /
    interaction_dpd["messages"]
)


print("\n" + "=" * 110)
print("INTERACTION × DPD — 30-45")
print("=" * 110)

display(
    interaction_dpd.style.format({
        "messages": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}"
    })
)

HISTORICAL WHATSAPP — DPD 30-45
Messages                     : 8,552
Unique customers             : 4,581
Customers paying within 72h  : 598
Payment events within 72h    : 609
Payment response rate        : 7.12%
Observed recovery            : R$ 345,485.07
Recovery / message           : R$ 40.40
WhatsApp cost                : R$ 8,552.00
Observed recovery - WA cost  : R$ 336,933.07

PERFORMANCE BY EXACT DPD


,days_past_due,messages,customers,payment_events,payment_rate,recovery_brl,avg_balance,avg_prior_msgs,recovery_per_message,cost_brl,observed_net_brl
0,30,"1,078","1,078",61,5.66%,"R$ 39,609.00",R$ 797.11,2.41,R$ 36.74,"R$ 1,078.00","R$ 38,531.00"
1,31,588,588,36,6.12%,"R$ 21,477.20",R$ 796.18,2.51,R$ 36.53,R$ 588.00,"R$ 20,889.20"
2,32,595,595,52,8.74%,"R$ 30,789.73",R$ 829.87,2.46,R$ 51.75,R$ 595.00,"R$ 30,194.73"
3,33,523,523,43,8.22%,"R$ 27,044.27",R$ 817.77,2.30,R$ 51.71,R$ 523.00,"R$ 26,521.27"
4,34,527,527,42,7.97%,"R$ 24,511.32",R$ 795.04,2.22,R$ 46.51,R$ 527.00,"R$ 23,984.32"
5,35,544,544,35,6.43%,"R$ 21,536.28",R$ 813.95,2.14,R$ 39.59,R$ 544.00,"R$ 20,992.28"
6,36,514,514,30,5.84%,"R$ 16,610.96",R$ 763.01,2.02,R$ 32.32,R$ 514.00,"R$ 16,096.96"
7,37,499,499,40,8.02%,"R$ 22,908.99",R$ 776.09,1.99,R$ 45.91,R$ 499.00,"R$ 22,409.99"
8,38,500,500,39,7.80%,"R$ 21,404.58",R$ 762.66,1.85,R$ 42.81,R$ 500.00,"R$ 20,904.58"
9,39,524,524,28,5.34%,"R$ 14,602.14",R$ 785.44,1.82,R$ 27.87,R$ 524.00,"R$ 14,078.14"



PERFORMANCE WITHIN DPD 30-45


,dpd_subbucket,messages,customers,payment_events,payment_rate,recovery_brl,avg_balance,avg_prior_msgs,recovery_per_message
0,30-34,"3,311","2,705",234,7.07%,"R$ 143,431.52",R$ 805.76,2.39,R$ 43.32
1,35-39,"2,581","2,150",172,6.66%,"R$ 97,062.95",R$ 780.76,1.97,R$ 37.61
2,40-45,"2,660","2,126",203,7.63%,"R$ 104,990.60",R$ 776.00,1.55,R$ 39.47



PERFORMANCE BY CONTACT PRESSURE — DPD 30-45


...

,dpd_subbucket,pressure_bucket,messages,customers,payment_rate,recovery_brl,recovery_per_message
0,30-34,0,209,209,7.66%,"R$ 7,198.61",R$ 34.44
1,30-34,1-2,"1,650","1,469",7.94%,"R$ 78,940.35",R$ 47.84
2,30-34,3-4,"1,229","1,053",6.27%,"R$ 51,303.83",R$ 41.74
3,30-34,5-6,214,185,4.67%,"R$ 5,988.73",R$ 27.98
4,30-34,7+,9,9,0.00%,R$ 0.00,R$ 0.00
5,35-39,0,279,279,9.68%,"R$ 11,657.00",R$ 41.78
6,35-39,1-2,"1,508","1,359",6.83%,"R$ 63,743.37",R$ 42.27
7,35-39,3-4,708,615,5.51%,"R$ 19,477.55",R$ 27.51
8,35-39,5-6,85,71,3.53%,"R$ 2,185.03",R$ 25.71
9,35-39,7+,1,1,0.00%,R$ 0.00,R$ 0.00



TEMPLATE × DPD — 30-45


,dpd_subbucket,template,messages,customers,payment_events,payment_rate,recovery_brl,avg_prior_msgs,recovery_per_message
0,30-34,discount_offer,"1,094","1,030",101,9.23%,"R$ 55,740.83",2.34,R$ 50.95
1,30-34,friendly_reminder,217,217,14,6.45%,"R$ 8,613.23",2.42,R$ 39.69
2,30-34,pix_link,744,709,59,7.93%,"R$ 35,171.59",2.47,R$ 47.27
3,30-34,urgent_reminder,"1,256","1,164",60,4.78%,"R$ 43,905.87",2.38,R$ 34.96
4,35-39,discount_offer,"1,298","1,177",104,8.01%,"R$ 57,304.59",1.97,R$ 44.15
5,35-39,pix_link,523,508,31,5.93%,"R$ 19,510.02",1.97,R$ 37.30
6,35-39,urgent_reminder,760,718,37,4.87%,"R$ 20,248.34",1.97,R$ 26.64
7,40-45,discount_offer,"1,360","1,235",129,9.49%,"R$ 67,596.50",1.54,R$ 49.70
8,40-45,pix_link,531,508,38,7.16%,"R$ 16,787.83",1.60,R$ 31.62
9,40-45,urgent_reminder,769,719,36,4.68%,"R$ 20,606.27",1.55,R$ 26.80



INTERACTION × DPD — 30-45


,dpd_subbucket,interaction,messages,payment_rate,recovery_brl,recovery_per_message
0,30-34,clicked_link,365,19.45%,"R$ 46,929.02",R$ 128.57
1,30-34,none,"2,158",3.75%,"R$ 47,245.91",R$ 21.89
2,30-34,read,623,10.11%,"R$ 37,739.06",R$ 60.58
3,30-34,replied,165,11.52%,"R$ 11,517.53",R$ 69.80
4,35-39,clicked_link,293,18.09%,"R$ 30,537.47",R$ 104.22
5,35-39,none,"1,703",3.88%,"R$ 41,293.81",R$ 24.25
6,35-39,read,462,8.87%,"R$ 19,351.84",R$ 41.89
7,35-39,replied,123,9.76%,"R$ 5,879.83",R$ 47.80
8,40-45,clicked_link,302,20.86%,"R$ 33,639.06",R$ 111.39
9,40-45,none,"1,708",4.74%,"R$ 42,311.52",R$ 24.77


In [15]:
# ============================================================
# DPD 30–45 — DISCOVERY DE SINAIS
# DTI | TRANSAÇÕES | ACCOUNT AGE | APP ACTIVITY
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 0. CONSTRUIR EXPLICITAMENTE O UNIVERSO DPD 30–45
# ------------------------------------------------------------

analysis = wa.loc[
    wa["days_past_due"].between(30, 45)
].copy()

# Garantir tipos
analysis["amount_paid_brl"] = pd.to_numeric(
    analysis["amount_paid_brl"],
    errors="coerce"
).fillna(0)

analysis["outstanding_balance_brl"] = pd.to_numeric(
    analysis["outstanding_balance_brl"],
    errors="coerce"
)

analysis["monthly_salary_brl"] = pd.to_numeric(
    analysis["monthly_salary_brl"],
    errors="coerce"
)

analysis["n_prior_transactions"] = pd.to_numeric(
    analysis["n_prior_transactions"],
    errors="coerce"
)

analysis["account_age_months"] = pd.to_numeric(
    analysis["account_age_months"],
    errors="coerce"
)

analysis["days_since_last_app_login"] = pd.to_numeric(
    analysis["days_since_last_app_login"],
    errors="coerce"
)


# ------------------------------------------------------------
# 1. TARGET / RECOVERY
# ------------------------------------------------------------

analysis["payment_event_72h"] = (
    analysis["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

analysis["recovery_72h"] = np.where(
    analysis["payment_event_72h"],
    analysis["amount_paid_brl"],
    0
)


# ------------------------------------------------------------
# 2. DTI / AFFORDABILITY
# outstanding balance / monthly salary
# ------------------------------------------------------------

analysis["dti"] = (
    analysis["outstanding_balance_brl"]
    / analysis["monthly_salary_brl"]
)

analysis["dti_bucket"] = pd.cut(
    analysis["dti"],
    bins=[-np.inf, .10, .25, .50, 1.00, np.inf],
    labels=[
        "<=10%",
        "10–25%",
        "25–50%",
        "50–100%",
        ">100%"
    ]
)


# ------------------------------------------------------------
# 3. TRANSAÇÕES ANTERIORES
# ------------------------------------------------------------

analysis["prior_tx_bucket"] = pd.cut(
    analysis["n_prior_transactions"],
    bins=[-np.inf, 2, 5, 10, 20, np.inf],
    labels=[
        "0–2",
        "3–5",
        "6–10",
        "11–20",
        "20+"
    ]
)


# ------------------------------------------------------------
# 4. TEMPO DE RELACIONAMENTO / "FIDELIDADE"
# ------------------------------------------------------------

analysis["account_age_bucket"] = pd.cut(
    analysis["account_age_months"],
    bins=[-np.inf, 3, 6, 12, 24, np.inf],
    labels=[
        "<=3m",
        "4–6m",
        "7–12m",
        "13–24m",
        "24m+"
    ]
)


# ------------------------------------------------------------
# 5. ATIVIDADE RECENTE NO APP
# ------------------------------------------------------------

analysis["login_recency_bucket"] = pd.cut(
    analysis["days_since_last_app_login"],
    bins=[-np.inf, 3, 7, 14, 30, np.inf],
    labels=[
        "0–3d",
        "4–7d",
        "8–14d",
        "15–30d",
        "30d+"
    ]
)


# ------------------------------------------------------------
# 6. BASELINE DO BUCKET DPD 30–45
# ------------------------------------------------------------

BASELINE = (
    analysis["recovery_72h"].sum()
    / len(analysis)
)


# ------------------------------------------------------------
# 7. FUNÇÃO PADRÃO DE DISCOVERY
# ------------------------------------------------------------

def signal_table(df, variable):

    out = (
        df
        .groupby(variable, observed=True)
        .agg(
            messages=("customer_id", "size"),
            customers=("customer_id", "nunique"),
            payment_events=("payment_event_72h", "sum"),
            recovery_brl=("recovery_72h", "sum"),
            avg_balance=("outstanding_balance_brl", "mean")
        )
        .reset_index()
    )

    out["payment_rate"] = (
        out["payment_events"]
        / out["messages"]
    )

    out["recovery_per_message"] = (
        out["recovery_brl"]
        / out["messages"]
    )

    out["delta_vs_baseline"] = (
        out["recovery_per_message"]
        - BASELINE
    )

    out["index_vs_baseline"] = (
        out["recovery_per_message"]
        / BASELINE
        * 100
    )

    return out


# ------------------------------------------------------------
# 8. RODAR DISCOVERY
# ------------------------------------------------------------

signals = {
    "DTI / AFFORDABILITY":
        signal_table(analysis, "dti_bucket"),

    "PRIOR TRANSACTIONS":
        signal_table(analysis, "prior_tx_bucket"),

    "ACCOUNT AGE":
        signal_table(analysis, "account_age_bucket"),

    "APP LOGIN RECENCY":
        signal_table(analysis, "login_recency_bucket")
}


# ------------------------------------------------------------
# 9. OUTPUT
# ------------------------------------------------------------

print("=" * 110)
print("DPD 30–45 — DISCOVERY DE SINAIS")
print("=" * 110)

print(f"Messages                  : {len(analysis):,}")
print(f"Unique customers          : {analysis['customer_id'].nunique():,}")
print(f"Baseline recovery/message : R$ {BASELINE:,.2f}")

for name, table in signals.items():

    print("\n")
    print("=" * 110)
    print(name)
    print("=" * 110)

    print(
        table.to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# 10. QA
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("QA")
print("=" * 110)

print(f"DPD min : {analysis['days_past_due'].min():.0f}")
print(f"DPD max : {analysis['days_past_due'].max():.0f}")
print(f"Rows    : {len(analysis):,}")

assert analysis["days_past_due"].between(30, 45).all()

print("\n✓ Universo DPD 30–45 validado")

DPD 30–45 — DISCOVERY DE SINAIS
Messages                  : 8,552
Unique customers          : 4,581
Baseline recovery/message : R$ 40.40


DTI / AFFORDABILITY
dti_bucket  messages  customers  payment_events  recovery_brl  avg_balance  payment_rate  recovery_per_message  delta_vs_baseline  index_vs_baseline
     <=10%       559        316              82     12,990.81       210.81          0.15                 23.24             -17.16              57.53
    10–25%      2726       1481             216     72,353.68       472.40          0.08                 26.54             -13.86              65.70
    25–50%      4334       2314             257    208,052.84       977.23          0.06                 48.00               7.61             118.83
   50–100%       933        513              54     52,087.74     1,185.70          0.06                 55.83              15.43             138.19


PRIOR TRANSACTIONS
prior_tx_bucket  messages  customers  payment_events  recovery_brl  avg_bal

In [16]:
# ============================================================
# DPD 30–45 — TRANSAÇÕES ANTERIORES
# REGRA SIMPLIFICADA: <=5 vs >5
# ============================================================

tx_analysis = analysis.copy()


# ------------------------------------------------------------
# 1. Criar regra binária
# ------------------------------------------------------------

tx_analysis["prior_tx_group"] = np.where(
    tx_analysis["n_prior_transactions"] <= 5,
    "<=5 transactions",
    ">5 transactions"
)


# ------------------------------------------------------------
# 2. Performance por grupo
# ------------------------------------------------------------

tx_summary = (
    tx_analysis
    .groupby("prior_tx_group")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median"),
        avg_transactions=("n_prior_transactions", "mean"),
        median_transactions=("n_prior_transactions", "median")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 3. Métricas
# ------------------------------------------------------------

tx_summary["payment_rate"] = (
    tx_summary["payment_events"]
    / tx_summary["messages"]
)

tx_summary["recovery_per_message"] = (
    tx_summary["recovery_brl"]
    / tx_summary["messages"]
)

tx_summary["delta_vs_baseline"] = (
    tx_summary["recovery_per_message"]
    - BASELINE
)

tx_summary["index_vs_baseline"] = (
    tx_summary["recovery_per_message"]
    / BASELINE
    * 100
)

tx_summary["message_share"] = (
    tx_summary["messages"]
    / tx_summary["messages"].sum()
)


# ------------------------------------------------------------
# 4. OUTPUT
# ------------------------------------------------------------

print("=" * 110)
print("DPD 30–45 — PRIOR TRANSACTIONS: <=5 vs >5")
print("=" * 110)

print(f"Baseline recovery/message : R$ {BASELINE:,.2f}")

print(
    tx_summary[
        [
            "prior_tx_group",
            "messages",
            "customers",
            "message_share",
            "payment_events",
            "payment_rate",
            "recovery_brl",
            "recovery_per_message",
            "delta_vs_baseline",
            "index_vs_baseline",
            "avg_balance",
            "median_balance",
            "avg_transactions",
            "median_transactions"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 5. COMPARAÇÃO DIRETA
# ------------------------------------------------------------

low = tx_summary.loc[
    tx_summary["prior_tx_group"] == "<=5 transactions"
].iloc[0]

high = tx_summary.loc[
    tx_summary["prior_tx_group"] == ">5 transactions"
].iloc[0]


print("\n" + "=" * 110)
print(">5 vs <=5 — COMPARAÇÃO")
print("=" * 110)

print(
    f"Payment rate       : "
    f"{low['payment_rate']:.2%} → {high['payment_rate']:.2%}"
)

print(
    f"Recovery / message : "
    f"R$ {low['recovery_per_message']:,.2f} → "
    f"R$ {high['recovery_per_message']:,.2f}"
)

print(
    f"Difference R$/msg  : "
    f"R$ {high['recovery_per_message'] - low['recovery_per_message']:,.2f}"
)

print(
    f"Relative index     : "
    f"{high['recovery_per_message'] / low['recovery_per_message']:.2f}x"
)

print(
    f"Average balance    : "
    f"R$ {low['avg_balance']:,.2f} → "
    f"R$ {high['avg_balance']:,.2f}"
)

DPD 30–45 — PRIOR TRANSACTIONS: <=5 vs >5
Baseline recovery/message : R$ 40.40
  prior_tx_group  messages  customers  message_share  payment_events  payment_rate  recovery_brl  recovery_per_message  delta_vs_baseline  index_vs_baseline  avg_balance  median_balance  avg_transactions  median_transactions
<=5 transactions      4125       2200           0.48             228          0.06    130,539.19                 31.65              -8.75              78.33       804.74          725.46              2.75                 3.00
 >5 transactions      4427       2381           0.52             381          0.09    214,945.88                 48.55               8.16             120.19       774.26          665.64             14.26                11.00

>5 vs <=5 — COMPARAÇÃO
Payment rate       : 5.53% → 8.61%
Recovery / message : R$ 31.65 → R$ 48.55
Difference R$/msg  : R$ 16.91
Relative index     : 1.53x
Average balance    : R$ 804.74 → R$ 774.26


In [18]:
# ============================================================
# DPD 30–45 — PRIOR ENGAGEMENT POINT-IN-TIME
#
# Para cada mensagem:
# "O cliente já havia engajado ANTES deste envio?"
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Trabalhar sobre TODO o histórico
# Precisamos enxergar mensagens anteriores mesmo que estejam
# fora do bucket DPD 30–45
# ------------------------------------------------------------

wa_pit = wa.copy()

wa_pit["sent_at"] = pd.to_datetime(
    wa_pit["sent_at"]
)

wa_pit = (
    wa_pit
    .sort_values(
        ["customer_id", "sent_at"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. Identificar engagement NA mensagem atual
# ------------------------------------------------------------

interaction_clean = (
    wa_pit["interaction"]
    .astype("string")
    .str.strip()
    .str.lower()
)

wa_pit["engaged_current"] = (
    interaction_clean.notna()
    & interaction_clean.ne("none")
)


# ------------------------------------------------------------
# 3. Construir engagement acumulado ANTES da mensagem atual
#
# shift(1) é crítico:
# exclui a interação da própria mensagem.
# ------------------------------------------------------------

wa_pit["has_prior_engagement"] = (
    wa_pit
    .groupby("customer_id")["engaged_current"]
    .transform(
        lambda x: x.shift(1)
                   .fillna(False)
                   .cummax()
    )
    .astype(bool)
)


# ------------------------------------------------------------
# 4. Recuperar somente DPD 30–45
# ------------------------------------------------------------

analysis = wa_pit.loc[
    wa_pit["days_past_due"].between(30, 45)
].copy()


# ------------------------------------------------------------
# 5. QA
# ------------------------------------------------------------

print("=" * 100)
print("DPD 30–45 — PRIOR ENGAGEMENT PIT")
print("=" * 100)

print(
    f"Messages              : "
    f"{len(analysis):,}"
)

print(
    f"Unique customers      : "
    f"{analysis['customer_id'].nunique():,}"
)

print(
    f"Prior engagement      : "
    f"{analysis['has_prior_engagement'].sum():,}"
)

print(
    f"No prior engagement   : "
    f"{(~analysis['has_prior_engagement']).sum():,}"
)

print(
    f"% prior engagement    : "
    f"{analysis['has_prior_engagement'].mean():.2%}"
)


# ------------------------------------------------------------
# 6. SANITY CHECK
# Primeira mensagem de cada cliente NUNCA pode ter
# prior engagement
# ------------------------------------------------------------

first_msg = (
    wa_pit
    .groupby("customer_id")
    .head(1)
)

assert (
    first_msg["has_prior_engagement"] == False
).all()


# ------------------------------------------------------------
# 7. RECONCILIAR UNIVERSO
# ------------------------------------------------------------

assert len(analysis) == 8_552

assert (
    analysis["customer_id"].nunique()
    == 4_581
)

print("\n✓ Prior engagement PIT construído corretamente")
print("✓ Current-message interaction NÃO entra no sinal")
print("✓ Universo DPD 30–45 permanece 8,552 / 4,581")

DPD 30–45 — PRIOR ENGAGEMENT PIT
Messages              : 8,552
Unique customers      : 4,581
Prior engagement      : 7,459
No prior engagement   : 1,093
% prior engagement    : 87.22%

✓ Prior engagement PIT construído corretamente
✓ Current-message interaction NÃO entra no sinal
✓ Universo DPD 30–45 permanece 8,552 / 4,581


In [20]:
# ============================================================
# FIX — RECONSTRUIR TARGET / RECOVERY NO ANALYSIS PIT
# ============================================================

# ------------------------------------------------------------
# 1. Payment event em até 72h
# ------------------------------------------------------------

analysis["payment_event_72h"] = (
    analysis["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)


# ------------------------------------------------------------
# 2. Recovery em até 72h
# ------------------------------------------------------------

analysis["amount_paid_brl"] = pd.to_numeric(
    analysis["amount_paid_brl"],
    errors="coerce"
).fillna(0)

analysis["recovery_72h"] = np.where(
    analysis["payment_event_72h"],
    analysis["amount_paid_brl"],
    0
)


# ------------------------------------------------------------
# 3. Recalcular baseline
# ------------------------------------------------------------

BASELINE = (
    analysis["recovery_72h"].sum()
    / len(analysis)
)


# ------------------------------------------------------------
# 4. QA
# ------------------------------------------------------------

print("=" * 90)
print("QA — ANALYSIS DPD 30–45")
print("=" * 90)

print(f"Messages                  : {len(analysis):,}")
print(f"Unique customers          : {analysis['customer_id'].nunique():,}")
print(f"Payment events            : {analysis['payment_event_72h'].sum():,}")
print(f"Recovery                  : R$ {analysis['recovery_72h'].sum():,.2f}")
print(f"Baseline recovery/message : R$ {BASELINE:,.2f}")
print(f"Prior engagement          : {analysis['has_prior_engagement'].sum():,}")
print(f"No prior engagement       : {(~analysis['has_prior_engagement']).sum():,}")


# ------------------------------------------------------------
# 5. VALIDAÇÕES
# ------------------------------------------------------------

assert len(analysis) == 8_552
assert analysis["customer_id"].nunique() == 4_581

print("\n✓ Targets reconstruídos")
print("✓ Prior engagement PIT preservado")

QA — ANALYSIS DPD 30–45
Messages                  : 8,552
Unique customers          : 4,581
Payment events            : 609
Recovery                  : R$ 345,485.07
Baseline recovery/message : R$ 40.40
Prior engagement          : 7,459
No prior engagement       : 1,093

✓ Targets reconstruídos
✓ Prior engagement PIT preservado


In [21]:
# ============================================================
# DPD 30–45
# PRIOR ENGAGEMENT × PRIOR TRANSACTIONS
#
# Pergunta:
# >5 transações continua sendo um bom sinal mesmo entre
# clientes SEM engagement anterior no WhatsApp?
# ============================================================

cross = analysis.copy()


# ------------------------------------------------------------
# 1. GRUPO DE TRANSAÇÕES
# ------------------------------------------------------------

cross["prior_tx_group"] = np.where(
    cross["n_prior_transactions"] <= 5,
    "<=5 transactions",
    ">5 transactions"
)


# ------------------------------------------------------------
# 2. GRUPO DE ENGAGEMENT
# ------------------------------------------------------------

cross["engagement_group"] = np.where(
    cross["has_prior_engagement"],
    "Prior engagement",
    "No prior engagement"
)


# ------------------------------------------------------------
# 3. SUMÁRIO DOS 4 QUADRANTES
# ------------------------------------------------------------

cross_summary = (
    cross
    .groupby(
        ["engagement_group", "prior_tx_group"],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median"),
        avg_transactions=("n_prior_transactions", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. MÉTRICAS
# ------------------------------------------------------------

cross_summary["payment_rate"] = (
    cross_summary["payment_events"]
    / cross_summary["messages"]
)

cross_summary["recovery_per_message"] = (
    cross_summary["recovery_brl"]
    / cross_summary["messages"]
)

cross_summary["delta_vs_baseline"] = (
    cross_summary["recovery_per_message"]
    - BASELINE
)

cross_summary["index_vs_baseline"] = (
    cross_summary["recovery_per_message"]
    / BASELINE
    * 100
)

cross_summary["share_messages"] = (
    cross_summary["messages"]
    / len(cross)
)


# ------------------------------------------------------------
# 5. OUTPUT — 4 QUADRANTES
# ------------------------------------------------------------

print("=" * 125)
print("DPD 30–45 — PRIOR ENGAGEMENT × PRIOR TRANSACTIONS")
print("=" * 125)

print(f"Baseline recovery/message : R$ {BASELINE:,.2f}")
print(f"Messages                  : {len(cross):,}")
print(f"Unique customers          : {cross['customer_id'].nunique():,}")

print("\n")

print(
    cross_summary[
        [
            "engagement_group",
            "prior_tx_group",
            "messages",
            "customers",
            "share_messages",
            "payment_events",
            "payment_rate",
            "recovery_brl",
            "recovery_per_message",
            "delta_vs_baseline",
            "index_vs_baseline",
            "avg_balance",
            "median_balance",
            "avg_transactions"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 6. >5 vs <=5 DENTRO DE CADA NÍVEL DE ENGAGEMENT
# ------------------------------------------------------------

print("\n" + "=" * 125)
print(">5 vs <=5 TRANSACTIONS — DENTRO DE CADA ENGAGEMENT")
print("=" * 125)

for engagement in [
    "No prior engagement",
    "Prior engagement"
]:

    temp = cross_summary.loc[
        cross_summary["engagement_group"] == engagement
    ].copy()

    low = temp.loc[
        temp["prior_tx_group"] == "<=5 transactions"
    ]

    high = temp.loc[
        temp["prior_tx_group"] == ">5 transactions"
    ]

    if low.empty or high.empty:
        print(f"\n{engagement}: grupo incompleto")
        continue

    low = low.iloc[0]
    high = high.iloc[0]

    print(f"\n{engagement}")
    print("-" * 85)

    print(
        f"Messages             : "
        f"{low['messages']:,.0f} → "
        f"{high['messages']:,.0f}"
    )

    print(
        f"Customers            : "
        f"{low['customers']:,.0f} → "
        f"{high['customers']:,.0f}"
    )

    print(
        f"Payment rate         : "
        f"{low['payment_rate']:.2%} → "
        f"{high['payment_rate']:.2%}"
    )

    print(
        f"Recovery / message   : "
        f"R$ {low['recovery_per_message']:,.2f} → "
        f"R$ {high['recovery_per_message']:,.2f}"
    )

    print(
        f"Difference R$/msg    : "
        f"R$ {high['recovery_per_message'] - low['recovery_per_message']:,.2f}"
    )

    print(
        f"Relative R$/msg      : "
        f"{high['recovery_per_message'] / low['recovery_per_message']:.2f}x"
    )

    print(
        f"Average balance      : "
        f"R$ {low['avg_balance']:,.2f} → "
        f"R$ {high['avg_balance']:,.2f}"
    )


# ------------------------------------------------------------
# 7. QA — RECONCILIAÇÃO
# ------------------------------------------------------------

print("\n" + "=" * 125)
print("QA")
print("=" * 125)

print(
    f"Messages reconciled       : "
    f"{cross_summary['messages'].sum():,}"
)

print(
    f"Payment events reconciled : "
    f"{cross_summary['payment_events'].sum():,}"
)

print(
    f"Recovery reconciled       : "
    f"R$ {cross_summary['recovery_brl'].sum():,.2f}"
)

assert cross_summary["messages"].sum() == 8_552
assert cross_summary["payment_events"].sum() == 609

print("\n✓ 4 quadrantes reconciliados")

DPD 30–45 — PRIOR ENGAGEMENT × PRIOR TRANSACTIONS
Baseline recovery/message : R$ 40.40
Messages                  : 8,552
Unique customers          : 4,581


   engagement_group   prior_tx_group  messages  customers  share_messages  payment_events  payment_rate  recovery_brl  recovery_per_message  delta_vs_baseline  index_vs_baseline  avg_balance  median_balance  avg_transactions
No prior engagement <=5 transactions       668        378            0.08              13          0.02      6,367.12                  9.53             -30.87              23.59       787.08          695.60              2.65
No prior engagement  >5 transactions       425        239            0.05              15          0.04     10,456.95                 24.60             -15.79              60.91       792.21          664.08             12.14
   Prior engagement <=5 transactions      3457       1864            0.40             215          0.06    124,172.07                 35.92              -4.48          

In [22]:
# ============================================================
# DPD 30–45
# ENGAGEMENT × TRANSACTIONS × APP ACTIVITY
#
# Objetivo:
# descobrir se atividade recente no app adiciona informação
# aos sinais de engagement e relacionamento
# ============================================================

app_cross = analysis.copy()


# ------------------------------------------------------------
# 1. GRUPO DE TRANSAÇÕES
# ------------------------------------------------------------

app_cross["prior_tx_group"] = np.where(
    app_cross["n_prior_transactions"] <= 5,
    "<=5 tx",
    ">5 tx"
)


# ------------------------------------------------------------
# 2. GRUPO DE ENGAGEMENT
# ------------------------------------------------------------

app_cross["engagement_group"] = np.where(
    app_cross["has_prior_engagement"],
    "Engaged",
    "No engagement"
)


# ------------------------------------------------------------
# 3. RECÊNCIA DE LOGIN
# ------------------------------------------------------------

app_cross["login_group"] = pd.cut(
    app_cross["days_since_last_app_login"],
    bins=[-np.inf, 14, 30, np.inf],
    labels=[
        "<=14d",
        "15–30d",
        ">30d"
    ]
)


# ------------------------------------------------------------
# 4. SUMÁRIO
# ------------------------------------------------------------

app_summary = (
    app_cross
    .groupby(
        [
            "engagement_group",
            "prior_tx_group",
            "login_group"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median"),
        avg_login_days=("days_since_last_app_login", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. MÉTRICAS
# ------------------------------------------------------------

app_summary["payment_rate"] = (
    app_summary["payment_events"]
    / app_summary["messages"]
)

app_summary["recovery_per_message"] = (
    app_summary["recovery_brl"]
    / app_summary["messages"]
)

app_summary["delta_vs_baseline"] = (
    app_summary["recovery_per_message"]
    - BASELINE
)

app_summary["index_vs_baseline"] = (
    app_summary["recovery_per_message"]
    / BASELINE
    * 100
)


# ------------------------------------------------------------
# 6. OUTPUT COMPLETO
# ------------------------------------------------------------

print("=" * 135)
print("DPD 30–45 — ENGAGEMENT × TRANSACTIONS × APP ACTIVITY")
print("=" * 135)

print(f"Baseline recovery/message : R$ {BASELINE:,.2f}")

print(
    app_summary[
        [
            "engagement_group",
            "prior_tx_group",
            "login_group",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_brl",
            "recovery_per_message",
            "delta_vs_baseline",
            "index_vs_baseline",
            "avg_balance",
            "median_balance",
            "avg_login_days"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 7. FOCO:
# SEM ENGAGEMENT + >5 TRANSAÇÕES
# ------------------------------------------------------------

focus = app_summary.loc[
    (app_summary["engagement_group"] == "No engagement")
    & (app_summary["prior_tx_group"] == ">5 tx")
].copy()


print("\n" + "=" * 135)
print("FOCUS — NO ENGAGEMENT + >5 TRANSACTIONS")
print("=" * 135)

print(
    focus[
        [
            "login_group",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_brl",
            "recovery_per_message",
            "delta_vs_baseline",
            "index_vs_baseline",
            "avg_balance"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 8. QA
# ------------------------------------------------------------

print("\n" + "=" * 135)
print("QA")
print("=" * 135)

print(f"Messages original    : {len(app_cross):,}")
print(f"Messages summarized  : {app_summary['messages'].sum():,}")
print(f"Payment events       : {app_summary['payment_events'].sum():,}")
print(f"Recovery             : R$ {app_summary['recovery_brl'].sum():,.2f}")

assert app_summary["messages"].sum() == len(app_cross)
assert app_summary["payment_events"].sum() == 609

print("\n✓ Universo reconciliado")

DPD 30–45 — ENGAGEMENT × TRANSACTIONS × APP ACTIVITY
Baseline recovery/message : R$ 40.40
engagement_group prior_tx_group login_group  messages  customers  payment_events  payment_rate  recovery_brl  recovery_per_message  delta_vs_baseline  index_vs_baseline  avg_balance  median_balance  avg_login_days
         Engaged         <=5 tx       <=14d       762        501              67          0.09     36,315.14                 47.66               7.26             117.97       793.21          735.95            7.62
         Engaged         <=5 tx      15–30d      1159        839              66          0.06     37,961.38                 32.75              -7.64              81.08       825.34          742.47           23.26
         Engaged         <=5 tx        >30d      1536        955              82          0.05     49,895.55                 32.48              -7.91              80.41       802.59          723.80           50.33
         Engaged          >5 tx       <=14d      1140 

In [23]:
# ============================================================
# DPD 30–45
# ENGAGEMENT × TRANSACTIONS × AFFORDABILITY
# ============================================================

dti_cross = analysis.copy()


# ------------------------------------------------------------
# 1. SINAIS JÁ DESCOBERTOS
# ------------------------------------------------------------

dti_cross["prior_tx_group"] = np.where(
    dti_cross["n_prior_transactions"] <= 5,
    "<=5 tx",
    ">5 tx"
)

dti_cross["engagement_group"] = np.where(
    dti_cross["has_prior_engagement"],
    "Engaged",
    "No engagement"
)


# ------------------------------------------------------------
# 2. DTI / AFFORDABILITY
# ------------------------------------------------------------

dti_cross["dti"] = (
    dti_cross["outstanding_balance_brl"]
    / dti_cross["monthly_salary_brl"]
)

# Corte simplificado para manter suporte amostral
dti_cross["dti_group"] = pd.cut(
    dti_cross["dti"],
    bins=[-np.inf, 0.25, 0.50, np.inf],
    labels=[
        "<=25%",
        "25–50%",
        ">50%"
    ]
)


# ------------------------------------------------------------
# 3. SUMÁRIO
# ------------------------------------------------------------

dti_summary = (
    dti_cross
    .groupby(
        [
            "engagement_group",
            "prior_tx_group",
            "dti_group"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median"),
        avg_salary=("monthly_salary_brl", "mean"),
        avg_dti=("dti", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. MÉTRICAS
# ------------------------------------------------------------

dti_summary["payment_rate"] = (
    dti_summary["payment_events"]
    / dti_summary["messages"]
)

dti_summary["recovery_per_message"] = (
    dti_summary["recovery_brl"]
    / dti_summary["messages"]
)

dti_summary["delta_vs_baseline"] = (
    dti_summary["recovery_per_message"]
    - BASELINE
)

dti_summary["index_vs_baseline"] = (
    dti_summary["recovery_per_message"]
    / BASELINE
    * 100
)


# ------------------------------------------------------------
# 5. OUTPUT COMPLETO
# ------------------------------------------------------------

print("=" * 140)
print("DPD 30–45 — ENGAGEMENT × TRANSACTIONS × AFFORDABILITY")
print("=" * 140)

print(f"Baseline recovery/message : R$ {BASELINE:,.2f}")

print(
    dti_summary[
        [
            "engagement_group",
            "prior_tx_group",
            "dti_group",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_brl",
            "recovery_per_message",
            "delta_vs_baseline",
            "index_vs_baseline",
            "avg_balance",
            "avg_salary",
            "avg_dti"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 6. FOCO — ENGAGED + >5 TX
# ------------------------------------------------------------

focus_good = dti_summary.loc[
    (dti_summary["engagement_group"] == "Engaged")
    & (dti_summary["prior_tx_group"] == ">5 tx")
].copy()

print("\n" + "=" * 140)
print("FOCUS 1 — ENGAGED + >5 TRANSACTIONS")
print("=" * 140)

print(
    focus_good[
        [
            "dti_group",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "avg_balance",
            "avg_salary",
            "avg_dti"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 7. FOCO — NO ENGAGEMENT + >5 TX
# ------------------------------------------------------------

focus_rescue = dti_summary.loc[
    (dti_summary["engagement_group"] == "No engagement")
    & (dti_summary["prior_tx_group"] == ">5 tx")
].copy()

print("\n" + "=" * 140)
print("FOCUS 2 — NO ENGAGEMENT + >5 TRANSACTIONS")
print("=" * 140)

print(
    focus_rescue[
        [
            "dti_group",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "avg_balance",
            "avg_salary",
            "avg_dti"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 8. QA
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("QA")
print("=" * 140)

print(f"Messages original    : {len(dti_cross):,}")
print(f"Messages summarized  : {dti_summary['messages'].sum():,}")
print(f"Payment events       : {dti_summary['payment_events'].sum():,}")
print(f"Recovery             : R$ {dti_summary['recovery_brl'].sum():,.2f}")

assert dti_summary["messages"].sum() == len(dti_cross)
assert dti_summary["payment_events"].sum() == 609

print("\n✓ Universo reconciliado")

DPD 30–45 — ENGAGEMENT × TRANSACTIONS × AFFORDABILITY
Baseline recovery/message : R$ 40.40
engagement_group prior_tx_group dti_group  messages  customers  payment_events  payment_rate  recovery_brl  recovery_per_message  delta_vs_baseline  index_vs_baseline  avg_balance  avg_salary  avg_dti
         Engaged         <=5 tx     <=25%      1247        685              98          0.08     26,623.72                 21.35             -19.05              52.85       424.06    2,632.86     0.16
         Engaged         <=5 tx    25–50%      1804        970              93          0.05     75,948.57                 42.10               1.70             104.21       983.56    2,667.41     0.37
         Engaged         <=5 tx      >50%       406        219              24          0.06     21,599.78                 53.20              12.80             131.69     1,208.45    2,186.95     0.56
         Engaged          >5 tx     <=25%      1666        907             187          0.11     54,425.7

In [25]:
# ============================================================
# DPD 30–45 — PRIOR PAYMENT POINT-IN-TIME
#
# Para cada mensagem:
# "O cliente já havia realizado algum pagamento
#  ANTES deste envio?"
# ============================================================

wa_pit = wa_pit.sort_values(
    ["customer_id", "sent_at"]
).copy()


# ------------------------------------------------------------
# 1. PAGAMENTO NA OBSERVAÇÃO ATUAL
# ------------------------------------------------------------

wa_pit["amount_paid_brl"] = pd.to_numeric(
    wa_pit["amount_paid_brl"],
    errors="coerce"
).fillna(0)

wa_pit["payment_current"] = (
    wa_pit["amount_paid_brl"] > 0
)


# ------------------------------------------------------------
# 2. PAGAMENTO ANTERIOR — PIT
#
# shift(1):
# pagamento associado à mensagem atual NÃO entra.
# ------------------------------------------------------------

wa_pit["has_prior_payment"] = (
    wa_pit
    .groupby("customer_id")["payment_current"]
    .transform(
        lambda x:
            x.shift(1)
             .fillna(False)
             .cummax()
    )
    .astype(bool)
)


# ------------------------------------------------------------
# 3. RECUPERAR UNIVERSO DPD 30–45
# ------------------------------------------------------------

payment_analysis = wa_pit.loc[
    wa_pit["days_past_due"].between(30, 45)
].copy()


# ------------------------------------------------------------
# 4. RECONSTRUIR TARGET
# ------------------------------------------------------------

payment_analysis["payment_event_72h"] = (
    payment_analysis["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

payment_analysis["recovery_72h"] = np.where(
    payment_analysis["payment_event_72h"],
    payment_analysis["amount_paid_brl"],
    0
)


# ------------------------------------------------------------
# 5. PERFORMANCE
# ------------------------------------------------------------

prior_payment_summary = (
    payment_analysis
    .groupby("has_prior_payment")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median")
    )
    .reset_index()
)

prior_payment_summary["payment_rate"] = (
    prior_payment_summary["payment_events"]
    / prior_payment_summary["messages"]
)

prior_payment_summary["recovery_per_message"] = (
    prior_payment_summary["recovery_brl"]
    / prior_payment_summary["messages"]
)

prior_payment_summary["delta_vs_baseline"] = (
    prior_payment_summary["recovery_per_message"]
    - BASELINE
)

prior_payment_summary["index_vs_baseline"] = (
    prior_payment_summary["recovery_per_message"]
    / BASELINE
    * 100
)


# ------------------------------------------------------------
# 6. OUTPUT
# ------------------------------------------------------------

print("=" * 110)
print("DPD 30–45 — PRIOR PAYMENT PIT")
print("=" * 110)

print(f"Baseline recovery/message : R$ {BASELINE:,.2f}")

print(
    prior_payment_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 7. QA
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("QA")
print("=" * 110)

print(
    f"Messages             : "
    f"{len(payment_analysis):,}"
)

print(
    f"Prior payment        : "
    f"{payment_analysis['has_prior_payment'].sum():,}"
)

print(
    f"No prior payment     : "
    f"{(~payment_analysis['has_prior_payment']).sum():,}"
)

print(
    f"% prior payment      : "
    f"{payment_analysis['has_prior_payment'].mean():.2%}"
)

print(
    f"Payment events       : "
    f"{payment_analysis['payment_event_72h'].sum():,}"
)

print(
    f"Recovery             : "
    f"R$ {payment_analysis['recovery_72h'].sum():,.2f}"
)

assert len(payment_analysis) == 8_552
assert payment_analysis["payment_event_72h"].sum() == 609

print("\n✓ Prior payment PIT construído corretamente")

DPD 30–45 — PRIOR PAYMENT PIT
Baseline recovery/message : R$ 40.40
 has_prior_payment  messages  customers  payment_events  recovery_brl  avg_balance  median_balance  payment_rate  recovery_per_message  delta_vs_baseline  index_vs_baseline
             False      7236       3903             433    298,868.03       861.32          771.51          0.06                 41.30               0.90             102.24
              True      1316        721             176     46,617.04       391.07          310.41          0.13                 35.42              -4.97              87.69

QA
Messages             : 8,552
Prior payment        : 1,316
No prior payment     : 7,236
% prior payment      : 15.39%
Payment events       : 609
Recovery             : R$ 345,485.07

✓ Prior payment PIT construído corretamente


In [26]:
# ============================================================
# DPD 30–45
# ENGAGEMENT × TRANSACTIONS × PRIOR PAYMENT
# ============================================================

cross_payment = payment_analysis.copy()


# ------------------------------------------------------------
# 1. TRANSACTIONS
# ------------------------------------------------------------

cross_payment["prior_tx_group"] = np.where(
    cross_payment["n_prior_transactions"] <= 5,
    "<=5 tx",
    ">5 tx"
)


# ------------------------------------------------------------
# 2. ENGAGEMENT
# ------------------------------------------------------------

cross_payment["engagement_group"] = np.where(
    cross_payment["has_prior_engagement"],
    "Engaged",
    "No engagement"
)


# ------------------------------------------------------------
# 3. PRIOR PAYMENT
# ------------------------------------------------------------

cross_payment["prior_payment_group"] = np.where(
    cross_payment["has_prior_payment"],
    "Prior payment",
    "No prior payment"
)


# ------------------------------------------------------------
# 4. SUMÁRIO
# ------------------------------------------------------------

payment_cross_summary = (
    cross_payment
    .groupby(
        [
            "engagement_group",
            "prior_tx_group",
            "prior_payment_group"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. MÉTRICAS
# ------------------------------------------------------------

payment_cross_summary["payment_rate"] = (
    payment_cross_summary["payment_events"]
    / payment_cross_summary["messages"]
)

payment_cross_summary["recovery_per_message"] = (
    payment_cross_summary["recovery_brl"]
    / payment_cross_summary["messages"]
)

payment_cross_summary["delta_vs_baseline"] = (
    payment_cross_summary["recovery_per_message"]
    - BASELINE
)

payment_cross_summary["index_vs_baseline"] = (
    payment_cross_summary["recovery_per_message"]
    / BASELINE
    * 100
)


# ------------------------------------------------------------
# 6. OUTPUT COMPLETO
# ------------------------------------------------------------

print("=" * 140)
print("DPD 30–45 — ENGAGEMENT × TRANSACTIONS × PRIOR PAYMENT")
print("=" * 140)

print(f"Baseline recovery/message : R$ {BASELINE:,.2f}")

print(
    payment_cross_summary[
        [
            "engagement_group",
            "prior_tx_group",
            "prior_payment_group",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_brl",
            "recovery_per_message",
            "delta_vs_baseline",
            "index_vs_baseline",
            "avg_balance",
            "median_balance"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 7. COMPARAÇÃO PRIOR PAYMENT vs NO PRIOR PAYMENT
# DENTRO DE CADA QUADRANTE
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("PRIOR PAYMENT vs NO PRIOR PAYMENT — DENTRO DE CADA SEGMENTO")
print("=" * 140)

for engagement in ["No engagement", "Engaged"]:

    for tx in ["<=5 tx", ">5 tx"]:

        temp = payment_cross_summary.loc[
            (payment_cross_summary["engagement_group"] == engagement)
            & (payment_cross_summary["prior_tx_group"] == tx)
        ]

        no_pay = temp.loc[
            temp["prior_payment_group"] == "No prior payment"
        ]

        prior_pay = temp.loc[
            temp["prior_payment_group"] == "Prior payment"
        ]

        if no_pay.empty or prior_pay.empty:
            continue

        no_pay = no_pay.iloc[0]
        prior_pay = prior_pay.iloc[0]

        print(f"\n{engagement} | {tx}")
        print("-" * 90)

        print(
            f"Messages             : "
            f"{no_pay['messages']:,.0f} → "
            f"{prior_pay['messages']:,.0f}"
        )

        print(
            f"Payment rate         : "
            f"{no_pay['payment_rate']:.2%} → "
            f"{prior_pay['payment_rate']:.2%}"
        )

        print(
            f"Recovery / message   : "
            f"R$ {no_pay['recovery_per_message']:,.2f} → "
            f"R$ {prior_pay['recovery_per_message']:,.2f}"
        )

        print(
            f"Average balance      : "
            f"R$ {no_pay['avg_balance']:,.2f} → "
            f"R$ {prior_pay['avg_balance']:,.2f}"
        )


# ------------------------------------------------------------
# 8. QA
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("QA")
print("=" * 140)

print(
    f"Messages reconciled       : "
    f"{payment_cross_summary['messages'].sum():,}"
)

print(
    f"Payment events reconciled : "
    f"{payment_cross_summary['payment_events'].sum():,}"
)

print(
    f"Recovery reconciled       : "
    f"R$ {payment_cross_summary['recovery_brl'].sum():,.2f}"
)

assert payment_cross_summary["messages"].sum() == 8_552
assert payment_cross_summary["payment_events"].sum() == 609

print("\n✓ Universo reconciliado")

DPD 30–45 — ENGAGEMENT × TRANSACTIONS × PRIOR PAYMENT
Baseline recovery/message : R$ 40.40
engagement_group prior_tx_group prior_payment_group  messages  customers  payment_events  payment_rate  recovery_brl  recovery_per_message  delta_vs_baseline  index_vs_baseline  avg_balance  median_balance
         Engaged         <=5 tx    No prior payment      3029       1633             163          0.05    111,769.39                 36.90              -3.50              91.34       870.09          791.15
         Engaged         <=5 tx       Prior payment       428        242              52          0.12     12,402.68                 28.98             -11.42              71.73       369.75          295.42
         Engaged          >5 tx    No prior payment      3158       1730             246          0.08    171,554.07                 54.32              13.93             134.47       870.80          768.32
         Engaged          >5 tx       Prior payment       844        460             

In [28]:
# ============================================================
# DPD 30–45
# ENGAGEMENT × TRANSACTIONS × CONTACT PRESSURE
#
# Objetivo:
# entender quando uma nova mensagem começa a perder valor
# dado o perfil comportamental do cliente
# ============================================================

pressure = analysis.copy()


# ------------------------------------------------------------
# 1. SEGMENTOS JÁ IDENTIFICADOS
# ------------------------------------------------------------

pressure["prior_tx_group"] = np.where(
    pressure["n_prior_transactions"] <= 5,
    "<=5 tx",
    ">5 tx"
)

pressure["engagement_group"] = np.where(
    pressure["has_prior_engagement"],
    "Engaged",
    "No engagement"
)


# ------------------------------------------------------------
# 2. PRESSÃO DE CONTATO
# ------------------------------------------------------------

pressure["pressure_group"] = pd.cut(
    pressure["n_msgs_last_14d"],
    bins=[-1, 0, 2, 4, np.inf],
    labels=[
        "0 contacts",
        "1–2 contacts",
        "3–4 contacts",
        "5+ contacts"
    ]
)


# ------------------------------------------------------------
# 3. SUMÁRIO
# ------------------------------------------------------------

pressure_summary = (
    pressure
    .groupby(
        [
            "engagement_group",
            "prior_tx_group",
            "pressure_group"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median"),
        avg_pressure=("n_msgs_last_14d", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. MÉTRICAS
# ------------------------------------------------------------

pressure_summary["payment_rate"] = (
    pressure_summary["payment_events"]
    / pressure_summary["messages"]
)

pressure_summary["recovery_per_message"] = (
    pressure_summary["recovery_brl"]
    / pressure_summary["messages"]
)

pressure_summary["delta_vs_baseline"] = (
    pressure_summary["recovery_per_message"]
    - BASELINE
)

pressure_summary["index_vs_baseline"] = (
    pressure_summary["recovery_per_message"]
    / BASELINE * 100
)


# ------------------------------------------------------------
# 5. OUTPUT
# ------------------------------------------------------------

print("=" * 140)
print("DPD 30–45 — ENGAGEMENT × TRANSACTIONS × CONTACT PRESSURE")
print("=" * 140)

print(f"Baseline recovery/message : R$ {BASELINE:,.2f}")

print(
    pressure_summary[
        [
            "engagement_group",
            "prior_tx_group",
            "pressure_group",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_brl",
            "recovery_per_message",
            "delta_vs_baseline",
            "index_vs_baseline",
            "avg_balance",
            "median_balance",
            "avg_pressure"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 6. FOCO — ENGAGED + >5 TX
# ------------------------------------------------------------

focus = pressure_summary.loc[
    (pressure_summary["engagement_group"] == "Engaged")
    & (pressure_summary["prior_tx_group"] == ">5 tx")
].copy()

print("\n" + "=" * 140)
print("FOCUS — ENGAGED + >5 TRANSACTIONS")
print("=" * 140)

print(
    focus[
        [
            "pressure_group",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "avg_balance",
            "avg_pressure"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 7. QA
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("QA")
print("=" * 140)

print(f"Messages original    : {len(pressure):,}")
print(f"Messages summarized  : {pressure_summary['messages'].sum():,}")
print(f"Payment events       : {pressure_summary['payment_events'].sum():,}")
print(f"Recovery             : R$ {pressure_summary['recovery_brl'].sum():,.2f}")

assert pressure_summary["messages"].sum() == 8_552
assert pressure_summary["payment_events"].sum() == 609

print("\n✓ Universo reconciliado")

DPD 30–45 — ENGAGEMENT × TRANSACTIONS × CONTACT PRESSURE
Baseline recovery/message : R$ 40.40
engagement_group prior_tx_group pressure_group  messages  customers  payment_events  payment_rate  recovery_brl  recovery_per_message  delta_vs_baseline  index_vs_baseline  avg_balance  median_balance  avg_pressure
         Engaged         <=5 tx     0 contacts       382        382              26          0.07     12,556.48                 32.87              -7.53              81.37       842.79          790.89          0.00
         Engaged         <=5 tx   1–2 contacts      1930       1277             130          0.07     75,461.92                 39.10              -1.30              96.79       798.38          708.52          1.52
         Engaged         <=5 tx   3–4 contacts       982        636              53          0.05     31,305.82                 31.88              -8.52              78.91       806.96          739.12          3.34
         Engaged         <=5 tx    5+ contacts

In [29]:
# ============================================================
# DPD 30–45
# WHO IS WORTH ONE MORE MESSAGE?
#
# ECONOMIC OUTCOME:
# Recovery / message
#
# LIFT:
# Observed lift vs historical DPD30–45 baseline
# ============================================================

econ = analysis.copy()

BASELINE = (
    econ["recovery_72h"].sum()
    / len(econ)
)


# ------------------------------------------------------------
# 1. SEGMENTOS JÁ IDENTIFICADOS
# ------------------------------------------------------------

econ["tx_group"] = np.where(
    econ["n_prior_transactions"] > 5,
    ">5 tx",
    "<=5 tx"
)

econ["engagement_group"] = np.where(
    econ["has_prior_engagement"],
    "Engaged",
    "No engagement"
)


# ------------------------------------------------------------
# 2. CONTACT PRESSURE
# ------------------------------------------------------------

econ["pressure_group"] = pd.cut(
    econ["n_msgs_last_14d"],
    bins=[-1, 0, 2, 4, np.inf],
    labels=[
        "0 contacts",
        "1–2 contacts",
        "3–4 contacts",
        "5+ contacts"
    ]
)


# ------------------------------------------------------------
# 3. PERFORMANCE
# ------------------------------------------------------------

summary = (
    econ
    .groupby(
        [
            "engagement_group",
            "tx_group",
            "pressure_group"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median")
    )
    .reset_index()
)

summary["payment_rate"] = (
    summary["payment_events"]
    / summary["messages"]
)

summary["recovery_per_message"] = (
    summary["recovery_brl"]
    / summary["messages"]
)

summary["lift_brl"] = (
    summary["recovery_per_message"]
    - BASELINE
)

summary["lift_pct"] = (
    summary["recovery_per_message"]
    / BASELINE - 1
)


# ------------------------------------------------------------
# 4. RANKING ECONÔMICO OBSERVACIONAL
# ------------------------------------------------------------

summary = summary.sort_values(
    "recovery_per_message",
    ascending=False
).reset_index(drop=True)


print("=" * 135)
print("DPD 30–45 — WHO IS WORTH ONE MORE MESSAGE?")
print("=" * 135)

print(f"Historical baseline : R$ {BASELINE:,.2f} / message")

print(
    summary[
        [
            "engagement_group",
            "tx_group",
            "pressure_group",
            "messages",
            "customers",
            "payment_rate",
            "recovery_per_message",
            "lift_brl",
            "lift_pct",
            "avg_balance",
            "median_balance"
        ]
    ].to_string(
        index=False,
        formatters={
            "payment_rate": "{:.2%}".format,
            "lift_pct": "{:+.1%}".format
        }
    )
)


# ------------------------------------------------------------
# 5. SOMENTE ENGAGED + >5 TX
# ------------------------------------------------------------

best_base = summary.loc[
    (summary["engagement_group"] == "Engaged")
    & (summary["tx_group"] == ">5 tx")
].copy()

print("\n" + "=" * 135)
print("FOCUS — ENGAGED + >5 TX")
print("=" * 135)

print(
    best_base[
        [
            "pressure_group",
            "messages",
            "customers",
            "payment_rate",
            "recovery_per_message",
            "lift_brl",
            "lift_pct",
            "avg_balance"
        ]
    ].to_string(
        index=False,
        formatters={
            "payment_rate": "{:.2%}".format,
            "lift_pct": "{:+.1%}".format
        }
    )
)

DPD 30–45 — WHO IS WORTH ONE MORE MESSAGE?
Historical baseline : R$ 40.40 / message
engagement_group tx_group pressure_group  messages  customers payment_rate  recovery_per_message  lift_brl lift_pct  avg_balance  median_balance
   No engagement    >5 tx     0 contacts        67         67       10.45%                 59.81     19.41   +48.0%       751.76          604.63
         Engaged    >5 tx   3–4 contacts      1150        746        8.43%                 53.87     13.47   +33.3%       795.09          686.89
         Engaged    >5 tx   1–2 contacts      2238       1458        9.43%                 51.35     10.96   +27.1%       764.03          665.88
         Engaged    >5 tx     0 contacts       457        457       10.72%                 48.85      8.45   +20.9%       746.45          632.68
         Engaged   <=5 tx   1–2 contacts      1930       1277        6.74%                 39.10     -1.30    -3.2%       798.38          708.52
         Engaged    >5 tx    5+ contacts      

In [31]:
# ============================================================
# DPD 30–45
# TEMPLATE ANALYSIS — ENGAGED CUSTOMERS ONLY
#
# Pergunta:
# dado que decidimos enviar para clientes com prior engagement,
# qual template esteve historicamente associado ao maior
# recovery / message?
#
# OBSERVACIONAL — NÃO CAUSAL
# ============================================================

template_analysis = analysis.loc[
    analysis["has_prior_engagement"]
].copy()


# ------------------------------------------------------------
# 1. BASELINE — ENGAGED ONLY
# ------------------------------------------------------------

ENGAGED_BASELINE = (
    template_analysis["recovery_72h"].sum()
    / len(template_analysis)
)


# ------------------------------------------------------------
# 2. PERFORMANCE POR TEMPLATE
# ------------------------------------------------------------

template_summary = (
    template_analysis
    .groupby(
        "template",
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median"),
        avg_dpd=("days_past_due", "mean"),
        avg_pressure=("n_msgs_last_14d", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 3. MÉTRICAS
# ------------------------------------------------------------

template_summary["payment_rate"] = (
    template_summary["payment_events"]
    / template_summary["messages"]
)

template_summary["recovery_per_message"] = (
    template_summary["recovery_brl"]
    / template_summary["messages"]
)

template_summary["lift_brl"] = (
    template_summary["recovery_per_message"]
    - ENGAGED_BASELINE
)

template_summary["lift_pct"] = (
    template_summary["recovery_per_message"]
    / ENGAGED_BASELINE
    - 1
)


# ------------------------------------------------------------
# 4. RANKING OBSERVACIONAL
# ------------------------------------------------------------

template_summary = (
    template_summary
    .sort_values(
        "recovery_per_message",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. OUTPUT
# ------------------------------------------------------------

print("=" * 130)
print("DPD 30–45 — TEMPLATE | PRIOR ENGAGEMENT ONLY")
print("=" * 130)

print(
    f"Engaged messages          : "
    f"{len(template_analysis):,}"
)

print(
    f"Engaged customers         : "
    f"{template_analysis['customer_id'].nunique():,}"
)

print(
    f"Engaged baseline R$/msg   : "
    f"R$ {ENGAGED_BASELINE:,.2f}"
)

print("\n")

print(
    template_summary[
        [
            "template",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_brl",
            "recovery_per_message",
            "lift_brl",
            "lift_pct",
            "avg_balance",
            "median_balance",
            "avg_dpd",
            "avg_pressure"
        ]
    ].to_string(
        index=False,
        formatters={
            "payment_rate": "{:.2%}".format,
            "lift_pct": "{:+.1%}".format
        }
    )
)


# ------------------------------------------------------------
# 6. QA
# ------------------------------------------------------------

print("\n" + "=" * 130)
print("QA")
print("=" * 130)

print(
    f"Messages reconciled       : "
    f"{template_summary['messages'].sum():,}"
)

print(
    f"Payment events reconciled : "
    f"{template_summary['payment_events'].sum():,}"
)

print(
    f"Recovery reconciled       : "
    f"R$ {template_summary['recovery_brl'].sum():,.2f}"
)

assert (
    template_summary["messages"].sum()
    == len(template_analysis)
)

print("\n✓ Engaged population reconciled")

DPD 30–45 — TEMPLATE | PRIOR ENGAGEMENT ONLY
Engaged messages          : 7,459
Engaged customers         : 4,026
Engaged baseline R$/msg   : R$ 44.06


         template  messages  customers  payment_events payment_rate  recovery_brl  recovery_per_message  lift_brl lift_pct  avg_balance  median_balance  avg_dpd  avg_pressure
   discount_offer      3290       2469             321        9.76%    172,348.53                 52.39      8.32   +18.9%       801.89          709.66    37.70          1.94
friendly_reminder       196        196              14        7.14%      8,613.23                 43.95     -0.12    -0.3%       764.86          680.18    30.00          2.50
         pix_link      1564       1387             121        7.74%     68,317.14                 43.68     -0.38    -0.9%       771.62          672.86    36.34          2.08
  urgent_reminder      2409       1947             125        5.19%     79,382.10                 32.95    -11.11   -25.2%       784.46          695

In [32]:
# ============================================================
# DPD 30–45 — WHEN TO SEND DISCOUNT?
#
# Population:
#   Prior engagement = True
#   Template = discount_offer
#
# Pergunta:
# em qual DPD uma mensagem de discount esteve historicamente
# associada ao maior retorno econômico?
#
# OBSERVACIONAL — NÃO CAUSAL
# ============================================================

timing = analysis.loc[
    (analysis["has_prior_engagement"])
    & (analysis["template"] == "discount_offer")
].copy()


# ------------------------------------------------------------
# 1. PERFORMANCE POR DPD EXATO
# ------------------------------------------------------------

timing_dpd = (
    timing
    .groupby(
        "days_past_due",
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median"),
        avg_pressure=("n_msgs_last_14d", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 2. MÉTRICAS
# ------------------------------------------------------------

timing_dpd["payment_rate"] = (
    timing_dpd["payment_events"]
    / timing_dpd["messages"]
)

timing_dpd["recovery_per_message"] = (
    timing_dpd["recovery_brl"]
    / timing_dpd["messages"]
)

timing_dpd["lift_vs_engaged_baseline"] = (
    timing_dpd["recovery_per_message"]
    / ENGAGED_BASELINE
    - 1
)


# ------------------------------------------------------------
# 3. OUTPUT
# ------------------------------------------------------------

print("=" * 125)
print("DPD 30–45 — DISCOUNT TIMING | ENGAGED ONLY")
print("=" * 125)

print(
    f"Engaged baseline       : "
    f"R$ {ENGAGED_BASELINE:,.2f}/msg"
)

print(
    f"Discount observations  : "
    f"{len(timing):,}"
)

print("\n")

print(
    timing_dpd[
        [
            "days_past_due",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "lift_vs_engaged_baseline",
            "avg_balance",
            "median_balance",
            "avg_pressure"
        ]
    ].to_string(
        index=False,
        formatters={
            "payment_rate": "{:.2%}".format,
            "lift_vs_engaged_baseline": "{:+.1%}".format
        }
    )
)


# ------------------------------------------------------------
# 4. BUCKETS MAIS ESTÁVEIS
# ------------------------------------------------------------

timing["dpd_timing_bucket"] = pd.cut(
    timing["days_past_due"],
    bins=[29, 34, 39, 45],
    labels=[
        "30–34",
        "35–39",
        "40–45"
    ]
)

timing_bucket = (
    timing
    .groupby(
        "dpd_timing_bucket",
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        avg_pressure=("n_msgs_last_14d", "mean")
    )
    .reset_index()
)

timing_bucket["payment_rate"] = (
    timing_bucket["payment_events"]
    / timing_bucket["messages"]
)

timing_bucket["recovery_per_message"] = (
    timing_bucket["recovery_brl"]
    / timing_bucket["messages"]
)

timing_bucket["lift_vs_engaged_baseline"] = (
    timing_bucket["recovery_per_message"]
    / ENGAGED_BASELINE
    - 1
)


print("\n" + "=" * 125)
print("STABLE VIEW — DPD BUCKETS")
print("=" * 125)

print(
    timing_bucket[
        [
            "dpd_timing_bucket",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "lift_vs_engaged_baseline",
            "avg_balance",
            "avg_pressure"
        ]
    ].to_string(
        index=False,
        formatters={
            "payment_rate": "{:.2%}".format,
            "lift_vs_engaged_baseline": "{:+.1%}".format
        }
    )
)

DPD 30–45 — DISCOUNT TIMING | ENGAGED ONLY
Engaged baseline       : R$ 44.06/msg
Discount observations  : 3,290


 days_past_due  messages  customers  payment_events payment_rate  recovery_per_message lift_vs_engaged_baseline  avg_balance  median_balance  avg_pressure
            31       251        251              20        7.97%                 51.89                   +17.8%       851.41          739.38          2.58
            32       233        233              28       12.02%                 70.97                   +61.1%       835.30          781.51          2.51
            33       220        220              27       12.27%                 58.43                   +32.6%       831.92          763.73          2.30
            34       238        238              24       10.08%                 53.71                   +21.9%       800.70          683.17          2.08
            35       237        237              20        8.44%                 49.02                   +11.2%

In [36]:
# ============================================================
# DPD 30–45 — HISTORICAL POLICY TABLE
#
# R1:
#   Prior engagement
#   -> SEND
#
# R2:
#   No prior engagement
#   + >5 prior transactions
#   + 0–2 contacts in last 14d
#   -> SEND
#
# REST:
#   -> NO SEND
#
# IMPORTANT:
# has_prior_engagement must be PIT / lagged
# ============================================================

policy = wa_pit.loc[
    wa_pit["days_past_due"].between(30, 45)
].copy()


# ------------------------------------------------------------
# 1. TARGETS
# ------------------------------------------------------------

policy["payment_event_72h"] = (
    policy["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

policy["amount_paid_brl"] = pd.to_numeric(
    policy["amount_paid_brl"],
    errors="coerce"
).fillna(0)

policy["recovery_72h"] = np.where(
    policy["payment_event_72h"],
    policy["amount_paid_brl"],
    0
)


# ------------------------------------------------------------
# 2. DEFINE RULES
# ------------------------------------------------------------

mask_r1 = (
    policy["has_prior_engagement"]
)

mask_r2 = (
    ~policy["has_prior_engagement"]
    & policy["n_prior_transactions"].gt(5)
    & policy["n_msgs_last_14d"].between(0, 2)
)

mask_rest = ~(mask_r1 | mask_r2)


# ------------------------------------------------------------
# 3. ASSIGN POLICY
# ------------------------------------------------------------

policy["rule"] = "REST_NO_SEND"

policy.loc[
    mask_r1,
    "rule"
] = "R1_ENGAGED"

policy.loc[
    mask_r2,
    "rule"
] = "R2_NO_ENGAGEMENT_GT5TX_0_2_CONTACTS"


policy["decision"] = np.where(
    policy["rule"].eq("REST_NO_SEND"),
    "NO SEND",
    "SEND"
)


# ------------------------------------------------------------
# 4. HISTORICAL BASELINE
# ------------------------------------------------------------

baseline_rpm = (
    policy["recovery_72h"].sum()
    / len(policy)
)


# ------------------------------------------------------------
# 5. RULE TABLE
# ------------------------------------------------------------

rule_table = (
    policy
    .groupby(
        ["rule", "decision"],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        avg_dpd=("days_past_due", "mean"),
        avg_transactions=("n_prior_transactions", "mean"),
        avg_contacts_14d=("n_msgs_last_14d", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 6. ECONOMIC METRICS
# ------------------------------------------------------------

rule_table["payment_rate"] = (
    rule_table["payment_events"]
    / rule_table["messages"]
)

rule_table["recovery_per_message"] = (
    rule_table["recovery_brl"]
    / rule_table["messages"]
)

rule_table["delta_vs_baseline"] = (
    rule_table["recovery_per_message"]
    - baseline_rpm
)

rule_table["lift_vs_baseline"] = (
    rule_table["recovery_per_message"]
    / baseline_rpm
    - 1
)


# ------------------------------------------------------------
# 7. FORMAT OUTPUT
# ------------------------------------------------------------

out = rule_table.copy()

out["payment_rate"] = out["payment_rate"].map(
    lambda x: f"{x:.2%}"
)

out["recovery_brl"] = out["recovery_brl"].map(
    lambda x: f"R$ {x:,.2f}"
)

out["recovery_per_message"] = out[
    "recovery_per_message"
].map(
    lambda x: f"R$ {x:,.2f}"
)

out["delta_vs_baseline"] = out[
    "delta_vs_baseline"
].map(
    lambda x: f"R$ {x:+,.2f}"
)

out["lift_vs_baseline"] = out[
    "lift_vs_baseline"
].map(
    lambda x: f"{x:+.1%}"
)

out["avg_balance"] = out["avg_balance"].map(
    lambda x: f"R$ {x:,.2f}"
)


# ------------------------------------------------------------
# 8. PRINT
# ------------------------------------------------------------

print("=" * 130)
print("DPD 30–45 — HISTORICAL POLICY VALIDATION")
print("=" * 130)

print(
    f"Historical baseline : "
    f"R$ {baseline_rpm:,.2f}/msg"
)

print(
    f"Messages            : "
    f"{len(policy):,}"
)

print(
    f"Customers           : "
    f"{policy['customer_id'].nunique():,}"
)

print()

print(
    out[
        [
            "rule",
            "decision",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "delta_vs_baseline",
            "lift_vs_baseline",
            "avg_balance",
            "avg_dpd",
            "avg_transactions",
            "avg_contacts_14d"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 9. QA
# ------------------------------------------------------------

print("\n" + "=" * 130)
print("QA")
print("=" * 130)

print(f"R1   : {mask_r1.sum():,}")
print(f"R2   : {mask_r2.sum():,}")
print(f"REST : {mask_rest.sum():,}")
print(f"TOTAL: {len(policy):,}")

assert not (mask_r1 & mask_r2).any()

assert (
    mask_r1.sum()
    + mask_r2.sum()
    + mask_rest.sum()
    == len(policy)
)

print("\n✓ R1 / R2 / REST are mutually exclusive")
print("✓ 100% of historical DPD30–45 observations classified")

DPD 30–45 — HISTORICAL POLICY VALIDATION
Historical baseline : R$ 40.40/msg
Messages            : 8,552
Customers           : 4,581

                               rule decision  messages  customers  payment_events payment_rate recovery_per_message delta_vs_baseline lift_vs_baseline avg_balance  avg_dpd  avg_transactions  avg_contacts_14d
                         R1_ENGAGED     SEND      7459       4026             581        7.79%             R$ 44.06          R$ +3.66            +9.1%   R$ 788.94    36.65              9.06              2.02
R2_NO_ENGAGEMENT_GT5TX_0_2_CONTACTS     SEND       298        198              15        5.03%             R$ 35.09          R$ -5.31           -13.1%   R$ 766.92    37.12             11.93              1.19
                       REST_NO_SEND  NO SEND       795        455              13        1.64%              R$ 8.01         R$ -32.39           -80.2%   R$ 797.38    36.13              4.25              2.12

QA
R1   : 7,459
R2   : 298
REST : 

In [37]:
# ============================================================
# DPD 30–45 — POLICY × MESSAGE TEMPLATE
#
# Open R1 / R2 / REST by historical template
# ============================================================

policy_template = (
    policy
    .groupby(
        ["rule", "decision", "template"],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median"),
        avg_dpd=("days_past_due", "mean"),
        avg_transactions=("n_prior_transactions", "mean"),
        avg_contacts_14d=("n_msgs_last_14d", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# ECONOMIC METRICS
# ------------------------------------------------------------

policy_template["payment_rate"] = (
    policy_template["payment_events"]
    / policy_template["messages"]
)

policy_template["recovery_per_message"] = (
    policy_template["recovery_brl"]
    / policy_template["messages"]
)

policy_template["delta_vs_baseline"] = (
    policy_template["recovery_per_message"]
    - baseline_rpm
)

policy_template["lift_vs_baseline"] = (
    policy_template["recovery_per_message"]
    / baseline_rpm
    - 1
)


# ------------------------------------------------------------
# SHARE WITHIN EACH RULE
# ------------------------------------------------------------

policy_template["message_share_within_rule"] = (
    policy_template["messages"]
    / policy_template.groupby("rule")["messages"].transform("sum")
)


# ------------------------------------------------------------
# FORMAT
# ------------------------------------------------------------

out = policy_template.copy()

out["payment_rate"] = out["payment_rate"].map(
    lambda x: f"{x:.2%}"
)

out["recovery_brl"] = out["recovery_brl"].map(
    lambda x: f"R$ {x:,.2f}"
)

out["recovery_per_message"] = out[
    "recovery_per_message"
].map(
    lambda x: f"R$ {x:,.2f}"
)

out["delta_vs_baseline"] = out[
    "delta_vs_baseline"
].map(
    lambda x: f"R$ {x:+,.2f}"
)

out["lift_vs_baseline"] = out[
    "lift_vs_baseline"
].map(
    lambda x: f"{x:+.1%}"
)

out["message_share_within_rule"] = out[
    "message_share_within_rule"
].map(
    lambda x: f"{x:.1%}"
)

out["avg_balance"] = out["avg_balance"].map(
    lambda x: f"R$ {x:,.2f}"
)


# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------

print("=" * 155)
print("DPD 30–45 — POLICY × MESSAGE TEMPLATE")
print("=" * 155)

print(
    f"Historical DPD30–45 baseline : "
    f"R$ {baseline_rpm:,.2f}/msg"
)

print()

print(
    out[
        [
            "rule",
            "decision",
            "template",
            "messages",
            "customers",
            "message_share_within_rule",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "delta_vs_baseline",
            "lift_vs_baseline",
            "avg_balance",
            "avg_dpd",
            "avg_contacts_14d"
        ]
    ]
    .sort_values(
        ["rule", "recovery_per_message"],
        ascending=[True, False]
    )
    .to_string(index=False)
)


# ------------------------------------------------------------
# QA
# ------------------------------------------------------------

print("\n" + "=" * 155)
print("QA")
print("=" * 155)

qa = (
    policy_template
    .groupby("rule")["messages"]
    .sum()
)

print(qa.to_string())

print(
    f"\nTotal classified messages : "
    f"{policy_template['messages'].sum():,}"
)

assert policy_template["messages"].sum() == len(policy)

print("\n✓ All 8,552 messages reconciled")

DPD 30–45 — POLICY × MESSAGE TEMPLATE
Historical DPD30–45 baseline : R$ 40.40/msg

                               rule decision          template  messages  customers message_share_within_rule  payment_events payment_rate recovery_per_message delta_vs_baseline lift_vs_baseline avg_balance  avg_dpd  avg_contacts_14d
                         R1_ENGAGED     SEND    discount_offer      3290       2469                     44.1%             321        9.76%             R$ 52.39         R$ +11.99           +29.7%   R$ 801.89    37.70              1.94
                         R1_ENGAGED     SEND friendly_reminder       196        196                      2.6%              14        7.14%             R$ 43.95          R$ +3.55            +8.8%   R$ 764.86    30.00              2.50
                         R1_ENGAGED     SEND          pix_link      1564       1387                     21.0%             121        7.74%             R$ 43.68          R$ +3.28            +8.1%   R$ 771.62    36.34

In [38]:
# ============================================================
# DPD 30–60 — UNIFIED POLICY VALIDATION
#
# Question:
# Can the same SEND / NO-SEND logic be used across DPD30–60?
#
# Dimensions:
#   1. Prior engagement (PIT)
#   2. Contact pressure in last 14d
#
# Outcome:
#   Recovery within 72h / message
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. START FROM FULL HISTORICAL WA
# ------------------------------------------------------------

unified = wa_pit.loc[
    wa_pit["days_past_due"].between(30, 60)
].copy()


# ------------------------------------------------------------
# 2. REBUILD OUTCOMES
# ------------------------------------------------------------

unified["payment_event_72h"] = (
    unified["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

unified["amount_paid_brl"] = pd.to_numeric(
    unified["amount_paid_brl"],
    errors="coerce"
).fillna(0)

unified["recovery_72h"] = np.where(
    unified["payment_event_72h"],
    unified["amount_paid_brl"],
    0
)


# ------------------------------------------------------------
# 3. DPD REGION
#
# Keep the two original regions visible.
# This lets us test whether the rule generalizes.
# ------------------------------------------------------------

unified["dpd_region"] = pd.cut(
    unified["days_past_due"],
    bins=[29, 45, 60],
    labels=["DPD30–45", "DPD46–60"]
)


# ------------------------------------------------------------
# 4. CONTACT PRESSURE
# ------------------------------------------------------------

unified["pressure_bucket"] = pd.cut(
    unified["n_msgs_last_14d"],
    bins=[-1, 0, 2, 4, np.inf],
    labels=[
        "0 contacts",
        "1–2 contacts",
        "3–4 contacts",
        "5+ contacts"
    ]
)


unified["engagement_group"] = np.where(
    unified["has_prior_engagement"],
    "Engaged",
    "No engagement"
)


# ------------------------------------------------------------
# 5. OVERALL BASELINES BY DPD REGION
# ------------------------------------------------------------

baseline_region = (
    unified
    .groupby(
        "dpd_region",
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum")
    )
    .reset_index()
)

baseline_region["payment_rate"] = (
    baseline_region["payment_events"]
    / baseline_region["messages"]
)

baseline_region["recovery_per_message"] = (
    baseline_region["recovery_brl"]
    / baseline_region["messages"]
)


# ------------------------------------------------------------
# 6. ENGAGEMENT × PRESSURE × DPD REGION
# ------------------------------------------------------------

validation = (
    unified
    .groupby(
        [
            "dpd_region",
            "engagement_group",
            "pressure_bucket"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        avg_dpd=("days_past_due", "mean")
    )
    .reset_index()
)


validation["payment_rate"] = (
    validation["payment_events"]
    / validation["messages"]
)

validation["recovery_per_message"] = (
    validation["recovery_brl"]
    / validation["messages"]
)


# ------------------------------------------------------------
# 7. ADD REGION BASELINE
# ------------------------------------------------------------

validation = validation.merge(
    baseline_region[
        [
            "dpd_region",
            "recovery_per_message"
        ]
    ].rename(
        columns={
            "recovery_per_message":
            "region_baseline_rpm"
        }
    ),
    on="dpd_region",
    how="left"
)


validation["delta_vs_region_baseline"] = (
    validation["recovery_per_message"]
    - validation["region_baseline_rpm"]
)

validation["lift_vs_region_baseline"] = (
    validation["recovery_per_message"]
    / validation["region_baseline_rpm"]
    - 1
)


# ------------------------------------------------------------
# 8. PRINT BASELINES
# ------------------------------------------------------------

print("=" * 135)
print("DPD 30–60 — BASELINES BY REGION")
print("=" * 135)

base_out = baseline_region.copy()

base_out["payment_rate"] = (
    base_out["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

base_out["recovery_per_message"] = (
    base_out["recovery_per_message"]
    .map(lambda x: f"R$ {x:,.2f}")
)

base_out["recovery_brl"] = (
    base_out["recovery_brl"]
    .map(lambda x: f"R$ {x:,.2f}")
)

print(
    base_out.to_string(index=False)
)


# ------------------------------------------------------------
# 9. FORMAT VALIDATION TABLE
# ------------------------------------------------------------

out = validation.copy()

out["payment_rate"] = (
    out["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

out["recovery_per_message"] = (
    out["recovery_per_message"]
    .map(lambda x: f"R$ {x:,.2f}")
)

out["region_baseline_rpm"] = (
    out["region_baseline_rpm"]
    .map(lambda x: f"R$ {x:,.2f}")
)

out["delta_vs_region_baseline"] = (
    out["delta_vs_region_baseline"]
    .map(lambda x: f"R$ {x:+,.2f}")
)

out["lift_vs_region_baseline"] = (
    out["lift_vs_region_baseline"]
    .map(lambda x: f"{x:+.1%}")
)

out["avg_balance"] = (
    out["avg_balance"]
    .map(lambda x: f"R$ {x:,.2f}")
)


# ------------------------------------------------------------
# 10. PRINT
# ------------------------------------------------------------

print("\n" + "=" * 135)
print("ENGAGEMENT × CONTACT PRESSURE × DPD REGION")
print("=" * 135)

print(
    out[
        [
            "dpd_region",
            "engagement_group",
            "pressure_bucket",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "region_baseline_rpm",
            "delta_vs_region_baseline",
            "lift_vs_region_baseline",
            "avg_balance"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 11. QA
# ------------------------------------------------------------

print("\n" + "=" * 135)
print("QA")
print("=" * 135)

print(
    f"DPD30–60 messages  : "
    f"{len(unified):,}"
)

print(
    f"DPD30–60 customers : "
    f"{unified['customer_id'].nunique():,}"
)

print(
    f"Classified messages: "
    f"{validation['messages'].sum():,}"
)

assert validation["messages"].sum() == len(unified)

print("\n✓ All DPD30–60 observations classified")

DPD 30–60 — BASELINES BY REGION
dpd_region  messages  customers  payment_events  recovery_brl payment_rate recovery_per_message
  DPD30–45      8552       4581             609 R$ 345,485.07        7.12%             R$ 40.40
  DPD46–60      4963       2868             229 R$ 125,253.31        4.61%             R$ 25.24

ENGAGEMENT × CONTACT PRESSURE × DPD REGION
dpd_region engagement_group pressure_bucket  messages  customers  payment_events payment_rate recovery_per_message region_baseline_rpm delta_vs_region_baseline lift_vs_region_baseline avg_balance
  DPD30–45          Engaged      0 contacts       839        839              75        8.94%             R$ 41.57            R$ 40.40                 R$ +1.17                   +2.9%   R$ 790.32
  DPD30–45          Engaged    1–2 contacts      4168       2735             341        8.18%             R$ 45.68            R$ 40.40                 R$ +5.28                  +13.1%   R$ 779.94
  DPD30–45          Engaged    3–4 contacts     

In [39]:
# ============================================================
# DPD 30–60 — FINAL UNIFIED RULES
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. FINAL RULE ASSIGNMENT
# ------------------------------------------------------------

final_policy = unified.copy()

final_policy["final_rule"] = pd.NA
final_policy["final_decision"] = pd.NA


# R1 — ENGAGED + 0–2 CONTACTS
mask_r1 = (
    final_policy["has_prior_engagement"]
    & final_policy["n_msgs_last_14d"].between(0, 2)
)

final_policy.loc[mask_r1, "final_rule"] = (
    "R1_ENGAGED_0_2"
)
final_policy.loc[mask_r1, "final_decision"] = (
    "SEND"
)


# R2 — ENGAGED + 3–4 CONTACTS
mask_r2 = (
    final_policy["has_prior_engagement"]
    & final_policy["n_msgs_last_14d"].between(3, 4)
)

final_policy.loc[mask_r2, "final_rule"] = (
    "R2_ENGAGED_3_4"
)
final_policy.loc[mask_r2, "final_decision"] = (
    "SEND_SELECTIVE"
)


# R3 — ENGAGED + 5+ CONTACTS
mask_r3 = (
    final_policy["has_prior_engagement"]
    & final_policy["n_msgs_last_14d"].ge(5)
)

final_policy.loc[mask_r3, "final_rule"] = (
    "R3_ENGAGED_5_PLUS"
)
final_policy.loc[mask_r3, "final_decision"] = (
    "SUPPRESS"
)


# R4 — NO ENGAGEMENT + 0 CONTACTS
mask_r4 = (
    ~final_policy["has_prior_engagement"]
    & final_policy["n_msgs_last_14d"].eq(0)
)

final_policy.loc[mask_r4, "final_rule"] = (
    "R4_NO_ENGAGEMENT_0"
)
final_policy.loc[mask_r4, "final_decision"] = (
    "EXPLORE_TEST"
)


# R5 — NO ENGAGEMENT + 1+ CONTACTS
mask_r5 = (
    ~final_policy["has_prior_engagement"]
    & final_policy["n_msgs_last_14d"].ge(1)
)

final_policy.loc[mask_r5, "final_rule"] = (
    "R5_NO_ENGAGEMENT_1_PLUS"
)
final_policy.loc[mask_r5, "final_decision"] = (
    "NO_SEND"
)


# ------------------------------------------------------------
# 2. QA — EVERY OBSERVATION MUST HAVE ONE RULE
# ------------------------------------------------------------

assert final_policy["final_rule"].notna().all()
assert final_policy["final_decision"].notna().all()

assert (
    mask_r1.astype(int)
    + mask_r2.astype(int)
    + mask_r3.astype(int)
    + mask_r4.astype(int)
    + mask_r5.astype(int)
).eq(1).all()


# ------------------------------------------------------------
# 3. OVERALL DPD30–60 BASELINE
# ------------------------------------------------------------

overall_baseline_rpm = (
    final_policy["recovery_72h"].sum()
    / len(final_policy)
)

overall_payment_rate = (
    final_policy["payment_event_72h"].sum()
    / len(final_policy)
)


# ============================================================
# SUMMARY 1 — FINAL RULE PERFORMANCE
# ============================================================

rule_summary = (
    final_policy
    .groupby(
        ["final_rule", "final_decision"],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        avg_dpd=("days_past_due", "mean"),
        avg_contacts_14d=("n_msgs_last_14d", "mean")
    )
    .reset_index()
)

rule_summary["payment_rate"] = (
    rule_summary["payment_events"]
    / rule_summary["messages"]
)

rule_summary["recovery_per_message"] = (
    rule_summary["recovery_brl"]
    / rule_summary["messages"]
)

rule_summary["delta_vs_overall_baseline"] = (
    rule_summary["recovery_per_message"]
    - overall_baseline_rpm
)

rule_summary["lift_vs_overall_baseline"] = (
    rule_summary["recovery_per_message"]
    / overall_baseline_rpm
    - 1
)


# ============================================================
# SUMMARY 2 — RULE × DPD REGION
# ============================================================

rule_by_dpd = (
    final_policy
    .groupby(
        [
            "final_rule",
            "final_decision",
            "dpd_region"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        avg_contacts_14d=("n_msgs_last_14d", "mean")
    )
    .reset_index()
)

rule_by_dpd["payment_rate"] = (
    rule_by_dpd["payment_events"]
    / rule_by_dpd["messages"]
)

rule_by_dpd["recovery_per_message"] = (
    rule_by_dpd["recovery_brl"]
    / rule_by_dpd["messages"]
)


# ------------------------------------------------------------
# REGION BASELINES
# ------------------------------------------------------------

region_baselines = (
    final_policy
    .groupby(
        "dpd_region",
        observed=True
    )
    .agg(
        region_messages=("customer_id", "size"),
        region_recovery=("recovery_72h", "sum")
    )
    .reset_index()
)

region_baselines["region_baseline_rpm"] = (
    region_baselines["region_recovery"]
    / region_baselines["region_messages"]
)

rule_by_dpd = rule_by_dpd.merge(
    region_baselines[
        [
            "dpd_region",
            "region_baseline_rpm"
        ]
    ],
    on="dpd_region",
    how="left"
)

rule_by_dpd["delta_vs_region_baseline"] = (
    rule_by_dpd["recovery_per_message"]
    - rule_by_dpd["region_baseline_rpm"]
)

rule_by_dpd["lift_vs_region_baseline"] = (
    rule_by_dpd["recovery_per_message"]
    / rule_by_dpd["region_baseline_rpm"]
    - 1
)


# ============================================================
# SUMMARY 3 — DECISION LEVEL
# ============================================================

decision_summary = (
    final_policy
    .groupby(
        "final_decision",
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

decision_summary["payment_rate"] = (
    decision_summary["payment_events"]
    / decision_summary["messages"]
)

decision_summary["recovery_per_message"] = (
    decision_summary["recovery_brl"]
    / decision_summary["messages"]
)

decision_summary["message_share"] = (
    decision_summary["messages"]
    / len(final_policy)
)


# ============================================================
# 4. PRINT — OVERALL
# ============================================================

print("=" * 125)
print("DPD 30–60 — FINAL UNIFIED POLICY")
print("=" * 125)

print(
    f"Messages             : "
    f"{len(final_policy):,}"
)

print(
    f"Customers            : "
    f"{final_policy['customer_id'].nunique():,}"
)

print(
    f"Historical recovery  : "
    f"R$ {final_policy['recovery_72h'].sum():,.2f}"
)

print(
    f"Payment rate         : "
    f"{overall_payment_rate:.2%}"
)

print(
    f"Baseline R$/msg      : "
    f"R$ {overall_baseline_rpm:,.2f}"
)


# ============================================================
# 5. PRINT — FINAL RULES
# ============================================================

display_rules = rule_summary.copy()

display_rules["payment_rate"] = (
    display_rules["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

display_rules["recovery_per_message"] = (
    display_rules["recovery_per_message"]
    .map(lambda x: f"R$ {x:,.2f}")
)

display_rules["delta_vs_overall_baseline"] = (
    display_rules["delta_vs_overall_baseline"]
    .map(lambda x: f"R$ {x:+,.2f}")
)

display_rules["lift_vs_overall_baseline"] = (
    display_rules["lift_vs_overall_baseline"]
    .map(lambda x: f"{x:+.1%}")
)

display_rules["recovery_brl"] = (
    display_rules["recovery_brl"]
    .map(lambda x: f"R$ {x:,.2f}")
)

display_rules["avg_balance"] = (
    display_rules["avg_balance"]
    .map(lambda x: f"R$ {x:,.2f}")
)


print("\n" + "=" * 125)
print("SUMMARY 1 — FINAL RULE PERFORMANCE")
print("=" * 125)

print(
    display_rules[
        [
            "final_rule",
            "final_decision",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "delta_vs_overall_baseline",
            "lift_vs_overall_baseline",
            "avg_balance",
            "avg_dpd",
            "avg_contacts_14d"
        ]
    ].to_string(index=False)
)


# ============================================================
# 6. PRINT — RULE × DPD
# ============================================================

display_dpd = rule_by_dpd.copy()

display_dpd["payment_rate"] = (
    display_dpd["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

display_dpd["recovery_per_message"] = (
    display_dpd["recovery_per_message"]
    .map(lambda x: f"R$ {x:,.2f}")
)

display_dpd["region_baseline_rpm"] = (
    display_dpd["region_baseline_rpm"]
    .map(lambda x: f"R$ {x:,.2f}")
)

display_dpd["delta_vs_region_baseline"] = (
    display_dpd["delta_vs_region_baseline"]
    .map(lambda x: f"R$ {x:+,.2f}")
)

display_dpd["lift_vs_region_baseline"] = (
    display_dpd["lift_vs_region_baseline"]
    .map(lambda x: f"{x:+.1%}")
)


print("\n" + "=" * 125)
print("SUMMARY 2 — FINAL RULE × DPD REGION")
print("=" * 125)

print(
    display_dpd[
        [
            "final_rule",
            "dpd_region",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "region_baseline_rpm",
            "delta_vs_region_baseline",
            "lift_vs_region_baseline"
        ]
    ].to_string(index=False)
)


# ============================================================
# 7. PRINT — DECISION LEVEL
# ============================================================

display_decision = decision_summary.copy()

display_decision["payment_rate"] = (
    display_decision["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

display_decision["recovery_per_message"] = (
    display_decision["recovery_per_message"]
    .map(lambda x: f"R$ {x:,.2f}")
)

display_decision["recovery_brl"] = (
    display_decision["recovery_brl"]
    .map(lambda x: f"R$ {x:,.2f}")
)

display_decision["message_share"] = (
    display_decision["message_share"]
    .map(lambda x: f"{x:.1%}")
)


print("\n" + "=" * 125)
print("SUMMARY 3 — DECISION LEVEL")
print("=" * 125)

print(
    display_decision[
        [
            "final_decision",
            "messages",
            "customers",
            "message_share",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "recovery_brl"
        ]
    ].to_string(index=False)
)


# ============================================================
# 8. QA
# ============================================================

print("\n" + "=" * 125)
print("QA")
print("=" * 125)

print(f"R1 : {mask_r1.sum():,}")
print(f"R2 : {mask_r2.sum():,}")
print(f"R3 : {mask_r3.sum():,}")
print(f"R4 : {mask_r4.sum():,}")
print(f"R5 : {mask_r5.sum():,}")
print("-" * 40)

print(
    f"TOTAL: "
    f"{sum([
        mask_r1.sum(),
        mask_r2.sum(),
        mask_r3.sum(),
        mask_r4.sum(),
        mask_r5.sum()
    ]):,}"
)

assert (
    rule_summary["messages"].sum()
    == len(final_policy)
)

assert (
    decision_summary["messages"].sum()
    == len(final_policy)
)

print("\n✓ 100% of DPD30–60 observations classified")
print("✓ Rules are mutually exclusive")
print("✓ Rule summary reconciles")
print("✓ Decision summary reconciles")

DPD 30–60 — FINAL UNIFIED POLICY
Messages             : 13,515
Customers            : 5,072
Historical recovery  : R$ 470,738.38
Payment rate         : 6.20%
Baseline R$/msg      : R$ 34.83

SUMMARY 1 — FINAL RULE PERFORMANCE
             final_rule final_decision  messages  customers  payment_events payment_rate recovery_per_message delta_vs_overall_baseline lift_vs_overall_baseline avg_balance  avg_dpd  avg_contacts_14d
         R1_ENGAGED_0_2           SEND      8794       4065             620        7.05%             R$ 38.14                  R$ +3.31                    +9.5%   R$ 784.55    43.85              1.16
         R2_ENGAGED_3_4 SEND_SELECTIVE      2715       1627             170        6.26%             R$ 38.79                  R$ +3.95                   +11.4%   R$ 798.25    38.89              3.30
      R3_ENGAGED_5_PLUS       SUPPRESS       352        245              17        4.83%             R$ 31.51                  R$ -3.32                    -9.5%   R$ 816.22  

In [40]:
# ============================================================
# DPD 30–60 — FINAL POLICY TABLE
# ENGAGEMENT × CONTACTS LAST 14D
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. BASE
# ------------------------------------------------------------

final_2var = wa_pit.loc[
    wa_pit["days_past_due"].between(30, 60)
].copy()


# ------------------------------------------------------------
# 2. OUTCOMES
# ------------------------------------------------------------

final_2var["payment_event_72h"] = (
    final_2var["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

final_2var["amount_paid_brl"] = pd.to_numeric(
    final_2var["amount_paid_brl"],
    errors="coerce"
).fillna(0)

final_2var["recovery_72h"] = np.where(
    final_2var["payment_event_72h"],
    final_2var["amount_paid_brl"],
    0
)


# ------------------------------------------------------------
# 3. DPD REGION
# ------------------------------------------------------------

final_2var["dpd_region"] = pd.cut(
    final_2var["days_past_due"],
    bins=[29, 45, 60],
    labels=[
        "DPD30–45",
        "DPD46–60"
    ]
)


# ------------------------------------------------------------
# 4. ENGAGEMENT
# ------------------------------------------------------------

final_2var["engagement_group"] = np.where(
    final_2var["has_prior_engagement"],
    "Engaged",
    "No engagement"
)


# ------------------------------------------------------------
# 5. CONTACT PRESSURE — LAST 14 DAYS
# ------------------------------------------------------------

final_2var["pressure_bucket"] = pd.cut(
    final_2var["n_msgs_last_14d"],
    bins=[-1, 0, 2, 4, np.inf],
    labels=[
        "0 contacts",
        "1–2 contacts",
        "3–4 contacts",
        "5+ contacts"
    ]
)


# ------------------------------------------------------------
# 6. REGION BASELINES
# ------------------------------------------------------------

region_baseline = (
    final_2var
    .groupby(
        "dpd_region",
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum")
    )
    .reset_index()
)

region_baseline["payment_rate"] = (
    region_baseline["payment_events"]
    / region_baseline["messages"]
)

region_baseline["baseline_rpm"] = (
    region_baseline["recovery_brl"]
    / region_baseline["messages"]
)


# ------------------------------------------------------------
# 7. FINAL TABLE
# ------------------------------------------------------------

final_table = (
    final_2var
    .groupby(
        [
            "dpd_region",
            "engagement_group",
            "pressure_bucket"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        avg_dpd=("days_past_due", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 8. METRICS
# ------------------------------------------------------------

final_table["payment_rate"] = (
    final_table["payment_events"]
    / final_table["messages"]
)

final_table["recovery_per_message"] = (
    final_table["recovery_brl"]
    / final_table["messages"]
)


# ------------------------------------------------------------
# 9. ADD CORRECT DPD BASELINE
# ------------------------------------------------------------

final_table = final_table.merge(
    region_baseline[
        [
            "dpd_region",
            "baseline_rpm"
        ]
    ],
    on="dpd_region",
    how="left"
)

final_table["delta_vs_baseline"] = (
    final_table["recovery_per_message"]
    - final_table["baseline_rpm"]
)

final_table["lift_vs_baseline"] = (
    final_table["recovery_per_message"]
    / final_table["baseline_rpm"]
    - 1
)


# ------------------------------------------------------------
# 10. FORMAT
# ------------------------------------------------------------

out = final_table.copy()

out["payment_rate"] = (
    out["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

out["recovery_brl"] = (
    out["recovery_brl"]
    .map(lambda x: f"R$ {x:,.2f}")
)

out["recovery_per_message"] = (
    out["recovery_per_message"]
    .map(lambda x: f"R$ {x:,.2f}")
)

out["baseline_rpm"] = (
    out["baseline_rpm"]
    .map(lambda x: f"R$ {x:,.2f}")
)

out["delta_vs_baseline"] = (
    out["delta_vs_baseline"]
    .map(lambda x: f"R$ {x:+,.2f}")
)

out["lift_vs_baseline"] = (
    out["lift_vs_baseline"]
    .map(lambda x: f"{x:+.1%}")
)

out["avg_balance"] = (
    out["avg_balance"]
    .map(lambda x: f"R$ {x:,.2f}")
)


# ------------------------------------------------------------
# 11. PRINT BASELINES
# ------------------------------------------------------------

print("=" * 145)
print("DPD 30–60 — BASELINES")
print("=" * 145)

baseline_out = region_baseline.copy()

baseline_out["payment_rate"] = (
    baseline_out["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

baseline_out["baseline_rpm"] = (
    baseline_out["baseline_rpm"]
    .map(lambda x: f"R$ {x:,.2f}")
)

baseline_out["recovery_brl"] = (
    baseline_out["recovery_brl"]
    .map(lambda x: f"R$ {x:,.2f}")
)

print(
    baseline_out[
        [
            "dpd_region",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "baseline_rpm",
            "recovery_brl"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 12. PRINT FINAL TABLE
# ------------------------------------------------------------

print("\n" + "=" * 145)
print("FINAL TABLE — ENGAGEMENT × CONTACTS LAST 14D")
print("=" * 145)

print(
    out[
        [
            "dpd_region",
            "engagement_group",
            "pressure_bucket",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "baseline_rpm",
            "delta_vs_baseline",
            "lift_vs_baseline",
            "avg_balance"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 13. QA
# ------------------------------------------------------------

print("\n" + "=" * 145)
print("QA")
print("=" * 145)

print(
    f"Messages classified : "
    f"{final_table['messages'].sum():,}"
)

print(
    f"Expected            : "
    f"{len(final_2var):,}"
)

assert final_table["messages"].sum() == len(final_2var)

print("\n✓ 100% of DPD30–60 messages classified")

DPD 30–60 — BASELINES
dpd_region  messages  customers  payment_events payment_rate baseline_rpm  recovery_brl
  DPD30–45      8552       4581             609        7.12%     R$ 40.40 R$ 345,485.07
  DPD46–60      4963       2868             229        4.61%     R$ 25.24 R$ 125,253.31

FINAL TABLE — ENGAGEMENT × CONTACTS LAST 14D
dpd_region engagement_group pressure_bucket  messages  customers  payment_events payment_rate recovery_per_message baseline_rpm delta_vs_baseline lift_vs_baseline avg_balance
  DPD30–45          Engaged      0 contacts       839        839              75        8.94%             R$ 41.57     R$ 40.40          R$ +1.17            +2.9%   R$ 790.32
  DPD30–45          Engaged    1–2 contacts      4168       2735             341        8.18%             R$ 45.68     R$ 40.40          R$ +5.28           +13.1%   R$ 779.94
  DPD30–45          Engaged    3–4 contacts      2132       1382             150        7.04%             R$ 43.74     R$ 40.40          R$ +3.

In [41]:
# ============================================================
# DPD 30–60 — ENGAGEMENT × CONTACTS × TEMPLATE
# ============================================================

template_table = (
    final_2var
    .groupby(
        [
            "dpd_region",
            "engagement_group",
            "pressure_bucket",
            "template"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean"),
        avg_dpd=("days_past_due", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

template_table["payment_rate"] = (
    template_table["payment_events"]
    / template_table["messages"]
)

template_table["recovery_per_message"] = (
    template_table["recovery_brl"]
    / template_table["messages"]
)


# ------------------------------------------------------------
# ADD REGION BASELINE
# ------------------------------------------------------------

template_table = template_table.merge(
    region_baseline[
        [
            "dpd_region",
            "baseline_rpm"
        ]
    ],
    on="dpd_region",
    how="left"
)

template_table["delta_vs_baseline"] = (
    template_table["recovery_per_message"]
    - template_table["baseline_rpm"]
)

template_table["lift_vs_baseline"] = (
    template_table["recovery_per_message"]
    / template_table["baseline_rpm"]
    - 1
)


# ------------------------------------------------------------
# SHARE OF TEMPLATE WITHIN EACH CELL
# ------------------------------------------------------------

template_table["template_share"] = (
    template_table["messages"]
    / template_table
        .groupby(
            [
                "dpd_region",
                "engagement_group",
                "pressure_bucket"
            ]
        )["messages"]
        .transform("sum")
)


# ------------------------------------------------------------
# FORMAT
# ------------------------------------------------------------

template_out = template_table.copy()

template_out["payment_rate"] = (
    template_out["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

template_out["recovery_per_message"] = (
    template_out["recovery_per_message"]
    .map(lambda x: f"R$ {x:,.2f}")
)

template_out["baseline_rpm"] = (
    template_out["baseline_rpm"]
    .map(lambda x: f"R$ {x:,.2f}")
)

template_out["delta_vs_baseline"] = (
    template_out["delta_vs_baseline"]
    .map(lambda x: f"R$ {x:+,.2f}")
)

template_out["lift_vs_baseline"] = (
    template_out["lift_vs_baseline"]
    .map(lambda x: f"{x:+.1%}")
)

template_out["template_share"] = (
    template_out["template_share"]
    .map(lambda x: f"{x:.1%}")
)

template_out["avg_balance"] = (
    template_out["avg_balance"]
    .map(lambda x: f"R$ {x:,.2f}")
)


# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------

print("=" * 160)
print("DPD 30–60 — ENGAGEMENT × CONTACT PRESSURE × TEMPLATE")
print("=" * 160)

print(
    template_out[
        [
            "dpd_region",
            "engagement_group",
            "pressure_bucket",
            "template",
            "messages",
            "customers",
            "template_share",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "baseline_rpm",
            "delta_vs_baseline",
            "lift_vs_baseline",
            "avg_balance"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# QA
# ------------------------------------------------------------

print("\n" + "=" * 160)
print("QA")
print("=" * 160)

print(
    f"Messages classified : "
    f"{template_table['messages'].sum():,}"
)

print(
    f"Expected            : "
    f"{len(final_2var):,}"
)

assert template_table["messages"].sum() == len(final_2var)

print("\n✓ 100% of messages reconciled by template")

DPD 30–60 — ENGAGEMENT × CONTACT PRESSURE × TEMPLATE
dpd_region engagement_group pressure_bucket          template  messages  customers template_share  payment_events payment_rate recovery_per_message baseline_rpm delta_vs_baseline lift_vs_baseline avg_balance
  DPD30–45          Engaged      0 contacts    discount_offer       421        421          50.2%              43       10.21%             R$ 53.19     R$ 40.40         R$ +12.79           +31.7%   R$ 786.31
  DPD30–45          Engaged      0 contacts friendly_reminder         4          4           0.5%               1       25.00%             R$ 62.54     R$ 40.40         R$ +22.14           +54.8%   R$ 490.42
  DPD30–45          Engaged      0 contacts          pix_link       168        168          20.0%              14        8.33%             R$ 30.79     R$ 40.40          R$ -9.61           -23.8%   R$ 758.46
  DPD30–45          Engaged      0 contacts   urgent_reminder       246        246          29.3%              17  

In [42]:
# ============================================================
# DPD 30–60 — FINAL SLIDE TABLE
# ENGAGEMENT × CONTACTS 14D × TEMPLATE
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. CREATE FINAL OPERATIONAL SEGMENTS
# ------------------------------------------------------------

slide_base = final_2var.copy()

slide_base["slide_rule"] = pd.NA
slide_base["recommended_template"] = pd.NA


# ============================================================
# DPD 30–45
# ============================================================

# Engaged + 0 contacts → Discount
m = (
    slide_base["dpd_region"].eq("DPD30–45")
    & slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].eq(0)
)

slide_base.loc[m, "slide_rule"] = "ENGAGED | 0 contacts"
slide_base.loc[m, "recommended_template"] = "discount_offer"


# Engaged + 1–2 contacts → Discount
m = (
    slide_base["dpd_region"].eq("DPD30–45")
    & slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].between(1, 2)
)

slide_base.loc[m, "slide_rule"] = "ENGAGED | 1–2 contacts"
slide_base.loc[m, "recommended_template"] = "discount_offer"


# Engaged + 3–4 contacts → Discount
m = (
    slide_base["dpd_region"].eq("DPD30–45")
    & slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].between(3, 4)
)

slide_base.loc[m, "slide_rule"] = "ENGAGED | 3–4 contacts"
slide_base.loc[m, "recommended_template"] = "discount_offer"


# Engaged + 5+ → Suppress
m = (
    slide_base["dpd_region"].eq("DPD30–45")
    & slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].ge(5)
)

slide_base.loc[m, "slide_rule"] = "ENGAGED | 5+ contacts"
slide_base.loc[m, "recommended_template"] = "SUPPRESS"


# No engagement + 0 → Explore Discount
m = (
    slide_base["dpd_region"].eq("DPD30–45")
    & ~slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].eq(0)
)

slide_base.loc[m, "slide_rule"] = "NO ENGAGEMENT | 0 contacts"
slide_base.loc[m, "recommended_template"] = "EXPLORE_DISCOUNT"


# No engagement + 1+ → No send
m = (
    slide_base["dpd_region"].eq("DPD30–45")
    & ~slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].ge(1)
)

slide_base.loc[m, "slide_rule"] = "NO ENGAGEMENT | 1+ contacts"
slide_base.loc[m, "recommended_template"] = "NO_SEND"


# ============================================================
# DPD 46–60
# ============================================================

# Engaged + 0 → Discount
m = (
    slide_base["dpd_region"].eq("DPD46–60")
    & slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].eq(0)
)

slide_base.loc[m, "slide_rule"] = "ENGAGED | 0 contacts"
slide_base.loc[m, "recommended_template"] = "discount_offer"


# Engaged + 1–2 → Urgent
m = (
    slide_base["dpd_region"].eq("DPD46–60")
    & slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].between(1, 2)
)

slide_base.loc[m, "slide_rule"] = "ENGAGED | 1–2 contacts"
slide_base.loc[m, "recommended_template"] = "urgent_reminder"


# Engaged + 3+ → Suppress
m = (
    slide_base["dpd_region"].eq("DPD46–60")
    & slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].ge(3)
)

slide_base.loc[m, "slide_rule"] = "ENGAGED | 3+ contacts"
slide_base.loc[m, "recommended_template"] = "SUPPRESS"


# No engagement + 0 → Explore Discount
m = (
    slide_base["dpd_region"].eq("DPD46–60")
    & ~slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].eq(0)
)

slide_base.loc[m, "slide_rule"] = "NO ENGAGEMENT | 0 contacts"
slide_base.loc[m, "recommended_template"] = "EXPLORE_DISCOUNT"


# No engagement + 1+ → No send
m = (
    slide_base["dpd_region"].eq("DPD46–60")
    & ~slide_base["has_prior_engagement"]
    & slide_base["n_msgs_last_14d"].ge(1)
)

slide_base.loc[m, "slide_rule"] = "NO ENGAGEMENT | 1+ contacts"
slide_base.loc[m, "recommended_template"] = "NO_SEND"


# ------------------------------------------------------------
# 2. FOR SEND / EXPLORE:
# KEEP ONLY HISTORICAL OBSERVATIONS OF THE RECOMMENDED TEMPLATE
#
# FOR SUPPRESS / NO_SEND:
# USE ALL HISTORICAL OBSERVATIONS IN THAT SEGMENT
# ------------------------------------------------------------

send_mask = (
    (
        slide_base["recommended_template"].eq("discount_offer")
        & slide_base["template"].eq("discount_offer")
    )
    |
    (
        slide_base["recommended_template"].eq("urgent_reminder")
        & slide_base["template"].eq("urgent_reminder")
    )
    |
    (
        slide_base["recommended_template"].eq("EXPLORE_DISCOUNT")
        & slide_base["template"].eq("discount_offer")
    )
)

no_send_mask = slide_base["recommended_template"].isin(
    ["SUPPRESS", "NO_SEND"]
)

slide_hist = slide_base.loc[
    send_mask | no_send_mask
].copy()


# ------------------------------------------------------------
# 3. AGGREGATE
# ------------------------------------------------------------

slide_table = (
    slide_hist
    .groupby(
        [
            "dpd_region",
            "slide_rule",
            "recommended_template"
        ],
        observed=True,
        dropna=False
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. METRICS
# ------------------------------------------------------------

slide_table["payment_rate"] = (
    slide_table["payment_events"]
    / slide_table["messages"]
)

slide_table["recovery_per_message"] = (
    slide_table["recovery_brl"]
    / slide_table["messages"]
)


# ------------------------------------------------------------
# 5. FRIENDLY LABELS
# ------------------------------------------------------------

template_labels = {
    "discount_offer": "DISCOUNT",
    "urgent_reminder": "URGENT",
    "EXPLORE_DISCOUNT": "EXPLORE — DISCOUNT",
    "SUPPRESS": "SUPPRESS",
    "NO_SEND": "NO SEND"
}

slide_table["decision"] = (
    slide_table["recommended_template"]
    .map(template_labels)
)


# ------------------------------------------------------------
# 6. ORDER FOR SLIDE
# ------------------------------------------------------------

rule_order = [
    ("DPD30–45", "ENGAGED | 0 contacts"),
    ("DPD30–45", "ENGAGED | 1–2 contacts"),
    ("DPD30–45", "ENGAGED | 3–4 contacts"),
    ("DPD30–45", "ENGAGED | 5+ contacts"),
    ("DPD30–45", "NO ENGAGEMENT | 0 contacts"),
    ("DPD30–45", "NO ENGAGEMENT | 1+ contacts"),

    ("DPD46–60", "ENGAGED | 0 contacts"),
    ("DPD46–60", "ENGAGED | 1–2 contacts"),
    ("DPD46–60", "ENGAGED | 3+ contacts"),
    ("DPD46–60", "NO ENGAGEMENT | 0 contacts"),
    ("DPD46–60", "NO ENGAGEMENT | 1+ contacts"),
]

order_map = {
    key: i
    for i, key in enumerate(rule_order)
}

slide_table["_order"] = slide_table.apply(
    lambda r: order_map.get(
        (str(r["dpd_region"]), r["slide_rule"]),
        999
    ),
    axis=1
)

slide_table = (
    slide_table
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. DISPLAY FORMAT
# ------------------------------------------------------------

slide_out = slide_table.copy()

slide_out["payment_rate"] = (
    slide_out["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

slide_out["recovery_per_message"] = (
    slide_out["recovery_per_message"]
    .map(lambda x: f"R$ {x:,.2f}")
)

slide_out["recovery_brl"] = (
    slide_out["recovery_brl"]
    .map(lambda x: f"R$ {x:,.2f}")
)


# ------------------------------------------------------------
# 8. FINAL TABLE
# ------------------------------------------------------------

print("=" * 125)
print("DPD 30–60 — FINAL TABLE FOR SLIDE")
print("=" * 125)

print(
    slide_out[
        [
            "dpd_region",
            "slide_rule",
            "decision",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 9. QA
# ------------------------------------------------------------

print("\n" + "=" * 125)
print("QA")
print("=" * 125)

print(
    f"Historical DPD30–60 messages : "
    f"{len(final_2var):,}"
)

print(
    f"Final policy rows            : "
    f"{len(slide_table):,}"
)

print("\n✓ Final slide table generated")

DPD 30–60 — FINAL TABLE FOR SLIDE
dpd_region                  slide_rule           decision  messages  customers  payment_events payment_rate recovery_per_message
  DPD30–45        ENGAGED | 0 contacts           DISCOUNT       421        421              43       10.21%             R$ 53.19
  DPD30–45      ENGAGED | 1–2 contacts           DISCOUNT      1851       1536             190       10.26%             R$ 51.56
  DPD30–45      ENGAGED | 3–4 contacts           DISCOUNT       898        727              80        8.91%             R$ 52.35
  DPD30–45       ENGAGED | 5+ contacts           SUPPRESS       320        229              15        4.69%             R$ 31.68
  DPD30–45  NO ENGAGEMENT | 0 contacts EXPLORE — DISCOUNT        83         83               6        7.23%             R$ 50.50
  DPD30–45 NO ENGAGEMENT | 1+ contacts            NO SEND       929        514              17        1.83%             R$ 11.24
  DPD46–60        ENGAGED | 0 contacts           DISCOUNT      

In [43]:
# ============================================================
# DPD 30–45 — SHOULD PRIOR PAYERS RECEIVE DISCOUNT?
#
# Population:
#   DPD 30–45
#   Prior engagement = True
#
# Question:
#   Among customers we already decided to contact,
#   should prior payment protect the customer from Discount?
#
# Compare:
#   prior payment × contact pressure × template
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. BASE POPULATION
# ------------------------------------------------------------

discount_test = wa_pit.loc[
    wa_pit["days_past_due"].between(30, 45)
    & wa_pit["has_prior_engagement"].eq(True)
].copy()


# ------------------------------------------------------------
# 2. TARGET / RECOVERY
# ------------------------------------------------------------

discount_test["payment_event_72h"] = (
    discount_test["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

discount_test["amount_paid_brl"] = pd.to_numeric(
    discount_test["amount_paid_brl"],
    errors="coerce"
).fillna(0)

discount_test["recovery_72h"] = np.where(
    discount_test["payment_event_72h"],
    discount_test["amount_paid_brl"],
    0
)


# ------------------------------------------------------------
# 3. PRIOR PAYMENT
# ------------------------------------------------------------
# has_prior_payment MUST already be PIT:
# only payments observed BEFORE the current message

discount_test["prior_payment_group"] = np.where(
    discount_test["has_prior_payment"],
    "Prior payment",
    "No prior payment"
)


# ------------------------------------------------------------
# 4. CONTACT PRESSURE
# ------------------------------------------------------------

discount_test["pressure_bucket"] = pd.cut(
    discount_test["n_msgs_last_14d"],
    bins=[-1, 0, 2, 4, np.inf],
    labels=[
        "0 contacts",
        "1–2 contacts",
        "3–4 contacts",
        "5+ contacts"
    ]
)


# ------------------------------------------------------------
# 5. PAYMENT TYPE
#
# Full payment proxy:
# amount paid approximately equals outstanding balance
#
# We use tolerance because of cents / rounding.
# ------------------------------------------------------------

discount_test["outstanding_balance_brl"] = pd.to_numeric(
    discount_test["outstanding_balance_brl"],
    errors="coerce"
)

discount_test["payment_to_balance_ratio"] = np.where(
    (
        discount_test["payment_event_72h"]
        & discount_test["outstanding_balance_brl"].gt(0)
    ),
    (
        discount_test["recovery_72h"]
        / discount_test["outstanding_balance_brl"]
    ),
    np.nan
)

discount_test["full_payment_event"] = (
    discount_test["payment_event_72h"]
    & discount_test["payment_to_balance_ratio"].ge(0.99)
)


# ------------------------------------------------------------
# 6. ANALYSIS TABLE
# ------------------------------------------------------------

prior_payment_template = (
    discount_test
    .groupby(
        [
            "prior_payment_group",
            "pressure_bucket",
            "template"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),

        payment_events=("payment_event_72h", "sum"),
        full_payment_events=("full_payment_event", "sum"),

        recovery_brl=("recovery_72h", "sum"),

        avg_balance=("outstanding_balance_brl", "mean"),
        median_balance=("outstanding_balance_brl", "median"),

        avg_payment_when_paid=(
            "recovery_72h",
            lambda x: x[x > 0].mean()
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 7. ECONOMIC METRICS
# ------------------------------------------------------------

prior_payment_template["payment_rate"] = (
    prior_payment_template["payment_events"]
    / prior_payment_template["messages"]
)

prior_payment_template["full_payment_rate"] = (
    prior_payment_template["full_payment_events"]
    / prior_payment_template["messages"]
)

prior_payment_template["full_given_payment"] = np.where(
    prior_payment_template["payment_events"].gt(0),
    (
        prior_payment_template["full_payment_events"]
        / prior_payment_template["payment_events"]
    ),
    np.nan
)

prior_payment_template["recovery_per_message"] = (
    prior_payment_template["recovery_brl"]
    / prior_payment_template["messages"]
)


# ------------------------------------------------------------
# 8. KEEP RELEVANT TEMPLATES
# ------------------------------------------------------------

comparison = prior_payment_template.loc[
    prior_payment_template["template"].isin(
        [
            "discount_offer",
            "pix_link",
            "urgent_reminder"
        ]
    )
].copy()


# ------------------------------------------------------------
# 9. SORT
# ------------------------------------------------------------

payment_order = {
    "No prior payment": 0,
    "Prior payment": 1
}

pressure_order = {
    "0 contacts": 0,
    "1–2 contacts": 1,
    "3–4 contacts": 2,
    "5+ contacts": 3
}

comparison["_payment_order"] = (
    comparison["prior_payment_group"].map(payment_order)
)

comparison["_pressure_order"] = (
    comparison["pressure_bucket"]
    .astype(str)
    .map(pressure_order)
)

comparison = (
    comparison
    .sort_values(
        [
            "_payment_order",
            "_pressure_order",
            "recovery_per_message"
        ],
        ascending=[True, True, False]
    )
    .drop(
        columns=[
            "_payment_order",
            "_pressure_order"
        ]
    )
)


# ------------------------------------------------------------
# 10. FORMAT FOR PRINT
# ------------------------------------------------------------

out = comparison.copy()

out["payment_rate"] = (
    out["payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

out["full_payment_rate"] = (
    out["full_payment_rate"]
    .map(lambda x: f"{x:.2%}")
)

out["full_given_payment"] = (
    out["full_given_payment"]
    .map(
        lambda x:
        f"{x:.1%}"
        if pd.notna(x)
        else "-"
    )
)

out["recovery_per_message"] = (
    out["recovery_per_message"]
    .map(lambda x: f"R$ {x:,.2f}")
)

out["avg_payment_when_paid"] = (
    out["avg_payment_when_paid"]
    .map(
        lambda x:
        f"R$ {x:,.2f}"
        if pd.notna(x)
        else "-"
    )
)

out["avg_balance"] = (
    out["avg_balance"]
    .map(lambda x: f"R$ {x:,.2f}")
)


# ------------------------------------------------------------
# 11. PRINT
# ------------------------------------------------------------

print("=" * 165)
print(
    "DPD30–45 | ENGAGED — "
    "PRIOR PAYMENT × PRESSURE × TEMPLATE"
)
print("=" * 165)

print(
    out[
        [
            "prior_payment_group",
            "pressure_bucket",
            "template",

            "messages",
            "customers",

            "payment_events",
            "payment_rate",

            "full_payment_events",
            "full_payment_rate",
            "full_given_payment",

            "avg_payment_when_paid",
            "recovery_per_message",
            "avg_balance"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 12. QA
# ------------------------------------------------------------

print("\n" + "=" * 165)
print("QA")
print("=" * 165)

print(
    f"DPD30–45 engaged messages : "
    f"{len(discount_test):,}"
)

print(
    f"Customers                 : "
    f"{discount_test['customer_id'].nunique():,}"
)

print(
    f"Prior payment             : "
    f"{discount_test['has_prior_payment'].sum():,} messages"
)

print(
    f"No prior payment          : "
    f"{(~discount_test['has_prior_payment']).sum():,} messages"
)

print(
    "\n✓ Prior payment is point-in-time; "
    "current-message payment is used only as outcome."
)

DPD30–45 | ENGAGED — PRIOR PAYMENT × PRESSURE × TEMPLATE
prior_payment_group pressure_bucket        template  messages  customers  payment_events payment_rate  full_payment_events full_payment_rate full_given_payment avg_payment_when_paid recovery_per_message avg_balance
   No prior payment      0 contacts  discount_offer       364        364              32        8.79%                    0             0.00%               0.0%             R$ 643.12             R$ 56.54   R$ 857.14
   No prior payment      0 contacts urgent_reminder       206        206              11        5.34%                    5             2.43%              45.5%             R$ 518.95             R$ 27.71   R$ 913.06
   No prior payment      0 contacts        pix_link       134        134               6        4.48%                    3             2.24%              50.0%             R$ 525.78             R$ 23.54   R$ 861.42
   No prior payment    1–2 contacts  discount_offer      1521       1270           

In [44]:
# ============================================================
# FINAL RESCUE POLICY — DPD 30–60
#
# RULES
#
# DPD 30–45
#   Engaged + 0 contacts + prior payment → PIX LINK
#   Engaged + otherwise                   → DISCOUNT
#   No engagement                         → NO SEND
#
# DPD 46–60
#   Engaged + 0 contacts                  → DISCOUNT
#   Engaged + >0 contacts                 → URGENT REMINDER
#   No engagement                         → NO SEND
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. BASE
# ------------------------------------------------------------

final_rules = wa_pit.loc[
    wa_pit["days_past_due"].between(30, 60)
].copy()


# ------------------------------------------------------------
# 2. DPD REGION
# ------------------------------------------------------------

final_rules["dpd_region"] = pd.cut(
    final_rules["days_past_due"],
    bins=[29, 45, 60],
    labels=[
        "DPD30–45",
        "DPD46–60"
    ]
)


# ------------------------------------------------------------
# 3. INITIALIZE POLICY
# ------------------------------------------------------------

final_rules["policy_rule"] = "NO_SEND"
final_rules["recommended_template"] = pd.NA
final_rules["send_flag"] = False


# ============================================================
# RULE 1 — DPD 46–60
# ============================================================

mask_46_60_engaged = (
    final_rules["dpd_region"].eq("DPD46–60")
    & final_rules["has_prior_engagement"].eq(True)
)


# ------------------------------------------------------------
# R1A — ENGAGED + 0 CONTACTS → DISCOUNT
# ------------------------------------------------------------

mask_r1a = (
    mask_46_60_engaged
    & final_rules["n_msgs_last_14d"].eq(0)
)

final_rules.loc[
    mask_r1a,
    "policy_rule"
] = "R1A_46_60_ENGAGED_0_CONTACT_DISCOUNT"

final_rules.loc[
    mask_r1a,
    "recommended_template"
] = "discount_offer"

final_rules.loc[
    mask_r1a,
    "send_flag"
] = True


# ------------------------------------------------------------
# R1B — ENGAGED + >0 CONTACTS → URGENT
# ------------------------------------------------------------

mask_r1b = (
    mask_46_60_engaged
    & final_rules["n_msgs_last_14d"].gt(0)
)

final_rules.loc[
    mask_r1b,
    "policy_rule"
] = "R1B_46_60_ENGAGED_RECENT_URGENT"

final_rules.loc[
    mask_r1b,
    "recommended_template"
] = "urgent_reminder"

final_rules.loc[
    mask_r1b,
    "send_flag"
] = True


# ============================================================
# RULE 2 — DPD 30–45
# ============================================================

mask_30_45_engaged = (
    final_rules["dpd_region"].eq("DPD30–45")
    & final_rules["has_prior_engagement"].eq(True)
)


# ------------------------------------------------------------
# R2A
# ENGAGED + 0 CONTACTS + PRIOR PAYMENT
# → PIX LINK
# ------------------------------------------------------------

mask_r2a = (
    mask_30_45_engaged
    & final_rules["n_msgs_last_14d"].eq(0)
    & final_rules["has_prior_payment"].eq(True)
)

final_rules.loc[
    mask_r2a,
    "policy_rule"
] = "R2A_30_45_ENGAGED_0_CONTACT_PRIOR_PAY_PIX"

final_rules.loc[
    mask_r2a,
    "recommended_template"
] = "pix_link"

final_rules.loc[
    mask_r2a,
    "send_flag"
] = True


# ------------------------------------------------------------
# R2B
# ALL OTHER ENGAGED DPD30–45
# → DISCOUNT
# ------------------------------------------------------------

mask_r2b = (
    mask_30_45_engaged
    & ~mask_r2a
)

final_rules.loc[
    mask_r2b,
    "policy_rule"
] = "R2B_30_45_ENGAGED_OTHER_DISCOUNT"

final_rules.loc[
    mask_r2b,
    "recommended_template"
] = "discount_offer"

final_rules.loc[
    mask_r2b,
    "send_flag"
] = True


# ============================================================
# 4. EXPLICIT NO-SEND REASONS
# ============================================================

mask_no_engagement = (
    ~final_rules["has_prior_engagement"]
)

final_rules.loc[
    mask_no_engagement,
    "policy_rule"
] = "NO_SEND_NO_ENGAGEMENT"


# ============================================================
# 5. QA — EVERY ROW MUST HAVE EXACTLY ONE DECISION
# ============================================================

assert final_rules["policy_rule"].notna().all()

assert (
    final_rules.loc[
        final_rules["send_flag"],
        "recommended_template"
    ]
    .notna()
    .all()
)

assert (
    final_rules.loc[
        ~final_rules["send_flag"],
        "recommended_template"
    ]
    .isna()
    .all()
)


# ============================================================
# 6. SUMMARY
# ============================================================

policy_summary = (
    final_rules
    .groupby(
        [
            "dpd_region",
            "policy_rule",
            "recommended_template",
            "send_flag"
        ],
        dropna=False,
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        avg_dpd=("days_past_due", "mean"),
        avg_contacts_14d=("n_msgs_last_14d", "mean"),
        prior_payment_share=("has_prior_payment", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# FORMAT
# ------------------------------------------------------------

out = policy_summary.copy()

out["prior_payment_share"] = (
    out["prior_payment_share"]
    .map(lambda x: f"{x:.1%}")
)

out["avg_dpd"] = (
    out["avg_dpd"]
    .map(lambda x: f"{x:.1f}")
)

out["avg_contacts_14d"] = (
    out["avg_contacts_14d"]
    .map(lambda x: f"{x:.2f}")
)


# ============================================================
# 7. PRINT
# ============================================================

print("=" * 150)
print("FINAL RESCUE POLICY — DPD30–60")
print("=" * 150)

print(
    out[
        [
            "dpd_region",
            "policy_rule",
            "recommended_template",
            "send_flag",
            "messages",
            "customers",
            "avg_dpd",
            "avg_contacts_14d",
            "prior_payment_share"
        ]
    ].to_string(index=False)
)


# ============================================================
# 8. SIMPLE DECISION TABLE
# ============================================================

print("\n" + "=" * 150)
print("OPERATIONAL DECISION TREE")
print("=" * 150)

print("""
DPD30–45
│
├── NO ENGAGEMENT
│      └── NO SEND
│
└── ENGAGED
       │
       ├── 0 contacts 14d + PRIOR PAYMENT
       │      └── PIX LINK
       │
       └── OTHERWISE
              └── DISCOUNT OFFER


DPD46–60
│
├── NO ENGAGEMENT
│      └── NO SEND
│
└── ENGAGED
       │
       ├── 0 contacts 14d
       │      └── DISCOUNT OFFER
       │
       └── >0 contacts 14d
              └── URGENT REMINDER
""")


# ============================================================
# 9. RECONCILIATION
# ============================================================

print("=" * 150)
print("RECONCILIATION")
print("=" * 150)

print(f"Total messages DPD30–60 : {len(final_rules):,}")
print(f"SEND                    : {final_rules['send_flag'].sum():,}")
print(f"NO SEND                 : {(~final_rules['send_flag']).sum():,}")

print("\nTemplates among SEND:")

print(
    final_rules.loc[
        final_rules["send_flag"],
        "recommended_template"
    ]
    .value_counts()
    .to_string()
)

assert (
    final_rules["send_flag"].sum()
    + (~final_rules["send_flag"]).sum()
    == len(final_rules)
)

print("\n✓ 100% da população DPD30–60 classificada.")

FINAL RESCUE POLICY — DPD30–60
dpd_region                               policy_rule recommended_template  send_flag  messages  customers avg_dpd avg_contacts_14d prior_payment_share
  DPD30–45                     NO_SEND_NO_ENGAGEMENT                  NaN      False      1093        617    36.4             1.87                4.0%
  DPD30–45 R2A_30_45_ENGAGED_0_CONTACT_PRIOR_PAY_PIX             pix_link       True       131        131    38.8             0.00              100.0%
  DPD30–45          R2B_30_45_ENGAGED_OTHER_DISCOUNT       discount_offer       True      7328       3942    36.6             2.06               15.6%
  DPD46–60                     NO_SEND_NO_ENGAGEMENT                  NaN      False       561        321    52.6             1.37                2.9%
  DPD46–60      R1A_46_60_ENGAGED_0_CONTACT_DISCOUNT       discount_offer       True      1029       1029    52.6             0.00               19.2%
  DPD46–60           R1B_46_60_ENGAGED_RECENT_URGENT      urgen

In [45]:
# ============================================================
# SLIDE — FINAL RESCUE STRATEGY
# DPD30–45 vs DPD46–60
#
# Objetivo:
# gerar os dados para um slide no mesmo formato do anterior:
#
# LEFT  : DPD30–45
# RIGHT : DPD46–60
#
# Métrica principal = recovery / message
# Métrica apoio     = payment rate
#
# ------------------------------------------------------------
# REGRAS A VALIDAR
#
# DPD30–45
#   Engaged + 0 contact + prior payment → PIX
#   Engaged + otherwise                 → DISCOUNT
#
# DPD46–60
#   Engaged + 0 contact                 → DISCOUNT
#   Engaged + >0 contact                → URGENT
#
# Sem engagement → NO SEND
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. BASE
# ============================================================

df = wa_pit.loc[
    wa_pit["days_past_due"].between(30, 60)
].copy()


# ------------------------------------------------------------
# OUTCOME
# ------------------------------------------------------------

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)


# ============================================================
# 2. DPD REGION
# ============================================================

df["dpd_region"] = pd.cut(
    df["days_past_due"],
    bins=[29, 45, 60],
    labels=[
        "DPD30–45",
        "DPD46–60"
    ]
)


# ============================================================
# 3. ENGAGEMENT
# ============================================================

df["engagement_group"] = np.where(
    df["has_prior_engagement"],
    "ENGAGED",
    "NO ENGAGEMENT"
)


# ============================================================
# 4. STATE / RULE SEGMENT
# ============================================================

df["state"] = pd.NA


# ------------------------------------------------------------
# DPD30–45
#
# A = engaged + 0 contact + prior payment
# B = other engaged
# ------------------------------------------------------------

m = (
    df["dpd_region"].eq("DPD30–45")
    & df["has_prior_engagement"]
    & df["n_msgs_last_14d"].eq(0)
    & df["has_prior_payment"]
)

df.loc[m, "state"] = (
    "A. 0 CONTACT + PRIOR PAYMENT"
)


m = (
    df["dpd_region"].eq("DPD30–45")
    & df["has_prior_engagement"]
    & ~(
        df["n_msgs_last_14d"].eq(0)
        & df["has_prior_payment"]
    )
)

df.loc[m, "state"] = (
    "B. OTHER ENGAGED"
)


# ------------------------------------------------------------
# DPD46–60
#
# A = engaged + 0 contact
# B = engaged + recent contact
# ------------------------------------------------------------

m = (
    df["dpd_region"].eq("DPD46–60")
    & df["has_prior_engagement"]
    & df["n_msgs_last_14d"].eq(0)
)

df.loc[m, "state"] = (
    "A. 14D SEM CONTATO"
)


m = (
    df["dpd_region"].eq("DPD46–60")
    & df["has_prior_engagement"]
    & df["n_msgs_last_14d"].gt(0)
)

df.loc[m, "state"] = (
    "B. CONTATO RECENTE"
)


# ============================================================
# 5. NO ENGAGEMENT — SEPARATE TABLE
# ============================================================

no_engagement = df.loc[
    ~df["has_prior_engagement"]
].copy()


# ============================================================
# 6. BASELINE BY DPD
# ============================================================

baseline = (
    df
    .groupby(
        "dpd_region",
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum")
    )
    .reset_index()
)

baseline["payment_rate"] = (
    baseline["payment_events"]
    / baseline["messages"]
)

baseline["recovery_per_message"] = (
    baseline["recovery_brl"]
    / baseline["messages"]
)


# ============================================================
# 7. ENGAGED vs NO ENGAGEMENT
#
# Isso alimenta a primeira parte do slide:
# "quem recebe +1 tentativa?"
# ============================================================

engagement_table = (
    df
    .groupby(
        [
            "dpd_region",
            "engagement_group"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum")
    )
    .reset_index()
)

engagement_table["payment_rate"] = (
    engagement_table["payment_events"]
    / engagement_table["messages"]
)

engagement_table["recovery_per_message"] = (
    engagement_table["recovery_brl"]
    / engagement_table["messages"]
)


# ============================================================
# 8. STATE × TEMPLATE
#
# Essa é a tabela PRINCIPAL para os dois painéis.
# ============================================================

template_table = (
    df.loc[
        df["state"].notna()
        & df["template"].isin(
            [
                "pix_link",
                "urgent_reminder",
                "discount_offer"
            ]
        )
    ]
    .groupby(
        [
            "dpd_region",
            "state",
            "template"
        ],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)


template_table["payment_rate"] = (
    template_table["payment_events"]
    / template_table["messages"]
)

template_table["recovery_per_message"] = (
    template_table["recovery_brl"]
    / template_table["messages"]
)


# ============================================================
# 9. ADD DPD BASELINE
# ============================================================

template_table = template_table.merge(
    baseline[
        [
            "dpd_region",
            "recovery_per_message"
        ]
    ].rename(
        columns={
            "recovery_per_message":
            "dpd_baseline_rpm"
        }
    ),
    on="dpd_region",
    how="left"
)


template_table["delta_vs_baseline"] = (
    template_table["recovery_per_message"]
    - template_table["dpd_baseline_rpm"]
)

template_table["lift_vs_baseline"] = (
    template_table["recovery_per_message"]
    / template_table["dpd_baseline_rpm"]
    - 1
)


# ============================================================
# 10. BEST TEMPLATE IN EACH STATE
#
# Serve para destacar a célula clara no slide.
# ============================================================

template_table["rank_rpm"] = (
    template_table
    .groupby(
        [
            "dpd_region",
            "state"
        ]
    )["recovery_per_message"]
    .rank(
        method="first",
        ascending=False
    )
)

template_table["best_template"] = (
    template_table["rank_rpm"].eq(1)
)


# ============================================================
# 11. NO ENGAGEMENT SUMMARY
#
# Sustenta NO SEND.
# ============================================================

no_engagement_table = (
    no_engagement
    .groupby(
        "dpd_region",
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum")
    )
    .reset_index()
)

no_engagement_table["payment_rate"] = (
    no_engagement_table["payment_events"]
    / no_engagement_table["messages"]
)

no_engagement_table["recovery_per_message"] = (
    no_engagement_table["recovery_brl"]
    / no_engagement_table["messages"]
)


# ============================================================
# 12. FORMAT FUNCTION
# ============================================================

def format_table(x):

    out = x.copy()

    if "payment_rate" in out.columns:
        out["payment_rate"] = (
            out["payment_rate"]
            .map(lambda v: f"{v:.2%}")
        )

    if "recovery_per_message" in out.columns:
        out["recovery_per_message"] = (
            out["recovery_per_message"]
            .map(lambda v: f"R$ {v:,.2f}")
        )

    if "dpd_baseline_rpm" in out.columns:
        out["dpd_baseline_rpm"] = (
            out["dpd_baseline_rpm"]
            .map(lambda v: f"R$ {v:,.2f}")
        )

    if "delta_vs_baseline" in out.columns:
        out["delta_vs_baseline"] = (
            out["delta_vs_baseline"]
            .map(lambda v: f"R$ {v:+,.2f}")
        )

    if "lift_vs_baseline" in out.columns:
        out["lift_vs_baseline"] = (
            out["lift_vs_baseline"]
            .map(lambda v: f"{v:+.1%}")
        )

    return out


# ============================================================
# 13. PRINT — SLIDE INPUT 1
# ENGAGEMENT
# ============================================================

print("\n")
print("=" * 150)
print("1. QUEM RECEBE +1 TENTATIVA?")
print("ENGAGED vs NO ENGAGEMENT")
print("=" * 150)

print(
    format_table(
        engagement_table
    )[
        [
            "dpd_region",
            "engagement_group",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message"
        ]
    ].to_string(index=False)
)


# ============================================================
# 14. PRINT — SLIDE INPUT 2
# DPD30–45
# ============================================================

print("\n")
print("=" * 150)
print("2A. DPD30–45 — STATE × TEMPLATE")
print("=" * 150)

t30 = template_table.loc[
    template_table["dpd_region"].eq(
        "DPD30–45"
    )
].copy()

print(
    format_table(t30)[
        [
            "state",
            "template",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "dpd_baseline_rpm",
            "delta_vs_baseline",
            "lift_vs_baseline",
            "best_template"
        ]
    ].to_string(index=False)
)


# ============================================================
# 15. PRINT — SLIDE INPUT 3
# DPD46–60
# ============================================================

print("\n")
print("=" * 150)
print("2B. DPD46–60 — STATE × TEMPLATE")
print("=" * 150)

t46 = template_table.loc[
    template_table["dpd_region"].eq(
        "DPD46–60"
    )
].copy()

print(
    format_table(t46)[
        [
            "state",
            "template",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "dpd_baseline_rpm",
            "delta_vs_baseline",
            "lift_vs_baseline",
            "best_template"
        ]
    ].to_string(index=False)
)


# ============================================================
# 16. PRINT — SLIDE INPUT 4
# NO ENGAGEMENT
# ============================================================

print("\n")
print("=" * 150)
print("3. NO ENGAGEMENT — NO SEND EVIDENCE")
print("=" * 150)

print(
    format_table(
        no_engagement_table
    )[
        [
            "dpd_region",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message"
        ]
    ].to_string(index=False)
)


# ============================================================
# 17. PRINT — FINAL RULES
# ============================================================

print("\n")
print("=" * 150)
print("4. FINAL RULES FOR SLIDE")
print("=" * 150)

print(
"""
DPD30–45
------------------------------------------------------------
ENGAGED
  0 contacts + prior payment  → PIX LINK
  otherwise                   → DISCOUNT OFFER

NO ENGAGEMENT                 → NO SEND


DPD46–60
------------------------------------------------------------
ENGAGED
  0 contacts                  → DISCOUNT OFFER
  >0 contacts                 → URGENT REMINDER

NO ENGAGEMENT                 → NO SEND
"""
)


# ============================================================
# 18. QA
# ============================================================

print("=" * 150)
print("QA")
print("=" * 150)

print(
    f"DPD30–60 messages  : "
    f"{len(df):,}"
)

print(
    f"DPD30–60 customers : "
    f"{df['customer_id'].nunique():,}"
)

print(
    f"Engaged messages   : "
    f"{df['has_prior_engagement'].sum():,}"
)

print(
    f"No engagement      : "
    f"{(~df['has_prior_engagement']).sum():,}"
)

print(
    "\n✓ Tabelas prontas para alimentar o slide."
)



1. QUEM RECEBE +1 TENTATIVA?
ENGAGED vs NO ENGAGEMENT
dpd_region engagement_group  messages  customers  payment_events payment_rate recovery_per_message
  DPD30–45          ENGAGED      7459       4026             581        7.79%             R$ 44.06
  DPD30–45    NO ENGAGEMENT      1093        617              28        2.56%             R$ 15.39
  DPD46–60          ENGAGED      4402       2551             226        5.13%             R$ 27.98
  DPD46–60    NO ENGAGEMENT       561        321               3        0.53%              R$ 3.73


2A. DPD30–45 — STATE × TEMPLATE
                       state        template  messages  customers  payment_events payment_rate recovery_per_message dpd_baseline_rpm delta_vs_baseline lift_vs_baseline  best_template
A. 0 CONTACT + PRIOR PAYMENT  discount_offer        57         57              11       19.30%             R$ 31.81         R$ 40.40          R$ -8.59           -21.3%          False
A. 0 CONTACT + PRIOR PAYMENT        pix_link     

In [47]:
# ============================================================
# STATISTICAL TEST
# DPD30–45 | ENGAGED | 0 CONTACTS | PRIOR PAYMENT
#
# QUESTION:
# Is PIX better than DISCOUNT?
#
# Primary metric:
#   Recovery / message
#
# Secondary metric:
#   Payment rate
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact


# ============================================================
# 1. SELECT COMPARABLE POPULATION
# ============================================================

test = wa_pit.loc[
    wa_pit["days_past_due"].between(30, 45)
    & wa_pit["has_prior_engagement"].eq(True)
    & wa_pit["n_msgs_last_14d"].eq(0)
    & wa_pit["has_prior_payment"].eq(True)
    & wa_pit["template"].isin([
        "pix_link",
        "discount_offer"
    ])
].copy()


# ============================================================
# 2. OUTCOME
# ============================================================

test["payment_event_72h"] = (
    test["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

test["amount_paid_brl"] = pd.to_numeric(
    test["amount_paid_brl"],
    errors="coerce"
).fillna(0)

test["recovery_72h"] = np.where(
    test["payment_event_72h"],
    test["amount_paid_brl"],
    0
)


# ============================================================
# 3. DESCRIPTIVE SUMMARY
# ============================================================

summary = (
    test
    .groupby("template")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
)

summary["payment_rate"] = (
    summary["payment_events"]
    / summary["messages"]
)

print("=" * 100)
print("DESCRIPTIVE")
print("=" * 100)

print(summary)


# ============================================================
# 4. ARRAYS
# ============================================================

pix = test.loc[
    test["template"].eq("pix_link"),
    "recovery_72h"
].to_numpy()

discount = test.loc[
    test["template"].eq("discount_offer"),
    "recovery_72h"
].to_numpy()


observed_diff = (
    pix.mean()
    - discount.mean()
)


# ============================================================
# 5. PERMUTATION TEST
#
# H0: template assignment does not change recovery distribution
# H1: PIX has higher mean recovery/msg than DISCOUNT
#
# One-sided test: PIX > DISCOUNT
# ============================================================

rng = np.random.default_rng(42)

values = np.concatenate([
    pix,
    discount
])

n_pix = len(pix)

N_PERM = 100_000

perm_diffs = np.empty(N_PERM)

for i in range(N_PERM):

    perm = rng.permutation(values)

    perm_diffs[i] = (
        perm[:n_pix].mean()
        - perm[n_pix:].mean()
    )


p_value_recovery = (
    np.sum(perm_diffs >= observed_diff) + 1
) / (N_PERM + 1)


# ============================================================
# 6. BOOTSTRAP CI FOR DIFFERENCE
#
# PIX recovery/msg - DISCOUNT recovery/msg
# ============================================================

N_BOOT = 100_000

boot_diff = np.empty(N_BOOT)

for i in range(N_BOOT):

    pix_boot = rng.choice(
        pix,
        size=len(pix),
        replace=True
    )

    discount_boot = rng.choice(
        discount,
        size=len(discount),
        replace=True
    )

    boot_diff[i] = (
        pix_boot.mean()
        - discount_boot.mean()
    )


ci_low, ci_high = np.percentile(
    boot_diff,
    [2.5, 97.5]
)


# ============================================================
# 7. PAYMENT RATE — FISHER EXACT TEST
#
# Better than chi-square here because n is small.
# ============================================================

pix_paid = int(
    test.loc[
        test["template"].eq("pix_link"),
        "payment_event_72h"
    ].sum()
)

pix_not_paid = len(pix) - pix_paid


discount_paid = int(
    test.loc[
        test["template"].eq("discount_offer"),
        "payment_event_72h"
    ].sum()
)

discount_not_paid = len(discount) - discount_paid


contingency = np.array([
    [pix_paid, pix_not_paid],
    [discount_paid, discount_not_paid]
])


odds_ratio, p_value_payment = fisher_exact(
    contingency,
    alternative="greater"
)


# ============================================================
# 8. OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("PIX vs DISCOUNT")
print("DPD30–45 | ENGAGED | 0 CONTACTS | PRIOR PAYMENT")
print("=" * 100)

print(
    f"""
PIX
--------------------------------------------------
n                    : {len(pix):,}
Payment events       : {pix_paid:,}
Payment rate         : {pix_paid / len(pix):.2%}
Recovery / msg       : R$ {pix.mean():,.2f}

DISCOUNT
--------------------------------------------------
n                    : {len(discount):,}
Payment events       : {discount_paid:,}
Payment rate         : {discount_paid / len(discount):.2%}
Recovery / msg       : R$ {discount.mean():,.2f}

DIFFERENCE
--------------------------------------------------
PIX - Discount       : R$ {observed_diff:,.2f} / msg
Relative difference  : {(pix.mean()/discount.mean()-1):+.1%}

RECOVERY / MESSAGE
--------------------------------------------------
Permutation p-value  : {p_value_recovery:.4f}
Bootstrap 95% CI     : R$ [{ci_low:,.2f}, {ci_high:,.2f}]

PAYMENT RATE
--------------------------------------------------
Fisher odds ratio    : {odds_ratio:.3f}
Fisher p-value       : {p_value_payment:.4f}
"""
)


# ============================================================
# 9. SIMPLE INTERPRETATION
# ============================================================

print("=" * 100)
print("INTERPRETATION")
print("=" * 100)

if p_value_recovery < 0.05 and ci_low > 0:

    print(
        "✓ Recovery/msg: evidence supports PIX > Discount "
        "at the conventional 5% level."
    )

elif p_value_recovery < 0.10:

    print(
        "△ Recovery/msg: directional evidence, "
        "but not strong enough for a conventional 5% conclusion."
    )

else:

    print(
        "✕ Recovery/msg: insufficient evidence to conclude "
        "PIX > Discount."
    )


if p_value_payment < 0.05:

    print(
        "✓ Payment rate: evidence supports PIX > Discount."
    )

else:

    print(
        "✕ Payment rate: insufficient evidence to conclude "
        "PIX > Discount."
    )

DESCRIPTIVE
                messages  customers  payment_events  recovery_brl  \
template                                                            
discount_offer        57         57              11      1,812.93   
pix_link              34         34               8      2,017.33   

                recovery_per_message  avg_balance  payment_rate  
template                                                         
discount_offer                 31.81       333.97          0.19  
pix_link                       59.33       352.69          0.24  

PIX vs DISCOUNT
DPD30–45 | ENGAGED | 0 CONTACTS | PRIOR PAYMENT

PIX
--------------------------------------------------
n                    : 34
Payment events       : 8
Payment rate         : 23.53%
Recovery / msg       : R$ 59.33

DISCOUNT
--------------------------------------------------
n                    : 57
Payment events       : 11
Payment rate         : 19.30%
Recovery / msg       : R$ 31.81

DIFFERENCE
--------------------------

In [48]:
# ============================================================
# STATISTICAL VALIDATION — TEMPLATE DECISIONS
#
# Remaining comparisons:
#
# 1. DPD30–45 | Other engaged
#       Discount vs Pix
#
# 2. DPD46–60 | Engaged | 0 contacts 14d
#       Discount vs Urgent
#
# 3. DPD46–60 | Engaged | >0 contacts 14d
#       Urgent vs Pix
#
# Primary metric:
#       Recovery / message
#
# Tests:
#       - Permutation test (one-sided)
#       - Bootstrap 95% CI
#
# Secondary:
#       - Payment rate
#       - Fisher exact test
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact


# ============================================================
# 0. PREPARE OUTCOMES
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)


# ============================================================
# 1. GENERIC TEST FUNCTION
# ============================================================

def compare_templates(
    data,
    winner,
    challenger,
    segment_name,
    n_perm=100_000,
    n_boot=100_000,
    seed=42
):

    rng = np.random.default_rng(seed)

    d = data.loc[
        data["template"].isin([winner, challenger])
    ].copy()

    # --------------------------------------------------------
    # Arrays — recovery / message
    # --------------------------------------------------------

    a = d.loc[
        d["template"].eq(winner),
        "recovery_72h"
    ].to_numpy()

    b = d.loc[
        d["template"].eq(challenger),
        "recovery_72h"
    ].to_numpy()

    mean_a = a.mean()
    mean_b = b.mean()

    observed_diff = mean_a - mean_b

    relative_diff = (
        mean_a / mean_b - 1
        if mean_b != 0
        else np.nan
    )


    # ========================================================
    # PERMUTATION TEST
    #
    # H0: winner is NOT better than challenger
    # H1: winner > challenger
    # ========================================================

    values = np.concatenate([a, b])

    n_a = len(a)

    perm_diffs = np.empty(n_perm)

    for i in range(n_perm):

        perm = rng.permutation(values)

        perm_diffs[i] = (
            perm[:n_a].mean()
            - perm[n_a:].mean()
        )

    p_recovery = (
        np.sum(perm_diffs >= observed_diff) + 1
    ) / (n_perm + 1)


    # ========================================================
    # BOOTSTRAP 95% CI
    # ========================================================

    boot_diff = np.empty(n_boot)

    for i in range(n_boot):

        boot_a = rng.choice(
            a,
            size=len(a),
            replace=True
        )

        boot_b = rng.choice(
            b,
            size=len(b),
            replace=True
        )

        boot_diff[i] = (
            boot_a.mean()
            - boot_b.mean()
        )

    ci_low, ci_high = np.percentile(
        boot_diff,
        [2.5, 97.5]
    )


    # ========================================================
    # PAYMENT RATE
    # ========================================================

    paid_a = int(
        d.loc[
            d["template"].eq(winner),
            "payment_event_72h"
        ].sum()
    )

    paid_b = int(
        d.loc[
            d["template"].eq(challenger),
            "payment_event_72h"
        ].sum()
    )

    not_paid_a = len(a) - paid_a
    not_paid_b = len(b) - paid_b

    contingency = np.array([
        [paid_a, not_paid_a],
        [paid_b, not_paid_b]
    ])

    odds_ratio, p_payment = fisher_exact(
        contingency,
        alternative="greater"
    )


    # ========================================================
    # INTERPRETATION
    # ========================================================

    if (
        p_recovery < 0.05
        and ci_low > 0
    ):
        evidence = "STRONG"

    elif p_recovery < 0.10:
        evidence = "DIRECTIONAL"

    else:
        evidence = "INCONCLUSIVE"


    # ========================================================
    # OUTPUT
    # ========================================================

    result = {

        "segment": segment_name,

        "candidate": winner,
        "alternative": challenger,

        "candidate_n": len(a),
        "alternative_n": len(b),

        "candidate_recovery_msg": mean_a,
        "alternative_recovery_msg": mean_b,

        "diff_brl_msg": observed_diff,
        "relative_diff": relative_diff,

        "ci95_low": ci_low,
        "ci95_high": ci_high,

        "p_recovery": p_recovery,

        "candidate_payment_rate": (
            paid_a / len(a)
        ),

        "alternative_payment_rate": (
            paid_b / len(b)
        ),

        "p_payment": p_payment,

        "evidence": evidence
    }

    return result


# ============================================================
# 2. SEGMENT 1
#
# DPD30–45
# Other engaged
#
# Definition:
# engaged customers EXCEPT
# 0 contacts + prior payment
#
# Candidate:
# Discount
#
# Challenger:
# Pix
# ============================================================

seg_30_45 = df.loc[
    df["days_past_due"].between(30, 45)
    & df["has_prior_engagement"].eq(True)
    & ~(
        df["n_msgs_last_14d"].eq(0)
        & df["has_prior_payment"].eq(True)
    )
].copy()


r1 = compare_templates(

    data=seg_30_45,

    winner="discount_offer",

    challenger="pix_link",

    segment_name=(
        "DPD30–45 | Other engaged"
    )
)


# ============================================================
# 3. SEGMENT 2
#
# DPD46–60
# Engaged
# 0 contacts last 14d
#
# Candidate:
# Discount
#
# Challenger:
# Urgent
# ============================================================

seg_46_60_zero = df.loc[
    df["days_past_due"].between(46, 60)
    & df["has_prior_engagement"].eq(True)
    & df["n_msgs_last_14d"].eq(0)
].copy()


r2 = compare_templates(

    data=seg_46_60_zero,

    winner="discount_offer",

    challenger="urgent_reminder",

    segment_name=(
        "DPD46–60 | Engaged | 0 contacts"
    )
)


# ============================================================
# 4. SEGMENT 3
#
# DPD46–60
# Engaged
# >0 contacts last 14d
#
# Candidate:
# Urgent
#
# Challenger:
# Pix
# ============================================================

seg_46_60_recent = df.loc[
    df["days_past_due"].between(46, 60)
    & df["has_prior_engagement"].eq(True)
    & df["n_msgs_last_14d"].gt(0)
].copy()


r3 = compare_templates(

    data=seg_46_60_recent,

    winner="urgent_reminder",

    challenger="pix_link",

    segment_name=(
        "DPD46–60 | Engaged | >0 contacts"
    )
)


# ============================================================
# 5. FINAL RESULTS
# ============================================================

results = pd.DataFrame([
    r1,
    r2,
    r3
])


# ============================================================
# 6. FORMATTED PRINT
# ============================================================

print("\n")
print("=" * 120)
print("STATISTICAL VALIDATION — TEMPLATE DECISIONS")
print("=" * 120)


for _, r in results.iterrows():

    print("\n")
    print("-" * 120)

    print(r["segment"])

    print("-" * 120)

    print(
        f"""
Candidate    : {r['candidate']}
Alternative  : {r['alternative']}

RECOVERY / MESSAGE
------------------------------------------------------------
Candidate              : R$ {r['candidate_recovery_msg']:,.2f}
Alternative            : R$ {r['alternative_recovery_msg']:,.2f}

Difference             : R$ {r['diff_brl_msg']:,.2f}
Relative difference    : {r['relative_diff']:+.1%}

Candidate n            : {r['candidate_n']:,}
Alternative n          : {r['alternative_n']:,}

Permutation p-value    : {r['p_recovery']:.4f}

Bootstrap 95% CI
Difference             : R$ [{r['ci95_low']:,.2f}, {r['ci95_high']:,.2f}]


PAYMENT RATE
------------------------------------------------------------
Candidate              : {r['candidate_payment_rate']:.2%}
Alternative            : {r['alternative_payment_rate']:.2%}

Fisher p-value         : {r['p_payment']:.4f}


EVIDENCE
------------------------------------------------------------
{r['evidence']}
"""
    )


# ============================================================
# 7. COMPACT TABLE
# ============================================================

display_cols = [

    "segment",

    "candidate",
    "alternative",

    "candidate_n",
    "alternative_n",

    "candidate_recovery_msg",
    "alternative_recovery_msg",

    "diff_brl_msg",

    "ci95_low",
    "ci95_high",

    "p_recovery",

    "candidate_payment_rate",
    "alternative_payment_rate",

    "p_payment",

    "evidence"
]


print("\n")
print("=" * 120)
print("FINAL SUMMARY")
print("=" * 120)

display(
    results[display_cols]
    .round({
        "candidate_recovery_msg": 2,
        "alternative_recovery_msg": 2,
        "diff_brl_msg": 2,
        "ci95_low": 2,
        "ci95_high": 2,
        "p_recovery": 4,
        "candidate_payment_rate": 4,
        "alternative_payment_rate": 4,
        "p_payment": 4
    })
)



STATISTICAL VALIDATION — TEMPLATE DECISIONS


------------------------------------------------------------------------------------------------------------------------
DPD30–45 | Other engaged
------------------------------------------------------------------------------------------------------------------------

Candidate    : discount_offer
Alternative  : pix_link

RECOVERY / MESSAGE
------------------------------------------------------------
Candidate              : R$ 52.75
Alternative            : R$ 43.33

Difference             : R$ 9.42
Relative difference    : +21.7%

Candidate n            : 3,233
Alternative n          : 1,530

Permutation p-value    : 0.0701

Bootstrap 95% CI
Difference             : R$ [-3.15, 21.65]


PAYMENT RATE
------------------------------------------------------------
Candidate              : 9.59%
Alternative            : 7.39%

Fisher p-value         : 0.0067


EVIDENCE
------------------------------------------------------------
DIRECTIONAL





,segment,candidate,alternative,candidate_n,alternative_n,candidate_recovery_msg,alternative_recovery_msg,diff_brl_msg,ci95_low,ci95_high,p_recovery,candidate_payment_rate,alternative_payment_rate,p_payment,evidence
0,DPD30–45 | Other engaged,discount_offer,pix_link,3233,1530,52.75,43.33,9.42,-3.15,21.65,0.07,0.10,0.07,0.01,DIRECTIONAL
1,DPD46–60 | Engaged | 0 contacts,discount_offer,urgent_reminder,497,313,38.52,26.74,11.78,-13.00,34.48,0.17,0.08,0.04,0.03,INCONCLUSIVE
2,DPD46–60 | Engaged | >0 contacts,urgent_reminder,pix_link,997,697,30.22,26.84,3.38,-13.84,19.74,0.35,0.04,0.04,0.43,INCONCLUSIVE


In [6]:
# ============================================================
# DPD 30-45 — "IS ONE MORE MESSAGE WORTH IT?"
#
# Grain:
# 1 row = 1 historical WhatsApp send at DPD 30-45
#
# IMPORTANT:
# All history variables below are calculated STRICTLY
# BEFORE the current message.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. PREPARE FULL WHATSAPP HISTORY
# ============================================================

hist = wa.copy()

hist["sent_at"] = pd.to_datetime(
    hist["sent_at"]
)

hist["days_past_due"] = pd.to_numeric(
    hist["days_past_due"],
    errors="coerce"
)

hist["amount_paid_brl"] = pd.to_numeric(
    hist["amount_paid_brl"],
    errors="coerce"
).fillna(0)

hist["outstanding_balance_brl"] = pd.to_numeric(
    hist["outstanding_balance_brl"],
    errors="coerce"
)

hist = (
    hist
    .sort_values(
        ["customer_id", "sent_at"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 2. MESSAGE NUMBER IN CUSTOMER JOURNEY
#
# cumcount = number of messages STRICTLY BEFORE current one
# ============================================================

hist["n_prior_msgs_total"] = (
    hist
    .groupby("customer_id")
    .cumcount()
)


# ============================================================
# 3. PREVIOUS MESSAGE
# ============================================================

hist["prev_sent_at"] = (
    hist
    .groupby("customer_id")["sent_at"]
    .shift(1)
)

hist["days_since_prev_msg"] = (
    hist["sent_at"]
    -
    hist["prev_sent_at"]
).dt.total_seconds() / 86400


# ============================================================
# 4. PREVIOUS TEMPLATE / DELIVERY / INTERACTION
# ============================================================

hist["prev_template"] = (
    hist
    .groupby("customer_id")["template"]
    .shift(1)
)

hist["prev_delivery_status"] = (
    hist
    .groupby("customer_id")["delivery_status"]
    .shift(1)
)

hist["prev_interaction"] = (
    hist
    .groupby("customer_id")["interaction"]
    .shift(1)
)


# ============================================================
# 5. PRIOR ENGAGEMENT
#
# IMPORTANT:
# interaction from CURRENT message cannot be used.
#
# We only use interactions from PREVIOUS messages.
# ============================================================

hist["engaged_current"] = (
    hist["interaction"]
    .isin([
        "read",
        "clicked_link",
        "replied"
    ])
    .astype(int)
)

hist["clicked_current"] = (
    hist["interaction"]
    .eq("clicked_link")
    .astype(int)
)

hist["replied_current"] = (
    hist["interaction"]
    .eq("replied")
    .astype(int)
)


hist["had_prior_engagement"] = (
    hist
    .groupby("customer_id")["engaged_current"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cummax()
    )
    .astype(bool)
)


hist["had_prior_click"] = (
    hist
    .groupby("customer_id")["clicked_current"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cummax()
    )
    .astype(bool)
)


hist["had_prior_reply"] = (
    hist
    .groupby("customer_id")["replied_current"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cummax()
    )
    .astype(bool)
)


# ============================================================
# 6. PRIOR PAYMENT
#
# Again: only payments BEFORE current message.
# ============================================================

hist["payment_current"] = (
    hist["amount_paid_brl"] > 0
).astype(int)


hist["had_prior_payment"] = (
    hist
    .groupby("customer_id")["payment_current"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cummax()
    )
    .astype(bool)
)


hist["prior_amount_paid_brl"] = (
    hist
    .groupby("customer_id")["amount_paid_brl"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cumsum()
    )
)


# ============================================================
# 7. PRIOR CONTACT PRESSURE — 14 DAYS
#
# We already have n_msgs_last_14d in the original dataset.
#
# First verify that it is a pre-treatment feature.
# ============================================================

hist["n_msgs_last_14d"] = pd.to_numeric(
    hist["n_msgs_last_14d"],
    errors="coerce"
)


# ============================================================
# 8. KEEP CURRENT SENDS AT DPD 30-45
# ============================================================

decision = (
    hist.loc[
        hist["days_past_due"]
        .between(30, 45)
    ]
    .copy()
)


# ============================================================
# 9. OUTCOME OF CURRENT MESSAGE
# ============================================================

decision["paid_72h"] = (
    decision["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

decision["recovery_72h"] = (
    decision["amount_paid_brl"]
)


# ============================================================
# 10. DPD BUCKET
# ============================================================

decision["dpd_stage"] = pd.cut(
    decision["days_past_due"],
    bins=[
        29,
        34,
        39,
        45
    ],
    labels=[
        "30-34",
        "35-39",
        "40-45"
    ]
)


# ============================================================
# 11. TOTAL PRIOR MESSAGE BUCKET
#
# This is much more informative than only n_msgs_last_14d.
# ============================================================

decision["prior_msgs_bucket"] = pd.cut(
    decision["n_prior_msgs_total"],
    bins=[
        -1,
        2,
        5,
        8,
        np.inf
    ],
    labels=[
        "0-2",
        "3-5",
        "6-8",
        "9+"
    ]
)


# ============================================================
# 12. RECENCY BUCKET
# ============================================================

decision["recency_bucket"] = pd.cut(
    decision["days_since_prev_msg"],
    bins=[
        -np.inf,
        3,
        7,
        14,
        np.inf
    ],
    labels=[
        "0-3d",
        "4-7d",
        "8-14d",
        "15+d"
    ]
)


# ============================================================
# 13. PRIOR ENGAGEMENT PROFILE
#
# Strongest historical signal available BEFORE current send.
# ============================================================

decision["prior_engagement_profile"] = np.select(

    [
        decision["had_prior_click"],
        decision["had_prior_reply"],
        decision["had_prior_engagement"]
    ],

    [
        "prior_click",
        "prior_reply",
        "prior_read_only"
    ],

    default="no_prior_engagement"
)


# ============================================================
# 14. MAIN DECISION TABLE
#
# DPD × cumulative pressure × recency
# ============================================================

decision_table = (
    decision
    .groupby(
        [
            "dpd_stage",
            "prior_msgs_bucket",
            "recency_bucket"
        ],
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        prior_engagement_rate=(
            "had_prior_engagement",
            "mean"
        ),

        prior_payment_rate=(
            "had_prior_payment",
            "mean"
        ),

        avg_msgs_14d=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


decision_table["recovery_per_message"] = (
    decision_table["recovery_brl"]
    /
    decision_table["messages"]
)


decision_table["wa_cost"] = (
    decision_table["messages"]
)


decision_table["observed_net"] = (
    decision_table["recovery_brl"]
    -
    decision_table["wa_cost"]
)


# ============================================================
# 15. SAMPLE SIZE FLAG
# ============================================================

decision_table["sample_flag"] = np.select(

    [
        decision_table["messages"] >= 100,
        decision_table["messages"] >= 50,
        decision_table["messages"] >= 30
    ],

    [
        "GOOD",
        "OK",
        "SMALL"
    ],

    default="VERY_SMALL"
)


# ============================================================
# 16. SORT FOR BUSINESS READING
# ============================================================

decision_table = (
    decision_table
    .sort_values(
        [
            "dpd_stage",
            "prior_msgs_bucket",
            "recency_bucket"
        ]
    )
    .reset_index(drop=True)
)


print("=" * 130)
print("ONE MORE MESSAGE? — DPD 30-45")
print("=" * 130)

display(
    decision_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "observed_net":
            "R$ {:,.2f}",

        "wa_cost":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "prior_engagement_rate":
            "{:.1%}",

        "prior_payment_rate":
            "{:.1%}",

        "avg_msgs_14d":
            "{:.2f}"
    })
)


# ============================================================
# 17. SECOND TABLE:
# PRIOR ENGAGEMENT
#
# This tells us whether previous signs of interest identify
# customers worth another attempt.
# ============================================================

engagement_table = (
    decision
    .groupby(
        [
            "dpd_stage",
            "prior_engagement_profile"
        ],
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        avg_days_since_prev_msg=(
            "days_since_prev_msg",
            "mean"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        )
    )
    .reset_index()
)


engagement_table["recovery_per_message"] = (
    engagement_table["recovery_brl"]
    /
    engagement_table["messages"]
)


print("\n" + "=" * 130)
print("PRIOR ENGAGEMENT → RESPONSE TO ANOTHER MESSAGE")
print("=" * 130)

display(
    engagement_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "avg_prior_msgs":
            "{:.1f}",

        "avg_days_since_prev_msg":
            "{:.1f}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}"
    })
)


# ============================================================
# 18. THIRD TABLE:
# PRIOR PAYMENT
# ============================================================

payment_table = (
    decision
    .groupby(
        [
            "dpd_stage",
            "had_prior_payment"
        ],
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        )
    )
    .reset_index()
)


payment_table["recovery_per_message"] = (
    payment_table["recovery_brl"]
    /
    payment_table["messages"]
)


print("\n" + "=" * 130)
print("PRIOR PAYMENT → RESPONSE TO ANOTHER MESSAGE")
print("=" * 130)

display(
    payment_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "avg_prior_msgs":
            "{:.1f}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}"
    })
)

ONE MORE MESSAGE? — DPD 30-45


,dpd_stage,prior_msgs_bucket,recency_bucket,messages,customers,payment_events,payment_rate,recovery_brl,avg_balance,prior_engagement_rate,prior_payment_rate,avg_msgs_14d,recovery_per_message,wa_cost,observed_net,sample_flag
0,30-34,0-2,0-3d,8,8,0,0.00%,R$ 0.00,R$ 787.45,87.5%,12.5%,1.25,R$ 0.00,R$ 8.00,R$ -8.00,VERY_SMALL
1,30-34,0-2,4-7d,2,2,0,0.00%,R$ 0.00,R$ 680.60,100.0%,0.0%,1.50,R$ 0.00,R$ 2.00,R$ -2.00,VERY_SMALL
2,30-34,0-2,8-14d,14,14,0,0.00%,R$ 0.00,R$ 862.14,57.1%,0.0%,1.07,R$ 0.00,R$ 14.00,R$ -14.00,VERY_SMALL
3,30-34,0-2,15+d,20,20,2,10.00%,R$ 731.32,R$ 860.53,50.0%,10.0%,0.00,R$ 36.57,R$ 20.00,R$ 711.32,VERY_SMALL
4,30-34,3-5,0-3d,176,162,11,6.25%,"R$ 6,071.53",R$ 784.88,85.2%,15.3%,1.95,R$ 34.50,R$ 176.00,"R$ 5,895.53",GOOD
5,30-34,3-5,4-7d,243,239,20,8.23%,"R$ 9,389.27",R$ 768.70,80.7%,16.5%,1.84,R$ 38.64,R$ 243.00,"R$ 9,146.27",GOOD
6,30-34,3-5,8-14d,240,240,23,9.58%,"R$ 11,672.16",R$ 765.36,87.9%,20.8%,1.39,R$ 48.63,R$ 240.00,"R$ 11,432.16",GOOD
7,30-34,3-5,15+d,143,143,13,9.09%,"R$ 7,751.61",R$ 822.75,79.0%,14.0%,0.13,R$ 54.21,R$ 143.00,"R$ 7,608.61",GOOD
8,30-34,6-8,0-3d,494,443,36,7.29%,"R$ 25,637.46",R$ 812.50,87.0%,13.4%,2.93,R$ 51.90,R$ 494.00,"R$ 25,143.46",GOOD
9,30-34,6-8,4-7d,651,635,54,8.29%,"R$ 36,857.68",R$ 795.24,88.0%,17.4%,2.53,R$ 56.62,R$ 651.00,"R$ 36,206.68",GOOD



PRIOR ENGAGEMENT → RESPONSE TO ANOTHER MESSAGE


,dpd_stage,prior_engagement_profile,messages,customers,avg_prior_msgs,avg_days_since_prev_msg,payment_events,payment_rate,recovery_brl,avg_balance,recovery_per_message
0,30-34,no_prior_engagement,441,366,6.6,6.6,11,2.49%,"R$ 6,173.64",R$ 795.23,R$ 14.00
1,30-34,prior_click,"1,517","1,262",7.3,5.6,127,8.37%,"R$ 75,730.48",R$ 816.44,R$ 49.92
2,30-34,prior_read_only,872,715,6.8,5.9,57,6.54%,"R$ 36,254.36",R$ 798.94,R$ 41.58
3,30-34,prior_reply,481,401,7.1,6.1,39,8.11%,"R$ 25,273.04",R$ 794.14,R$ 52.54
4,35-39,no_prior_engagement,324,269,7.2,8.0,8,2.47%,"R$ 5,397.09",R$ 816.24,R$ 16.66
5,35-39,prior_click,"1,247","1,044",7.8,7.4,98,7.86%,"R$ 52,651.95",R$ 768.37,R$ 42.22
6,35-39,prior_read_only,625,536,7.2,8.1,32,5.12%,"R$ 19,418.65",R$ 786.01,R$ 31.07
7,35-39,prior_reply,385,323,7.3,8.0,34,8.83%,"R$ 19,595.26",R$ 782.51,R$ 50.90
8,40-45,no_prior_engagement,328,265,7.7,9.6,9,2.74%,"R$ 5,253.34",R$ 753.97,R$ 16.02
9,40-45,prior_click,"1,367","1,079",8.4,8.5,121,8.85%,"R$ 62,187.34",R$ 766.82,R$ 45.49



PRIOR PAYMENT → RESPONSE TO ANOTHER MESSAGE


,dpd_stage,had_prior_payment,messages,customers,avg_prior_msgs,payment_events,payment_rate,recovery_brl,avg_balance,recovery_per_message
0,30-34,False,"2,843","2,317",7.1,176,6.19%,"R$ 127,267.95",R$ 870.97,R$ 44.77
1,30-34,True,468,389,6.7,58,12.39%,"R$ 16,163.57",R$ 409.67,R$ 34.54
2,35-39,False,"2,169","1,806",7.6,115,5.30%,"R$ 81,691.77",R$ 854.25,R$ 37.66
3,35-39,True,412,344,7.3,57,13.83%,"R$ 15,371.18",R$ 393.90,R$ 37.31
4,40-45,False,"2,224","1,791",8.2,142,6.38%,"R$ 89,908.31",R$ 855.90,R$ 40.43
5,40-45,True,436,339,8.0,61,13.99%,"R$ 15,082.29",R$ 368.43,R$ 34.59


In [7]:
# ============================================================
# SLIDE SUPPORT — EARLY RESCUE ANALYSIS — DPD 30-45
#
# Objective:
# Identify which customers still show historical evidence
# of response to ONE MORE WhatsApp contact in DPD 30-45.
#
# IMPORTANT:
# All segmentation variables are known BEFORE current message.
# Outcomes refer to the CURRENT message.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 0. BASE
# ============================================================

df = decision.copy()


# ------------------------------------------------------------
# Standardize prior engagement
# ------------------------------------------------------------

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)


# ------------------------------------------------------------
# Recent contact
#
# Definition:
# recent = previous WhatsApp <= 7 days ago
#
# We will ALSO keep exact recency bands so we can validate
# whether this threshold is reasonable.
# ------------------------------------------------------------

df["recent_contact_7d"] = (
    df["days_since_prev_msg"] <= 7
)

df["recent_contact_7d"] = (
    df["recent_contact_7d"]
    .fillna(False)
)


# ============================================================
# HELPER
# ============================================================

def summarize(group_cols):

    out = (
        df
        .groupby(
            group_cols,
            observed=True,
            dropna=False
        )
        .agg(
            messages=(
                "customer_id",
                "size"
            ),

            customers=(
                "customer_id",
                "nunique"
            ),

            payment_events=(
                "paid_72h",
                "sum"
            ),

            payment_rate=(
                "paid_72h",
                "mean"
            ),

            recovery_brl=(
                "recovery_72h",
                "sum"
            ),

            outstanding_brl=(
                "outstanding_balance_brl",
                "sum"
            ),

            avg_balance=(
                "outstanding_balance_brl",
                "mean"
            ),

            avg_prior_msgs=(
                "n_prior_msgs_total",
                "mean"
            ),

            avg_days_since_prev_msg=(
                "days_since_prev_msg",
                "mean"
            )
        )
        .reset_index()
    )

    out["recovery_per_message"] = (
        out["recovery_brl"]
        /
        out["messages"]
    )

    out["message_share"] = (
        out["messages"]
        /
        len(df)
    )

    return out


# ============================================================
# TABLE 1
# COMBINATION — ENGAGEMENT × PAYMENT
#
# This should be the FIRST evidence on the slide.
# ============================================================

combo = summarize([
    "has_prior_engagement",
    "has_prior_payment"
])


combo["profile"] = np.select(

    [
        combo["has_prior_engagement"]
        & combo["has_prior_payment"],

        combo["has_prior_engagement"]
        & ~combo["has_prior_payment"],

        ~combo["has_prior_engagement"]
        & combo["has_prior_payment"]
    ],

    [
        "Engaged + prior payment",
        "Engaged + no prior payment",
        "No engagement + prior payment"
    ],

    default="No engagement + no prior payment"
)


combo = (
    combo[
        [
            "profile",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "recovery_brl",
            "avg_balance",
            "avg_prior_msgs",
            "avg_days_since_prev_msg",
            "message_share"
        ]
    ]
    .sort_values(
        "payment_rate",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 120)
print("TABLE 1 — COMBINED WILLINGNESS SIGNAL")
print("ENGAGEMENT × PRIOR PAYMENT")
print("=" * 120)

display(
    combo.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.1f}",
        "avg_days_since_prev_msg": "{:.1f}",
        "message_share": "{:.1%}"
    })
)


# ============================================================
# TABLE 2
# COMBINATION × DPD
#
# Critical:
# Does the combined signal survive across DPD 40-45?
# ============================================================

combo_dpd = summarize([
    "dpd_stage",
    "has_prior_engagement",
    "has_prior_payment"
])


combo_dpd["profile"] = np.select(

    [
        combo_dpd["has_prior_engagement"]
        & combo_dpd["has_prior_payment"],

        combo_dpd["has_prior_engagement"]
        & ~combo_dpd["has_prior_payment"],

        ~combo_dpd["has_prior_engagement"]
        & combo_dpd["has_prior_payment"]
    ],

    [
        "Engaged + prior payment",
        "Engaged + no prior payment",
        "No engagement + prior payment"
    ],

    default="No engagement + no prior payment"
)


combo_dpd = combo_dpd[
    [
        "dpd_stage",
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "avg_balance"
    ]
]


print("\n" + "=" * 120)
print("TABLE 2 — COMBINED SIGNAL × DPD")
print("=" * 120)

display(
    combo_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# TABLE 3
# ENGAGEMENT ALONE
# ============================================================

engagement = summarize([
    "has_prior_engagement"
])


engagement["profile"] = np.where(
    engagement["has_prior_engagement"],
    "Prior engagement",
    "No prior engagement"
)


engagement = engagement[
    [
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "recovery_brl",
        "avg_balance"
    ]
]


print("\n" + "=" * 120)
print("TABLE 3 — PRIOR ENGAGEMENT")
print("=" * 120)

display(
    engagement.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# TABLE 4
# PRIOR PAYMENT ALONE
# ============================================================

payment = summarize([
    "has_prior_payment"
])


payment["profile"] = np.where(
    payment["has_prior_payment"],
    "Prior payment",
    "No prior payment"
)


payment = payment[
    [
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "recovery_brl",
        "avg_balance"
    ]
]


print("\n" + "=" * 120)
print("TABLE 4 — PRIOR PAYMENT")
print("=" * 120)

display(
    payment.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# TABLE 5
# RECENCY ALONE
#
# First show granular recency.
# Do NOT jump directly to <=7 vs >7.
# ============================================================

recency = summarize([
    "recency_bucket"
])


recency = recency[
    [
        "recency_bucket",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "recovery_brl",
        "avg_balance",
        "avg_prior_msgs"
    ]
]


print("\n" + "=" * 120)
print("TABLE 5 — DAYS SINCE PREVIOUS CONTACT")
print("=" * 120)

display(
    recency.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.1f}"
    })
)


# ============================================================
# TABLE 6
# RECENT vs NOT RECENT
# ============================================================

recent = summarize([
    "recent_contact_7d"
])


recent["profile"] = np.where(
    recent["recent_contact_7d"],
    "Recent contact <=7d",
    "No recent contact >7d"
)


recent = recent[
    [
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "recovery_brl",
        "avg_balance",
        "avg_prior_msgs"
    ]
]


print("\n" + "=" * 120)
print("TABLE 6 — RECENT CONTACT <=7 DAYS")
print("=" * 120)

display(
    recent.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.1f}"
    })
)


# ============================================================
# TABLE 7
# ALL THREE SIGNALS TOGETHER
#
# Engagement × payment × recent contact
#
# This is the candidate RULE HIERARCHY table.
# ============================================================

all_signals = summarize([
    "has_prior_engagement",
    "has_prior_payment",
    "recent_contact_7d"
])


all_signals["profile"] = (
    np.where(
        all_signals["has_prior_engagement"],
        "Engaged",
        "No engagement"
    )
    + " | "
    +
    np.where(
        all_signals["has_prior_payment"],
        "Prior payment",
        "No payment"
    )
    + " | "
    +
    np.where(
        all_signals["recent_contact_7d"],
        "Contact <=7d",
        "Contact >7d"
    )
)


all_signals = (
    all_signals[
        [
            "profile",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "recovery_brl",
            "avg_balance",
            "avg_prior_msgs",
            "avg_days_since_prev_msg"
        ]
    ]
    .sort_values(
        [
            "payment_rate",
            "recovery_per_message"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 120)
print("TABLE 7 — ALL SIGNALS COMBINED")
print("ENGAGEMENT × PAYMENT × RECENCY")
print("=" * 120)

display(
    all_signals.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "recovery_brl": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}",
        "avg_prior_msgs": "{:.1f}",
        "avg_days_since_prev_msg": "{:.1f}"
    })
)


# ============================================================
# TABLE 8
# ALL SIGNALS × DPD
#
# This is our final robustness check before choosing timing.
# ============================================================

all_signals_dpd = summarize([
    "dpd_stage",
    "has_prior_engagement",
    "has_prior_payment",
    "recent_contact_7d"
])


all_signals_dpd["profile"] = (
    np.where(
        all_signals_dpd["has_prior_engagement"],
        "Engaged",
        "No engagement"
    )
    + " | "
    +
    np.where(
        all_signals_dpd["has_prior_payment"],
        "Prior payment",
        "No payment"
    )
    + " | "
    +
    np.where(
        all_signals_dpd["recent_contact_7d"],
        "Contact <=7d",
        "Contact >7d"
    )
)


all_signals_dpd = all_signals_dpd[
    [
        "dpd_stage",
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "avg_balance"
    ]
]


print("\n" + "=" * 120)
print("TABLE 8 — ALL SIGNALS × DPD")
print("=" * 120)

display(
    all_signals_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# QA / RECONCILIATION
# ============================================================

print("\n" + "=" * 120)
print("QA")
print("=" * 120)

print(f"Historical DPD30-45 messages : {len(df):,}")
print(f"Unique customers             : {df['customer_id'].nunique():,}")
print(f"Payment events               : {df['paid_72h'].sum():,}")
print(f"Recovery                     : R$ {df['recovery_72h'].sum():,.2f}")

print("\nCombination table messages:")
print(f"{combo['messages'].sum():,}")

print("\nAll-signals table messages:")
print(f"{all_signals['messages'].sum():,}")

assert combo["messages"].sum() == len(df)
assert all_signals["messages"].sum() == len(df)

print("\n✓ ALL HISTORICAL MESSAGES RECONCILED")

TABLE 1 — COMBINED WILLINGNESS SIGNAL
ENGAGEMENT × PRIOR PAYMENT


,profile,messages,customers,payment_events,payment_rate,recovery_per_message,recovery_brl,avg_balance,avg_prior_msgs,avg_days_since_prev_msg,message_share
0,Engaged + prior payment,"1,272",702,172,13.52%,R$ 35.64,"R$ 45,337.54",R$ 392.48,7.4,7.2,14.9%
1,No engagement + prior payment,44,29,4,9.09%,R$ 29.08,"R$ 1,279.50",R$ 350.37,5.4,9.3,0.5%
2,Engaged + no prior payment,"6,187","3,363",409,6.61%,R$ 45.79,"R$ 283,323.46",R$ 870.45,7.7,7.4,72.3%
3,No engagement + no prior payment,"1,049",588,24,2.29%,R$ 14.82,"R$ 15,544.57",R$ 807.48,7.2,7.9,12.3%



TABLE 2 — COMBINED SIGNAL × DPD


,dpd_stage,profile,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance
0,30-34,No engagement + no prior payment,423,351,9,2.13%,R$ 12.26,R$ 812.92
1,30-34,No engagement + prior payment,18,15,2,11.11%,R$ 54.84,R$ 379.48
2,30-34,Engaged + no prior payment,"2,420","1,978",167,6.90%,R$ 50.45,R$ 881.11
3,30-34,Engaged + prior payment,450,375,56,12.44%,R$ 33.73,R$ 410.87
4,35-39,No engagement + no prior payment,310,257,6,1.94%,R$ 16.47,R$ 838.63
5,35-39,No engagement + prior payment,14,12,2,14.29%,R$ 20.88,R$ 320.58
6,35-39,Engaged + no prior payment,"1,859","1,553",109,5.86%,R$ 41.20,R$ 856.85
7,35-39,Engaged + prior payment,398,333,55,13.82%,R$ 37.89,R$ 396.48
8,40-45,No engagement + no prior payment,316,255,9,2.85%,R$ 16.62,R$ 769.64
9,40-45,No engagement + prior payment,12,10,0,0.00%,R$ 0.00,R$ 341.44



TABLE 3 — PRIOR ENGAGEMENT


...


TABLE 7 — ALL SIGNALS COMBINED
ENGAGEMENT × PAYMENT × RECENCY


,profile,messages,customers,payment_events,payment_rate,recovery_per_message,recovery_brl,avg_balance,avg_prior_msgs,avg_days_since_prev_msg
0,Engaged | Prior payment | Contact >7d,538,497,78,14.50%,R$ 38.11,"R$ 20,501.67",R$ 384.85,6.8,12.0
1,Engaged | Prior payment | Contact <=7d,734,450,94,12.81%,R$ 33.84,"R$ 24,835.87",R$ 398.07,7.8,3.6
2,No engagement | Prior payment | Contact >7d,21,20,2,9.52%,R$ 13.92,R$ 292.29,R$ 322.52,4.8,15.3
3,No engagement | Prior payment | Contact <=7d,23,18,2,8.70%,R$ 42.92,R$ 987.21,R$ 375.79,6.0,3.8
4,Engaged | No payment | Contact <=7d,"3,561","2,155",236,6.63%,R$ 46.71,"R$ 166,351.75",R$ 871.76,8.1,3.5
5,Engaged | No payment | Contact >7d,"2,626","2,379",173,6.59%,R$ 44.54,"R$ 116,971.71",R$ 868.67,7.1,12.6
6,No engagement | No payment | Contact >7d,475,427,17,3.58%,R$ 22.86,"R$ 10,858.39",R$ 832.36,6.4,13.2
7,No engagement | No payment | Contact <=7d,574,354,7,1.22%,R$ 8.16,"R$ 4,686.18",R$ 786.89,7.8,3.5



TABLE 8 — ALL SIGNALS × DPD


,dpd_stage,profile,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance
0,30-34,No engagement | No payment | Contact >7d,145,145,6,4.14%,R$ 25.12,R$ 833.44
1,30-34,No engagement | No payment | Contact <=7d,278,228,3,1.08%,R$ 5.56,R$ 802.21
2,30-34,No engagement | Prior payment | Contact >7d,5,5,0,0.00%,R$ 0.00,R$ 345.70
3,30-34,No engagement | Prior payment | Contact <=7d,13,12,2,15.38%,R$ 75.94,R$ 392.47
4,30-34,Engaged | No payment | Contact >7d,721,721,46,6.38%,R$ 43.41,R$ 874.75
5,30-34,Engaged | No payment | Contact <=7d,"1,699","1,391",121,7.12%,R$ 53.43,R$ 883.82
6,30-34,Engaged | Prior payment | Contact >7d,143,143,18,12.59%,R$ 34.11,R$ 409.32
7,30-34,Engaged | Prior payment | Contact <=7d,307,250,38,12.38%,R$ 33.54,R$ 411.60
8,35-39,No engagement | No payment | Contact >7d,153,153,3,1.96%,R$ 14.22,R$ 880.84
9,35-39,No engagement | No payment | Contact <=7d,157,133,3,1.91%,R$ 18.66,R$ 797.49



QA
Historical DPD30-45 messages : 8,552
Unique customers             : 4,581
Payment events               : 609
Recovery                     : R$ 345,485.07

Combination table messages:
8,552

All-signals table messages:
8,552

✓ ALL HISTORICAL MESSAGES RECONCILED


In [8]:
# ============================================================
# VALIDATION:
# PRIOR ENGAGEMENT × CONTACT PRESSURE / TIME WITHOUT STIMULUS
#
# Question:
# Does "no prior engagement" remain a bad segment even when
# the customer has NOT been stimulated recently?
#
# Grain:
# 1 row = historical send at DPD 30-45
# ============================================================

import pandas as pd
import numpy as np


df = decision.copy()


# ============================================================
# 1. STANDARDIZE VARIABLES
# ============================================================

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)

df["n_msgs_last_14d"] = pd.to_numeric(
    df["n_msgs_last_14d"],
    errors="coerce"
)

df["days_since_prev_msg"] = pd.to_numeric(
    df["days_since_prev_msg"],
    errors="coerce"
)


# ============================================================
# 2. CONTACT PRESSURE
#
# Most important split:
#
# 0 messages in last 14d
# versus
# >=1 message in last 14d
# ============================================================

df["no_msg_last_14d"] = (
    df["n_msgs_last_14d"].eq(0)
)


# More granular version

df["pressure_14d"] = pd.cut(
    df["n_msgs_last_14d"],
    bins=[
        -1,
        0,
        2,
        4,
        np.inf
    ],
    labels=[
        "0 msgs",
        "1-2 msgs",
        "3-4 msgs",
        "5+ msgs"
    ]
)


# ============================================================
# 3. HELPER
# ============================================================

def make_summary(data, group_cols):

    out = (
        data
        .groupby(
            group_cols,
            observed=True,
            dropna=False
        )
        .agg(

            messages=(
                "customer_id",
                "size"
            ),

            customers=(
                "customer_id",
                "nunique"
            ),

            payment_events=(
                "paid_72h",
                "sum"
            ),

            payment_rate=(
                "paid_72h",
                "mean"
            ),

            recovery_brl=(
                "recovery_72h",
                "sum"
            ),

            avg_balance=(
                "outstanding_balance_brl",
                "mean"
            ),

            avg_prior_msgs_total=(
                "n_prior_msgs_total",
                "mean"
            ),

            avg_msgs_last_14d=(
                "n_msgs_last_14d",
                "mean"
            ),

            avg_days_since_prev_msg=(
                "days_since_prev_msg",
                "mean"
            ),

            prior_payment_rate=(
                "has_prior_payment",
                "mean"
            )
        )
        .reset_index()
    )

    out["recovery_per_message"] = (
        out["recovery_brl"]
        /
        out["messages"]
    )

    return out


# ============================================================
# TABLE A
# ENGAGEMENT × ZERO CONTACTS LAST 14D
#
# THIS IS THE MAIN TABLE.
# ============================================================

table_a = make_summary(
    df,
    [
        "has_prior_engagement",
        "no_msg_last_14d"
    ]
)


table_a["profile"] = np.select(

    [
        table_a["has_prior_engagement"]
        & table_a["no_msg_last_14d"],

        table_a["has_prior_engagement"]
        & ~table_a["no_msg_last_14d"],

        ~table_a["has_prior_engagement"]
        & table_a["no_msg_last_14d"]
    ],

    [
        "Engaged | 0 msgs last 14d",
        "Engaged | >=1 msg last 14d",
        "No engagement | 0 msgs last 14d"
    ],

    default="No engagement | >=1 msg last 14d"
)


table_a = (
    table_a[
        [
            "profile",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "recovery_brl",
            "avg_balance",
            "avg_prior_msgs_total",
            "avg_days_since_prev_msg",
            "prior_payment_rate"
        ]
    ]
    .sort_values(
        "payment_rate",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 120)
print("TABLE A — ENGAGEMENT × CONTACT IN LAST 14 DAYS")
print("=" * 120)

display(
    table_a.style.format({

        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",

        "payment_rate": "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "recovery_brl":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs_total":
            "{:.1f}",

        "avg_days_since_prev_msg":
            "{:.1f}",

        "prior_payment_rate":
            "{:.1%}"
    })
)


# ============================================================
# TABLE B
# ENGAGEMENT × FULL PRESSURE BUCKET
#
# Shows deterioration as pressure increases.
# ============================================================

table_b = make_summary(
    df,
    [
        "has_prior_engagement",
        "pressure_14d"
    ]
)


table_b["engagement"] = np.where(
    table_b["has_prior_engagement"],
    "Prior engagement",
    "No prior engagement"
)


table_b = table_b[
    [
        "engagement",
        "pressure_14d",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "avg_balance",
        "avg_prior_msgs_total"
    ]
]


print("\n" + "=" * 120)
print("TABLE B — ENGAGEMENT × CONTACT PRESSURE")
print("=" * 120)

display(
    table_b.style.format({

        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs_total":
            "{:.1f}"
    })
)


# ============================================================
# TABLE C
# ENGAGEMENT × 0 MSG LAST 14D × PRIOR PAYMENT
#
# This tests whether prior payment can "rescue" a customer
# without previous engagement.
# ============================================================

table_c = make_summary(
    df,
    [
        "has_prior_engagement",
        "has_prior_payment",
        "no_msg_last_14d"
    ]
)


table_c["profile"] = (
    np.where(
        table_c["has_prior_engagement"],
        "Engaged",
        "No engagement"
    )
    + " | "
    +
    np.where(
        table_c["has_prior_payment"],
        "Prior payment",
        "No prior payment"
    )
    + " | "
    +
    np.where(
        table_c["no_msg_last_14d"],
        "0 msgs 14d",
        ">=1 msg 14d"
    )
)


table_c = (
    table_c[
        [
            "profile",
            "messages",
            "customers",
            "payment_events",
            "payment_rate",
            "recovery_per_message",
            "avg_balance",
            "avg_prior_msgs_total",
            "avg_days_since_prev_msg"
        ]
    ]
    .sort_values(
        "payment_rate",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 120)
print("TABLE C — ENGAGEMENT × PAYMENT × 14D PRESSURE")
print("=" * 120)

display(
    table_c.style.format({

        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs_total":
            "{:.1f}",

        "avg_days_since_prev_msg":
            "{:.1f}"
    })
)


# ============================================================
# TABLE D
# MOST IMPORTANT ROBUSTNESS CHECK:
#
# ENGAGEMENT × ZERO MSG 14D × DPD
#
# Does the "reactivation" effect survive in 30-34,
# 35-39 and 40-45?
# ============================================================

table_d = make_summary(
    df,
    [
        "dpd_stage",
        "has_prior_engagement",
        "no_msg_last_14d"
    ]
)


table_d["profile"] = (
    np.where(
        table_d["has_prior_engagement"],
        "Engaged",
        "No engagement"
    )
    + " | "
    +
    np.where(
        table_d["no_msg_last_14d"],
        "0 msgs 14d",
        ">=1 msg 14d"
    )
)


table_d = table_d[
    [
        "dpd_stage",
        "profile",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "avg_balance",
        "avg_prior_msgs_total"
    ]
]


print("\n" + "=" * 120)
print("TABLE D — ENGAGEMENT × 14D PRESSURE × DPD")
print("=" * 120)

display(
    table_d.style.format({

        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs_total":
            "{:.1f}"
    })
)


# ============================================================
# QA
# ============================================================

print("\n" + "=" * 120)
print("QA")
print("=" * 120)

print(
    f"DPD30-45 messages : "
    f"{len(df):,}"
)

print(
    f"0 msgs last 14d   : "
    f"{df['no_msg_last_14d'].sum():,}"
)

print(
    f">=1 msg last 14d  : "
    f"{(~df['no_msg_last_14d']).sum():,}"
)

print(
    f"No engagement     : "
    f"{(~df['has_prior_engagement']).sum():,}"
)

print(
    f"Prior engagement  : "
    f"{df['has_prior_engagement'].sum():,}"
)

assert table_a["messages"].sum() == len(df)
assert table_b["messages"].sum() == len(df)
assert table_c["messages"].sum() == len(df)
assert table_d["messages"].sum() == len(df)

print("\n✓ ALL TABLES RECONCILED")

TABLE A — ENGAGEMENT × CONTACT IN LAST 14 DAYS


,profile,messages,customers,payment_events,payment_rate,recovery_per_message,recovery_brl,avg_balance,avg_prior_msgs_total,avg_days_since_prev_msg,prior_payment_rate
0,Engaged | 0 msgs last 14d,839,839,75,8.94%,R$ 41.57,"R$ 34,879.53",R$ 790.32,6.1,19.1,15.6%
1,Engaged | >=1 msg last 14d,"6,620","3,492",506,7.64%,R$ 44.38,"R$ 293,781.47",R$ 788.77,7.8,5.8,17.2%
2,No engagement | 0 msgs last 14d,164,163,11,6.71%,R$ 38.89,"R$ 6,378.38",R$ 780.22,5.0,19.7,7.3%
3,No engagement | >=1 msg last 14d,929,514,17,1.83%,R$ 11.24,"R$ 10,445.69",R$ 790.64,7.5,5.9,3.4%



TABLE B — ENGAGEMENT × CONTACT PRESSURE


,engagement,pressure_14d,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance,avg_prior_msgs_total
0,No prior engagement,0 msgs,164,163,11,6.71%,R$ 38.89,R$ 780.22,5.0
1,No prior engagement,1-2 msgs,605,414,15,2.48%,R$ 15.63,R$ 794.19,6.8
2,No prior engagement,3-4 msgs,289,182,2,0.69%,R$ 3.42,R$ 757.93,8.6
3,No prior engagement,5+ msgs,35,24,0,0.00%,R$ 0.00,R$ 999.32,10.3
4,Prior engagement,0 msgs,839,839,75,8.94%,R$ 41.57,R$ 790.32,6.1
5,Prior engagement,1-2 msgs,"4,168","2,735",341,8.18%,R$ 45.68,R$ 779.94,7.2
6,Prior engagement,3-4 msgs,"2,132","1,382",150,7.04%,R$ 43.74,R$ 800.56,8.7
7,Prior engagement,5+ msgs,320,229,15,4.69%,R$ 31.68,R$ 825.20,10.6



TABLE C — ENGAGEMENT × PAYMENT × 14D PRESSURE


,profile,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance,avg_prior_msgs_total,avg_days_since_prev_msg
0,Engaged | Prior payment | 0 msgs 14d,131,131,25,19.08%,R$ 39.59,R$ 348.04,5.8,18.1
1,No engagement | Prior payment | 0 msgs 14d,12,12,2,16.67%,R$ 24.36,R$ 281.22,4.2,20.4
2,Engaged | Prior payment | >=1 msg 14d,"1,141",618,147,12.88%,R$ 35.19,R$ 397.58,7.5,5.9
3,Engaged | No prior payment | 0 msgs 14d,708,708,50,7.06%,R$ 41.94,R$ 872.15,6.1,19.2
4,Engaged | No prior payment | >=1 msg 14d,"5,479","2,911",359,6.55%,R$ 46.29,R$ 870.23,7.9,5.8
5,No engagement | Prior payment | >=1 msg 14d,32,19,2,6.25%,R$ 30.85,R$ 376.30,5.9,5.1
6,No engagement | No prior payment | 0 msgs 14d,152,151,9,5.92%,R$ 40.04,R$ 819.61,5.1,19.6
7,No engagement | No prior payment | >=1 msg 14d,897,495,15,1.67%,R$ 10.54,R$ 805.42,7.5,5.9



TABLE D — ENGAGEMENT × 14D PRESSURE × DPD


,dpd_stage,profile,messages,customers,payment_events,payment_rate,recovery_per_message,avg_balance,avg_prior_msgs_total
0,30-34,No engagement | >=1 msg 14d,393,322,9,2.29%,R$ 14.00,R$ 782.81,6.9
1,30-34,No engagement | 0 msgs 14d,48,48,2,4.17%,R$ 13.96,R$ 896.90,4.0
2,30-34,Engaged | >=1 msg 14d,"2,709","2,217",209,7.72%,R$ 48.26,R$ 808.32,7.3
3,30-34,Engaged | 0 msgs 14d,161,161,14,8.70%,R$ 40.55,R$ 791.69,4.8
4,35-39,No engagement | >=1 msg 14d,281,232,4,1.42%,R$ 11.47,R$ 832.64,7.6
5,35-39,No engagement | 0 msgs 14d,43,43,4,9.30%,R$ 50.57,R$ 709.12,4.8
6,35-39,Engaged | >=1 msg 14d,"2,021","1,685",141,6.98%,R$ 40.66,R$ 777.85,7.8
7,35-39,Engaged | 0 msgs 14d,236,236,23,9.75%,R$ 40.18,R$ 757.00,5.6
8,40-45,No engagement | >=1 msg 14d,255,209,4,1.57%,R$ 6.74,R$ 756.44,8.2
9,40-45,No engagement | 0 msgs 14d,73,73,5,6.85%,R$ 48.41,R$ 745.37,5.8



QA
DPD30-45 messages : 8,552
0 msgs last 14d   : 1,003
>=1 msg last 14d  : 7,549
No engagement     : 1,093
Prior engagement  : 7,459

✓ ALL TABLES RECONCILED


In [10]:
# ============================================================
# FINAL RULE TABLE — ONE MORE MESSAGE DPD30-45
#
# Mutually exclusive rules
# Comparison against historical DPD30-45 baseline
#
# IMPORTANT:
# "uplift" below is OBSERVED association, NOT causal incrementality.
# ============================================================

import pandas as pd
import numpy as np

df = decision.copy()


# ============================================================
# 1. STANDARDIZE SIGNALS
# ============================================================

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)

df["n_msgs_last_14d"] = pd.to_numeric(
    df["n_msgs_last_14d"],
    errors="coerce"
)

df["no_msg_last_14d"] = (
    df["n_msgs_last_14d"].eq(0)
)


# ============================================================
# 2. BASELINE — ALL DPD30-45 MESSAGES
# ============================================================

BASELINE_MESSAGES = len(df)

BASELINE_PAYMENT_RATE = (
    df["paid_72h"].mean()
)

BASELINE_RECOVERY = (
    df["recovery_72h"].sum()
)

BASELINE_RECOVERY_PER_MSG = (
    BASELINE_RECOVERY
    / BASELINE_MESSAGES
)

BASELINE_COST_PER_MSG = 1.0

BASELINE_NET_PER_MSG = (
    BASELINE_RECOVERY_PER_MSG
    - BASELINE_COST_PER_MSG
)


print("=" * 100)
print("HISTORICAL BASELINE — DPD30-45")
print("=" * 100)

print(
    f"Messages              : "
    f"{BASELINE_MESSAGES:,}"
)

print(
    f"Payment rate          : "
    f"{BASELINE_PAYMENT_RATE:.2%}"
)

print(
    f"Recovery / message    : "
    f"R$ {BASELINE_RECOVERY_PER_MSG:,.2f}"
)

print(
    f"Net recovery / message: "
    f"R$ {BASELINE_NET_PER_MSG:,.2f}"
)


# ============================================================
# 3. MUTUALLY EXCLUSIVE RULES
# ============================================================

conditions = [

    # R1
    (
        df["has_prior_engagement"]
        & df["has_prior_payment"]
        & df["no_msg_last_14d"]
    ),

    # R2
    (
        df["has_prior_engagement"]
        & df["has_prior_payment"]
        & ~df["no_msg_last_14d"]
    ),

    # R3
    (
        df["has_prior_engagement"]
        & ~df["has_prior_payment"]
        & df["no_msg_last_14d"]
    ),

    # R4
    (
        df["has_prior_engagement"]
        & ~df["has_prior_payment"]
        & ~df["no_msg_last_14d"]
    ),

    # R5
    (
        ~df["has_prior_engagement"]
        & df["no_msg_last_14d"]
    ),

    # R6
    (
        ~df["has_prior_engagement"]
        & ~df["no_msg_last_14d"]
    )
]


labels = [

    "R1 | Engaged + prior payment + 0 msgs/14d",

    "R2 | Engaged + prior payment + recent contact",

    "R3 | Engaged + no prior payment + 0 msgs/14d",

    "R4 | Engaged + no prior payment + recent contact",

    "R5 | No engagement + 0 msgs/14d",

    "R6 | No engagement + recent contact"
]


df["rule"] = np.select(
    conditions,
    labels,
    default="UNCLASSIFIED"
)


# ============================================================
# 4. AGGREGATE
# ============================================================

rule_table = (
    df
    .groupby(
        "rule",
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        avg_msgs_last_14d=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 5. ECONOMICS
# ============================================================

rule_table["recovery_per_message"] = (
    rule_table["recovery_brl"]
    /
    rule_table["messages"]
)


# WhatsApp = R$1 per attempt

rule_table["cost_brl"] = (
    rule_table["messages"]
    * 1.0
)


rule_table["net_recovery_brl"] = (
    rule_table["recovery_brl"]
    -
    rule_table["cost_brl"]
)


rule_table["net_per_message"] = (
    rule_table["recovery_per_message"]
    - 1
)


# ============================================================
# 6. COMPARISON AGAINST BASELINE
# ============================================================

rule_table["delta_payment_rate_pp"] = (
    (
        rule_table["payment_rate"]
        - BASELINE_PAYMENT_RATE
    )
    * 100
)


rule_table["delta_recovery_per_msg"] = (
    rule_table["recovery_per_message"]
    - BASELINE_RECOVERY_PER_MSG
)


rule_table["recovery_index"] = (
    rule_table["recovery_per_message"]
    /
    BASELINE_RECOVERY_PER_MSG
    * 100
)


# Example:
# 138 = R$/msg is 38% above historical baseline


rule_table["recovery_uplift_pct"] = (
    (
        rule_table["recovery_per_message"]
        /
        BASELINE_RECOVERY_PER_MSG
    )
    - 1
)


# ============================================================
# 7. ORDER BY BUSINESS RULE
# ============================================================

rule_order = {
    label: i + 1
    for i, label in enumerate(labels)
}

rule_table["priority"] = (
    rule_table["rule"]
    .map(rule_order)
)


rule_table = (
    rule_table
    .sort_values("priority")
    .reset_index(drop=True)
)


# ============================================================
# 8. BUSINESS DECISION LABEL
#
# Deliberately descriptive.
# We will validate thresholds from the output.
# ============================================================

rule_table["candidate_action"] = np.select(

    [
        rule_table["priority"].isin([1, 2, 3]),

        rule_table["priority"].eq(4),

        rule_table["priority"].eq(5),

        rule_table["priority"].eq(6)
    ],

    [
        "PRIORITIZE",
        "CONTACT / LOWER PRIORITY",
        "TEST / REACTIVATION",
        "SUPPRESS"
    ],

    default="REVIEW"
)


# ============================================================
# 9. FINAL SLIDE TABLE
# ============================================================

slide_table = rule_table[
    [
        "priority",
        "rule",
        "messages",
        "customers",
        "payment_events",
        "payment_rate",
        "recovery_per_message",
        "delta_recovery_per_msg",
        "recovery_uplift_pct",
        "recovery_index",
        "net_per_message",
        "avg_balance",
        "candidate_action"
    ]
].copy()


print("\n" + "=" * 130)
print("FINAL RULE TABLE — HISTORICAL PERFORMANCE")
print("=" * 130)

display(
    slide_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "delta_recovery_per_msg":
            "R$ {:+,.2f}",

        "recovery_uplift_pct":
            "{:+.1%}",

        "recovery_index":
            "{:.0f}",

        "net_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}"
    })
)


# ============================================================
# 10. QA
# ============================================================

print("\n" + "=" * 130)
print("QA")
print("=" * 130)

print(
    f"Total historical messages : "
    f"{len(df):,}"
)

print(
    f"Messages classified       : "
    f"{rule_table['messages'].sum():,}"
)

print(
    f"Unclassified              : "
    f"{(df['rule'] == 'UNCLASSIFIED').sum():,}"
)

print(
    f"\nBaseline payment rate     : "
    f"{BASELINE_PAYMENT_RATE:.2%}"
)

print(
    f"Baseline recovery/msg     : "
    f"R$ {BASELINE_RECOVERY_PER_MSG:,.2f}"
)

assert (
    rule_table["messages"].sum()
    == len(df)
)

assert (
    (df["rule"] == "UNCLASSIFIED").sum()
    == 0
)

print("\n✓ ALL MESSAGES CLASSIFIED")

HISTORICAL BASELINE — DPD30-45
Messages              : 8,552
Payment rate          : 7.12%
Recovery / message    : R$ 40.40
Net recovery / message: R$ 39.40

FINAL RULE TABLE — HISTORICAL PERFORMANCE


,priority,rule,messages,customers,payment_events,payment_rate,recovery_per_message,delta_recovery_per_msg,recovery_uplift_pct,recovery_index,net_per_message,avg_balance,candidate_action
0,1,R1 | Engaged + prior payment + 0 msgs/14d,131,131,25,19.08%,R$ 39.59,R$ -0.81,-2.0%,98,R$ 38.59,R$ 348.04,PRIORITIZE
1,2,R2 | Engaged + prior payment + recent contact,"1,141",618,147,12.88%,R$ 35.19,R$ -5.21,-12.9%,87,R$ 34.19,R$ 397.58,PRIORITIZE
2,3,R3 | Engaged + no prior payment + 0 msgs/14d,708,708,50,7.06%,R$ 41.94,R$ +1.54,+3.8%,104,R$ 40.94,R$ 872.15,PRIORITIZE
3,4,R4 | Engaged + no prior payment + recent contact,"5,479","2,911",359,6.55%,R$ 46.29,R$ +5.89,+14.6%,115,R$ 45.29,R$ 870.23,CONTACT / LOWER PRIORITY
4,5,R5 | No engagement + 0 msgs/14d,164,163,11,6.71%,R$ 38.89,R$ -1.51,-3.7%,96,R$ 37.89,R$ 780.22,TEST / REACTIVATION
5,6,R6 | No engagement + recent contact,929,514,17,1.83%,R$ 11.24,R$ -29.15,-72.2%,28,R$ 10.24,R$ 790.64,SUPPRESS



QA
Total historical messages : 8,552
Messages classified       : 8,552
Unclassified              : 0

Baseline payment rate     : 7.12%
Baseline recovery/msg     : R$ 40.40

✓ ALL MESSAGES CLASSIFIED


In [11]:
# ============================================================
# FINAL POLICY VALIDATION
# SELECTED RULES (R1 + R3 + R4) vs REST
#
# Goal:
# Validate whether the proposed policy concentrates
# higher observed recovery per message.
#
# IMPORTANT:
# Historical association — NOT causal incrementality.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. BASE
# ============================================================

df = decision.copy()


# ------------------------------------------------------------
# Standardize signals
# ------------------------------------------------------------

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)

df["n_msgs_last_14d"] = pd.to_numeric(
    df["n_msgs_last_14d"],
    errors="coerce"
)

df["no_msg_last_14d"] = (
    df["n_msgs_last_14d"].eq(0)
)


# ============================================================
# 2. RECREATE THE SIX MUTUALLY EXCLUSIVE RULES
# ============================================================

conditions = [

    # R1
    (
        df["has_prior_engagement"]
        & df["has_prior_payment"]
        & df["no_msg_last_14d"]
    ),

    # R2
    (
        df["has_prior_engagement"]
        & df["has_prior_payment"]
        & ~df["no_msg_last_14d"]
    ),

    # R3
    (
        df["has_prior_engagement"]
        & ~df["has_prior_payment"]
        & df["no_msg_last_14d"]
    ),

    # R4
    (
        df["has_prior_engagement"]
        & ~df["has_prior_payment"]
        & ~df["no_msg_last_14d"]
    ),

    # R5
    (
        ~df["has_prior_engagement"]
        & df["no_msg_last_14d"]
    ),

    # R6
    (
        ~df["has_prior_engagement"]
        & ~df["no_msg_last_14d"]
    )
]


labels = [
    "R1",
    "R2",
    "R3",
    "R4",
    "R5",
    "R6"
]


df["rule"] = np.select(
    conditions,
    labels,
    default="UNCLASSIFIED"
)


# ============================================================
# 3. FINAL POLICY
#
# Selected:
# R1 + R3 + R4
#
# Rest:
# R2 + R5 + R6
# ============================================================

selected_rules = ["R1", "R3", "R4"]

df["policy_group"] = np.where(
    df["rule"].isin(selected_rules),
    "SEND — R1 + R3 + R4",
    "REST — R2 + R5 + R6"
)


# ============================================================
# 4. AGGREGATE
# ============================================================

policy_table = (
    df
    .groupby(
        "policy_group",
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        avg_msgs_last_14d=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 5. ECONOMICS
# ============================================================

policy_table["recovery_per_message"] = (
    policy_table["recovery_brl"]
    /
    policy_table["messages"]
)


policy_table["cost_brl"] = (
    policy_table["messages"]
    * 1
)


policy_table["net_recovery_brl"] = (
    policy_table["recovery_brl"]
    -
    policy_table["cost_brl"]
)


policy_table["net_per_message"] = (
    policy_table["recovery_per_message"]
    - 1
)


# ============================================================
# 6. POPULATION / RECOVERY SHARES
# ============================================================

policy_table["message_share"] = (
    policy_table["messages"]
    /
    policy_table["messages"].sum()
)


policy_table["recovery_share"] = (
    policy_table["recovery_brl"]
    /
    policy_table["recovery_brl"].sum()
)


# ============================================================
# 7. FINAL TABLE
# ============================================================

policy_table = policy_table[
    [
        "policy_group",
        "messages",
        "customers",
        "message_share",
        "payment_events",
        "payment_rate",
        "recovery_brl",
        "recovery_share",
        "recovery_per_message",
        "net_per_message",
        "avg_balance",
        "avg_prior_msgs",
        "avg_msgs_last_14d"
    ]
]


print("=" * 130)
print("FINAL POLICY — SELECTED RULES vs REST")
print("=" * 130)

display(
    policy_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "message_share":
            "{:.1%}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_share":
            "{:.1%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "net_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_prior_msgs":
            "{:.1f}",

        "avg_msgs_last_14d":
            "{:.1f}"
    })
)


# ============================================================
# 8. DIRECT COMPARISON
# ============================================================

send = policy_table.loc[
    policy_table["policy_group"]
    == "SEND — R1 + R3 + R4"
].iloc[0]

rest = policy_table.loc[
    policy_table["policy_group"]
    == "REST — R2 + R5 + R6"
].iloc[0]


delta_recovery_msg = (
    send["recovery_per_message"]
    -
    rest["recovery_per_message"]
)


relative_recovery_msg = (
    send["recovery_per_message"]
    /
    rest["recovery_per_message"]
    - 1
)


delta_payment_rate_pp = (
    send["payment_rate"]
    -
    rest["payment_rate"]
) * 100


print("\n" + "=" * 130)
print("SELECTED POLICY vs REST")
print("=" * 130)

print(
    f"Recovery/msg — selected : "
    f"R$ {send['recovery_per_message']:,.2f}"
)

print(
    f"Recovery/msg — rest     : "
    f"R$ {rest['recovery_per_message']:,.2f}"
)

print(
    f"Difference              : "
    f"R$ {delta_recovery_msg:+,.2f} / msg"
)

print(
    f"Relative difference     : "
    f"{relative_recovery_msg:+.1%}"
)

print(
    f"\nPayment rate — selected : "
    f"{send['payment_rate']:.2%}"
)

print(
    f"Payment rate — rest     : "
    f"{rest['payment_rate']:.2%}"
)

print(
    f"Difference              : "
    f"{delta_payment_rate_pp:+.2f} pp"
)


# ============================================================
# 9. QA
# ============================================================

print("\n" + "=" * 130)
print("QA")
print("=" * 130)

print(
    f"Historical messages : "
    f"{len(df):,}"
)

print(
    f"Table messages      : "
    f"{policy_table['messages'].sum():,}"
)

print(
    f"Unclassified        : "
    f"{(df['rule'] == 'UNCLASSIFIED').sum():,}"
)

assert (
    policy_table["messages"].sum()
    == len(df)
)

assert (
    (df["rule"] == "UNCLASSIFIED").sum()
    == 0
)

print("\n✓ RECONCILED")

FINAL POLICY — SELECTED RULES vs REST


,policy_group,messages,customers,message_share,payment_events,payment_rate,recovery_brl,recovery_share,recovery_per_message,net_per_message,avg_balance,avg_prior_msgs,avg_msgs_last_14d
0,REST — R2 + R5 + R6,"2,234","1,221",26.1%,175,7.83%,"R$ 56,975.10",16.5%,R$ 25.50,R$ 24.50,R$ 589.12,7.3,2.0
1,SEND — R1 + R3 + R4,"6,318","3,494",73.9%,434,6.87%,"R$ 288,509.97",83.5%,R$ 45.66,R$ 44.66,R$ 859.62,7.6,2.0



SELECTED POLICY vs REST
Recovery/msg — selected : R$ 45.66
Recovery/msg — rest     : R$ 25.50
Difference              : R$ +20.16 / msg
Relative difference     : +79.1%

Payment rate — selected : 6.87%
Payment rate — rest     : 7.83%
Difference              : -0.96 pp

QA
Historical messages : 8,552
Table messages      : 8,552
Unclassified        : 0

✓ RECONCILED


In [12]:
# ============================================================
# FINAL OPEN TABLE
#
# TOTAL UNIVERSE
#   ├── R1
#   ├── R2
#   ├── R3
#   └── REST
#
# R1/R2/R3 = selected rules
# REST      = everything else
# ============================================================

import pandas as pd
import numpy as np


df = decision.copy()


# ============================================================
# 1. STANDARDIZE SIGNALS
# ============================================================

df["has_prior_engagement"] = (
    df["had_prior_engagement"]
    .fillna(False)
    .astype(bool)
)

df["has_prior_payment"] = (
    df["had_prior_payment"]
    .fillna(False)
    .astype(bool)
)

df["n_msgs_last_14d"] = pd.to_numeric(
    df["n_msgs_last_14d"],
    errors="coerce"
)

df["no_msg_last_14d"] = (
    df["n_msgs_last_14d"].eq(0)
)


# ============================================================
# 2. FINAL SELECTED RULES
# ============================================================

r1 = (
    df["has_prior_engagement"]
    & df["has_prior_payment"]
    & df["no_msg_last_14d"]
)

r2 = (
    df["has_prior_engagement"]
    & ~df["has_prior_payment"]
    & df["no_msg_last_14d"]
)

r3 = (
    df["has_prior_engagement"]
    & ~df["has_prior_payment"]
    & ~df["no_msg_last_14d"]
)


df["final_rule"] = np.select(
    [r1, r2, r3],
    [
        "R1 | Engaged + prior payment + 0 msgs/14d",
        "R2 | Engaged + no prior payment + 0 msgs/14d",
        "R3 | Engaged + no prior payment + recent contact"
    ],
    default="REST"
)


# ============================================================
# 3. SUMMARY FUNCTION
# ============================================================

def calculate_metrics(data):

    messages = len(data)

    customers = data["customer_id"].nunique()

    payment_events = data["paid_72h"].sum()

    payment_rate = (
        data["paid_72h"].mean()
        if messages > 0
        else np.nan
    )

    recovery = data["recovery_72h"].sum()

    recovery_per_message = (
        recovery / messages
        if messages > 0
        else np.nan
    )

    avg_balance = (
        data["outstanding_balance_brl"].mean()
    )

    return {
        "messages": messages,
        "customers": customers,
        "payment_events": payment_events,
        "payment_rate": payment_rate,
        "recovery_brl": recovery,
        "recovery_per_message": recovery_per_message,
        "avg_balance": avg_balance
    }


# ============================================================
# 4. TOTAL UNIVERSE
# ============================================================

rows = []

total_metrics = calculate_metrics(df)

rows.append({
    "segment": "TOTAL | DPD30-45",
    **total_metrics
})


# ============================================================
# 5. EACH SELECTED RULE + REST
# ============================================================

rule_order = [
    "R1 | Engaged + prior payment + 0 msgs/14d",
    "R2 | Engaged + no prior payment + 0 msgs/14d",
    "R3 | Engaged + no prior payment + recent contact",
    "REST"
]


for rule in rule_order:

    subset = df.loc[
        df["final_rule"].eq(rule)
    ]

    metrics = calculate_metrics(subset)

    rows.append({
        "segment": rule,
        **metrics
    })


# ============================================================
# 6. FINAL TABLE
# ============================================================

final_table = pd.DataFrame(rows)


# ------------------------------------------------------------
# Shares relative to TOTAL universe
# ------------------------------------------------------------

total_messages = len(df)
total_recovery = df["recovery_72h"].sum()


final_table["message_share"] = (
    final_table["messages"]
    / total_messages
)


final_table["recovery_share"] = (
    final_table["recovery_brl"]
    / total_recovery
)


# ------------------------------------------------------------
# Difference vs TOTAL historical average
# ------------------------------------------------------------

baseline_rpm = (
    total_recovery
    / total_messages
)


final_table["delta_rpm_vs_total"] = (
    final_table["recovery_per_message"]
    - baseline_rpm
)


# ============================================================
# 7. DISPLAY
# ============================================================

final_table = final_table[
    [
        "segment",
        "messages",
        "customers",
        "message_share",
        "payment_events",
        "payment_rate",
        "recovery_brl",
        "recovery_share",
        "recovery_per_message",
        "delta_rpm_vs_total",
        "avg_balance"
    ]
]


print("=" * 140)
print("DPD30-45 — FINAL RULE DECOMPOSITION")
print("=" * 140)

display(
    final_table.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "message_share":
            "{:.1%}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_share":
            "{:.1%}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "delta_rpm_vs_total":
            "R$ {:+,.2f}",

        "avg_balance":
            "R$ {:,.2f}"
    })
)


# ============================================================
# 8. RECONCILIATION
#
# R1 + R2 + R3 + REST must equal TOTAL
# ============================================================

detail = final_table.iloc[1:]

print("\n" + "=" * 100)
print("RECONCILIATION")
print("=" * 100)

print(
    f"TOTAL messages       : "
    f"{total_messages:,}"
)

print(
    f"R1 + R2 + R3 + REST : "
    f"{detail['messages'].sum():,.0f}"
)

print(
    f"\nTOTAL recovery       : "
    f"R$ {total_recovery:,.2f}"
)

print(
    f"Rules + REST recovery: "
    f"R$ {detail['recovery_brl'].sum():,.2f}"
)

assert (
    detail["messages"].sum()
    == total_messages
)

assert np.isclose(
    detail["recovery_brl"].sum(),
    total_recovery
)

print("\n✓ R1 + R2 + R3 + REST = TOTAL")

DPD30-45 — FINAL RULE DECOMPOSITION


,segment,messages,customers,message_share,payment_events,payment_rate,recovery_brl,recovery_share,recovery_per_message,delta_rpm_vs_total,avg_balance
0,TOTAL | DPD30-45,"8,552","4,581",100.0%,609,7.12%,"R$ 345,485.07",100.0%,R$ 40.40,R$ +0.00,R$ 788.96
1,R1 | Engaged + prior payment + 0 msgs/14d,131,131,1.5%,25,19.08%,"R$ 5,186.51",1.5%,R$ 39.59,R$ -0.81,R$ 348.04
2,R2 | Engaged + no prior payment + 0 msgs/14d,708,708,8.3%,50,7.06%,"R$ 29,693.02",8.6%,R$ 41.94,R$ +1.54,R$ 872.15
3,R3 | Engaged + no prior payment + recent contact,"5,479","2,911",64.1%,359,6.55%,"R$ 253,630.44",73.4%,R$ 46.29,R$ +5.89,R$ 870.23
4,REST,"2,234","1,221",26.1%,175,7.83%,"R$ 56,975.10",16.5%,R$ 25.50,R$ -14.89,R$ 589.12



RECONCILIATION
TOTAL messages       : 8,552
R1 + R2 + R3 + REST : 8,552

TOTAL recovery       : R$ 345,485.07
Rules + REST recovery: R$ 345,485.07

✓ R1 + R2 + R3 + REST = TOTAL


In [13]:
# ============================================================
# TEMPLATE PERFORMANCE WITHIN SELECTED RULES
#
# Population:
# DPD30-45
# Only selected profiles R1 / R2 / R3
#
# Question:
# Within comparable selected profiles, which template is
# historically associated with higher payment / recovery?
#
# IMPORTANT:
# Observational association — NOT causal template effect.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. SELECT ONLY R1 / R2 / R3
# ============================================================

selected = df.loc[
    df["final_rule"].ne("REST")
].copy()


print("=" * 110)
print("SELECTED POPULATION")
print("=" * 110)

print(f"Messages  : {len(selected):,}")
print(f"Customers : {selected['customer_id'].nunique():,}")

print("\nTemplates:")
print(
    selected["template"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# 2. RULE × TEMPLATE
# ============================================================

template_table = (
    selected
    .groupby(
        ["final_rule", "template"],
        observed=True
    )
    .agg(

        messages=(
            "customer_id",
            "size"
        ),

        customers=(
            "customer_id",
            "nunique"
        ),

        payment_events=(
            "paid_72h",
            "sum"
        ),

        payment_rate=(
            "paid_72h",
            "mean"
        ),

        recovery_brl=(
            "recovery_72h",
            "sum"
        ),

        avg_balance=(
            "outstanding_balance_brl",
            "mean"
        ),

        avg_dpd=(
            "days_past_due",
            "mean"
        ),

        avg_prior_msgs=(
            "n_prior_msgs_total",
            "mean"
        ),

        avg_msgs_last_14d=(
            "n_msgs_last_14d",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 3. ECONOMICS
# ============================================================

template_table["recovery_per_message"] = (
    template_table["recovery_brl"]
    /
    template_table["messages"]
)

template_table["net_per_message"] = (
    template_table["recovery_per_message"]
    - 1
)


# ============================================================
# 4. SHARE OF EACH TEMPLATE WITHIN RULE
#
# Important for common-support check:
# Was each template actually used enough inside each profile?
# ============================================================

rule_totals = (
    template_table
    .groupby("final_rule")["messages"]
    .transform("sum")
)

template_table["template_share_within_rule"] = (
    template_table["messages"]
    /
    rule_totals
)


# ============================================================
# 5. SAMPLE SIZE FLAG
# ============================================================

template_table["sample_flag"] = np.select(
    [
        template_table["messages"] < 30,
        template_table["messages"] < 100
    ],
    [
        "VERY SMALL",
        "SMALL"
    ],
    default="OK"
)


# ============================================================
# 6. ORDER
# ============================================================

rule_order = {
    "R1 | Engaged + prior payment + 0 msgs/14d": 1,
    "R2 | Engaged + no prior payment + 0 msgs/14d": 2,
    "R3 | Engaged + no prior payment + recent contact": 3
}

template_table["rule_order"] = (
    template_table["final_rule"]
    .map(rule_order)
)

template_table = (
    template_table
    .sort_values(
        [
            "rule_order",
            "recovery_per_message"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# ============================================================
# 7. FINAL TABLE
# ============================================================

result = template_table[
    [
        "final_rule",
        "template",
        "messages",
        "customers",
        "template_share_within_rule",
        "payment_events",
        "payment_rate",
        "recovery_brl",
        "recovery_per_message",
        "net_per_message",
        "avg_balance",
        "avg_dpd",
        "avg_prior_msgs",
        "avg_msgs_last_14d",
        "sample_flag"
    ]
].copy()


display(
    result.style.format({

        "messages":
            "{:,.0f}",

        "customers":
            "{:,.0f}",

        "template_share_within_rule":
            "{:.1%}",

        "payment_events":
            "{:,.0f}",

        "payment_rate":
            "{:.2%}",

        "recovery_brl":
            "R$ {:,.2f}",

        "recovery_per_message":
            "R$ {:,.2f}",

        "net_per_message":
            "R$ {:,.2f}",

        "avg_balance":
            "R$ {:,.2f}",

        "avg_dpd":
            "{:.1f}",

        "avg_prior_msgs":
            "{:.1f}",

        "avg_msgs_last_14d":
            "{:.1f}"
    })
)


# ============================================================
# 8. QA
# ============================================================

print("\n" + "=" * 110)
print("QA")
print("=" * 110)

print(
    f"Selected messages       : "
    f"{len(selected):,}"
)

print(
    f"Table messages          : "
    f"{template_table['messages'].sum():,}"
)

print(
    f"Selected recovery       : "
    f"R$ {selected['recovery_72h'].sum():,.2f}"
)

print(
    f"Table recovery          : "
    f"R$ {template_table['recovery_brl'].sum():,.2f}"
)

assert template_table["messages"].sum() == len(selected)

assert np.isclose(
    template_table["recovery_brl"].sum(),
    selected["recovery_72h"].sum()
)

print("\n✓ RECONCILED")

SELECTED POPULATION
Messages  : 6,318
Customers : 3,494

Templates:
template
discount_offer       2794
urgent_reminder      2040
pix_link             1324
friendly_reminder     160


,final_rule,template,messages,customers,template_share_within_rule,payment_events,payment_rate,recovery_brl,recovery_per_message,net_per_message,avg_balance,avg_dpd,avg_prior_msgs,avg_msgs_last_14d,sample_flag
0,R1 | Engaged + prior payment + 0 msgs/14d,pix_link,34,34,26.0%,8,23.53%,"R$ 2,017.33",R$ 59.33,R$ 58.33,R$ 352.69,38.4,6.0,0.0,SMALL
1,R1 | Engaged + prior payment + 0 msgs/14d,urgent_reminder,40,40,30.5%,6,15.00%,"R$ 1,356.25",R$ 33.91,R$ 32.91,R$ 364.15,38.2,5.6,0.0,SMALL
2,R1 | Engaged + prior payment + 0 msgs/14d,discount_offer,57,57,43.5%,11,19.30%,"R$ 1,812.93",R$ 31.81,R$ 30.81,R$ 333.97,39.4,5.9,0.0,SMALL
3,R2 | Engaged + no prior payment + 0 msgs/14d,friendly_reminder,4,4,0.6%,1,25.00%,R$ 250.16,R$ 62.54,R$ 61.54,R$ 490.42,30.0,5.0,0.0,VERY SMALL
4,R2 | Engaged + no prior payment + 0 msgs/14d,discount_offer,364,364,51.4%,32,8.79%,"R$ 20,579.80",R$ 56.54,R$ 55.54,R$ 857.14,40.0,6.4,0.0,OK
5,R2 | Engaged + no prior payment + 0 msgs/14d,urgent_reminder,206,206,29.1%,11,5.34%,"R$ 5,708.40",R$ 27.71,R$ 26.71,R$ 913.06,38.1,5.9,0.0,OK
6,R2 | Engaged + no prior payment + 0 msgs/14d,pix_link,134,134,18.9%,6,4.48%,"R$ 3,154.66",R$ 23.54,R$ 22.54,R$ 861.42,38.8,6.0,0.0,OK
7,R3 | Engaged + no prior payment + recent contact,discount_offer,"2,373","1,763",43.3%,193,8.13%,"R$ 128,358.56",R$ 54.09,R$ 53.09,R$ 889.74,37.3,8.0,2.2,OK
8,R3 | Engaged + no prior payment + recent contact,pix_link,"1,156","1,025",21.1%,78,6.75%,"R$ 55,444.86",R$ 47.96,R$ 46.96,R$ 849.06,36.0,7.9,2.3,OK
9,R3 | Engaged + no prior payment + recent contact,friendly_reminder,156,156,2.8%,9,5.77%,"R$ 7,463.32",R$ 47.84,R$ 46.84,R$ 876.48,30.0,6.9,2.6,OK



QA
Selected messages       : 6,318
Table messages          : 6,318
Selected recovery       : R$ 288,509.97
Table recovery          : R$ 288,509.97

✓ RECONCILED


In [53]:
# ============================================================
# GLOBAL TEST — TEMPLATE
# ENGAGED ONLY
#
# Pergunta:
# Dentro dos clientes ENGAGED,
# existe diferença estatística entre
# PIX vs URGENT vs DISCOUNT?
#
# Métrica principal:
# recovery_72h / mensagem
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import kruskal


# ============================================================
# 1. PREPARAR BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)

templates = [
    "pix_link",
    "urgent_reminder",
    "discount_offer"
]


# ============================================================
# 2. FUNÇÃO
# ============================================================

def test_engaged_templates(df, dpd_min, dpd_max):

    d = df.loc[
        df["days_past_due"].between(dpd_min, dpd_max)
        & df["has_prior_engagement"].eq(True)
        & df["template"].isin(templates)
    ].copy()

    # --------------------------------------------------------
    # DESCRITIVO
    # --------------------------------------------------------

    summary = (
        d.groupby("template")
        .agg(
            messages=("customer_id", "size"),
            customers=("customer_id", "nunique"),
            payments=("payment_event_72h", "sum"),
            recovery_brl=("recovery_72h", "sum"),
            recovery_per_message=("recovery_72h", "mean")
        )
        .reset_index()
    )

    summary["payment_rate"] = (
        summary["payments"]
        / summary["messages"]
    )


    # --------------------------------------------------------
    # ARRAYS
    # --------------------------------------------------------

    pix = d.loc[
        d["template"].eq("pix_link"),
        "recovery_72h"
    ]

    urgent = d.loc[
        d["template"].eq("urgent_reminder"),
        "recovery_72h"
    ]

    discount = d.loc[
        d["template"].eq("discount_offer"),
        "recovery_72h"
    ]


    # --------------------------------------------------------
    # KRUSKAL-WALLIS
    # --------------------------------------------------------

    statistic, p_value = kruskal(
        pix,
        urgent,
        discount
    )


    # ========================================================
    # PRINT
    # ========================================================

    print("\n")
    print("=" * 100)
    print(
        f"DPD {dpd_min}–{dpd_max} | ENGAGED"
    )
    print("=" * 100)

    print("\nDESCRIPTIVE\n")

    print(
        summary.to_string(
            index=False,
            formatters={
                "recovery_brl":
                    lambda x: f"R$ {x:,.2f}",

                "recovery_per_message":
                    lambda x: f"R$ {x:,.2f}",

                "payment_rate":
                    lambda x: f"{x:.2%}"
            }
        )
    )

    print("\nGLOBAL TEST")
    print("-" * 60)

    print(
        f"Kruskal-Wallis statistic : {statistic:.4f}"
    )

    print(
        f"Global p-value           : {p_value:.6f}"
    )

    print()

    if p_value < 0.05:

        print(
            "✓ Há evidência estatística de diferença "
            "entre os templates."
        )

    else:

        print(
            "✕ Não há evidência estatística suficiente "
            "de diferença entre os templates."
        )

    return summary, p_value


# ============================================================
# 3. DPD 30–45
# ============================================================

summary_30_45, p_30_45 = test_engaged_templates(
    df,
    30,
    45
)


# ============================================================
# 4. DPD 46–60
# ============================================================

summary_46_60, p_46_60 = test_engaged_templates(
    df,
    46,
    60
)


# ============================================================
# 5. RESUMO FINAL
# ============================================================

print("\n")
print("=" * 100)
print("RESULTADO FINAL")
print("=" * 100)

print(
    f"""
DPD 30–45 | ENGAGED
p-value = {p_30_45:.6f}

DPD 46–60 | ENGAGED
p-value = {p_46_60:.6f}
"""
)



DPD 30–45 | ENGAGED

DESCRIPTIVE

       template  messages  customers  payments  recovery_brl recovery_per_message payment_rate
 discount_offer      3290       2469       321 R$ 172,348.53             R$ 52.39        9.76%
       pix_link      1564       1387       121  R$ 68,317.14             R$ 43.68        7.74%
urgent_reminder      2409       1947       125  R$ 79,382.10             R$ 32.95        5.19%

GLOBAL TEST
------------------------------------------------------------
Kruskal-Wallis statistic : 39.0200
Global p-value           : 0.000000

✓ Há evidência estatística de diferença entre os templates.


DPD 46–60 | ENGAGED

DESCRIPTIVE

       template  messages  customers  payments recovery_brl recovery_per_message payment_rate
 discount_offer      2176       1635       129 R$ 60,990.68             R$ 28.03        5.93%
       pix_link       916        821        40 R$ 23,666.74             R$ 25.84        4.37%
urgent_reminder      1310       1108        57 R$ 38,504.80 

In [54]:
# ============================================================
# DPD 30–45
# SIMPLIFIED POLICY:
#
# ENGAGEMENT + PRIOR PAYMENT ONLY
#
# ENGAGED
#    ├── PRIOR PAYMENT = YES
#    │       └── PIX vs URGENT vs DISCOUNT
#    │
#    └── PRIOR PAYMENT = NO
#            └── PIX vs URGENT vs DISCOUNT
#
# No 14d contact segmentation.
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import kruskal


# ============================================================
# 1. PREPARE DATA
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)

templates = [
    "pix_link",
    "urgent_reminder",
    "discount_offer"
]


# ============================================================
# 2. DPD30–45 + ENGAGED
# ============================================================

base = df.loc[
    df["days_past_due"].between(30, 45)
    & df["has_prior_engagement"].eq(True)
    & df["template"].isin(templates)
].copy()


print("=" * 100)
print("DPD30–45 | ENGAGED")
print("=" * 100)

print(f"Messages  : {len(base):,}")
print(f"Customers : {base['customer_id'].nunique():,}")


# ============================================================
# 3. FUNCTION
# ============================================================

def analyze_prior_payment_segment(
    data,
    prior_payment_value,
    label
):

    d = data.loc[
        data["has_prior_payment"].eq(
            prior_payment_value
        )
    ].copy()


    # --------------------------------------------------------
    # DESCRIPTIVE
    # --------------------------------------------------------

    summary = (
        d.groupby("template")
        .agg(
            messages=("customer_id", "size"),
            customers=("customer_id", "nunique"),
            payments=("payment_event_72h", "sum"),
            recovery_brl=("recovery_72h", "sum"),
            avg_recovery_msg=("recovery_72h", "mean"),
            avg_balance=("outstanding_balance_brl", "mean")
        )
        .reset_index()
    )

    summary["payment_rate"] = (
        summary["payments"]
        / summary["messages"]
    )


    # --------------------------------------------------------
    # ORDER BY ECONOMIC RESULT
    # --------------------------------------------------------

    summary = summary.sort_values(
        "avg_recovery_msg",
        ascending=False
    ).reset_index(drop=True)


    # --------------------------------------------------------
    # GLOBAL TEST
    # --------------------------------------------------------

    groups = []

    for template in templates:

        values = d.loc[
            d["template"].eq(template),
            "recovery_72h"
        ]

        groups.append(values)


    statistic, p_value = kruskal(
        *groups
    )


    # ========================================================
    # PRINT
    # ========================================================

    print("\n")
    print("=" * 100)
    print(label)
    print("=" * 100)

    print(
        summary.to_string(
            index=False,
            formatters={

                "recovery_brl":
                    lambda x: f"R$ {x:,.2f}",

                "avg_recovery_msg":
                    lambda x: f"R$ {x:,.2f}",

                "avg_balance":
                    lambda x: f"R$ {x:,.2f}",

                "payment_rate":
                    lambda x: f"{x:.2%}"
            }
        )
    )


    print("\nGLOBAL TEMPLATE TEST")
    print("-" * 60)

    print(
        f"Kruskal-Wallis statistic : {statistic:.4f}"
    )

    print(
        f"Global p-value           : {p_value:.6f}"
    )


    print("\nOBSERVED RANKING")
    print("-" * 60)

    for i, row in summary.iterrows():

        print(
            f"{i+1}. "
            f"{row['template']:<18} "
            f"R$ {row['avg_recovery_msg']:,.2f}/msg | "
            f"payment {row['payment_rate']:.2%} | "
            f"n={row['messages']:,}"
        )


    print("\nINTERPRETATION")
    print("-" * 60)

    if p_value < 0.05:

        print(
            "✓ Há evidência de que pelo menos "
            "um template difere dos demais."
        )

    elif p_value < 0.10:

        print(
            "△ Há evidência direcional de diferença "
            "entre templates."
        )

    else:

        print(
            "✕ Não há evidência suficiente de diferença "
            "entre os templates."
        )


    return summary, p_value


# ============================================================
# 4. PRIOR PAYMENT = YES
# ============================================================

summary_prior_yes, p_prior_yes = (
    analyze_prior_payment_segment(

        data=base,

        prior_payment_value=True,

        label=(
            "DPD30–45 | ENGAGED | PRIOR PAYMENT = YES"
        )
    )
)


# ============================================================
# 5. PRIOR PAYMENT = NO
# ============================================================

summary_prior_no, p_prior_no = (
    analyze_prior_payment_segment(

        data=base,

        prior_payment_value=False,

        label=(
            "DPD30–45 | ENGAGED | PRIOR PAYMENT = NO"
        )
    )
)


# ============================================================
# 6. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 100)
print("SIMPLIFIED POLICY — DPD30–45")
print("=" * 100)

print(
    f"""
ENGAGED + PRIOR PAYMENT = YES
Global p-value : {p_prior_yes:.6f}

ENGAGED + PRIOR PAYMENT = NO
Global p-value : {p_prior_no:.6f}
"""
)

DPD30–45 | ENGAGED
Messages  : 7,263
Customers : 3,964


DPD30–45 | ENGAGED | PRIOR PAYMENT = YES
       template  messages  customers  payments recovery_brl avg_recovery_msg avg_balance payment_rate
 discount_offer       553        413        96 R$ 23,410.17         R$ 42.33   R$ 388.58       17.36%
       pix_link       274        241        37  R$ 9,717.62         R$ 35.47   R$ 400.96       13.50%
urgent_reminder       409        336        35 R$ 11,310.00         R$ 27.65   R$ 399.19        8.56%

GLOBAL TEMPLATE TEST
------------------------------------------------------------
Kruskal-Wallis statistic : 14.4096
Global p-value           : 0.000743

OBSERVED RANKING
------------------------------------------------------------
1. discount_offer     R$ 42.33/msg | payment 17.36% | n=553
2. pix_link           R$ 35.47/msg | payment 13.50% | n=274
3. urgent_reminder    R$ 27.65/msg | payment 8.56% | n=409

INTERPRETATION
------------------------------------------------------------
✓ Há 

In [56]:
# ============================================================
# FINAL TEST — DPD30–45
#
# DISCOUNT vs PIX
#
# Split:
#   1. Prior payment = YES
#   2. Prior payment = NO
#
# Goal:
# Determine whether prior payment is actually needed
# for template choice.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact


def discount_vs_pix_test(data, prior_payment, label,
                         n_perm=50_000,
                         n_boot=50_000,
                         seed=42):

    rng = np.random.default_rng(seed)

    d = data.loc[
        data["has_prior_payment"].eq(prior_payment)
        & data["template"].isin([
            "discount_offer",
            "pix_link"
        ])
    ].copy()

    discount = d.loc[
        d["template"].eq("discount_offer"),
        "recovery_72h"
    ].to_numpy()

    pix = d.loc[
        d["template"].eq("pix_link"),
        "recovery_72h"
    ].to_numpy()


    # ========================================================
    # OBSERVED DIFFERENCE
    # Discount - Pix
    # ========================================================

    diff = discount.mean() - pix.mean()


    # ========================================================
    # PERMUTATION TEST
    #
    # H1: Discount > Pix
    # ========================================================

    values = np.concatenate([
        discount,
        pix
    ])

    n_discount = len(discount)

    perm_diffs = np.empty(n_perm)

    for i in range(n_perm):

        perm = rng.permutation(values)

        perm_diffs[i] = (
            perm[:n_discount].mean()
            -
            perm[n_discount:].mean()
        )

    p_recovery = (
        np.sum(perm_diffs >= diff) + 1
    ) / (n_perm + 1)


    # ========================================================
    # BOOTSTRAP CI
    # ========================================================

    boot_diff = np.empty(n_boot)

    for i in range(n_boot):

        d_boot = rng.choice(
            discount,
            len(discount),
            replace=True
        )

        p_boot = rng.choice(
            pix,
            len(pix),
            replace=True
        )

        boot_diff[i] = (
            d_boot.mean()
            -
            p_boot.mean()
        )

    ci_low, ci_high = np.percentile(
        boot_diff,
        [2.5, 97.5]
    )


    # ========================================================
    # PAYMENT RATE
    # ========================================================

    discount_paid = int(
        d.loc[
            d["template"].eq("discount_offer"),
            "payment_event_72h"
        ].sum()
    )

    pix_paid = int(
        d.loc[
            d["template"].eq("pix_link"),
            "payment_event_72h"
        ].sum()
    )

    table = [
        [
            discount_paid,
            len(discount) - discount_paid
        ],
        [
            pix_paid,
            len(pix) - pix_paid
        ]
    ]

    odds_ratio, p_payment = fisher_exact(
        table,
        alternative="greater"
    )


    # ========================================================
    # OUTPUT
    # ========================================================

    print("\n")
    print("=" * 90)
    print(label)
    print("=" * 90)

    print(
        f"""
DISCOUNT
n                 : {len(discount):,}
Recovery / msg    : R$ {discount.mean():,.2f}
Payment rate      : {discount_paid / len(discount):.2%}

PIX
n                 : {len(pix):,}
Recovery / msg    : R$ {pix.mean():,.2f}
Payment rate      : {pix_paid / len(pix):.2%}

DIFFERENCE
Discount - Pix    : R$ {diff:,.2f}
Relative lift     : {(discount.mean()/pix.mean()-1):+.1%}

RECOVERY / MESSAGE
Permutation p     : {p_recovery:.4f}
Bootstrap 95% CI  : R$ [{ci_low:,.2f}, {ci_high:,.2f}]

PAYMENT RATE
Fisher p          : {p_payment:.4f}
"""
    )

    if p_recovery < .05 and ci_low > 0:

        print(
            "✓ Evidence supports DISCOUNT > PIX "
            "for recovery/msg."
        )

    elif p_recovery < .10:

        print(
            "△ Directional evidence for "
            "DISCOUNT > PIX."
        )

    else:

        print(
            "✕ Insufficient evidence to conclude "
            "DISCOUNT > PIX."
        )

    return {
        "segment": label,
        "discount_n": len(discount),
        "pix_n": len(pix),
        "discount_rpm": discount.mean(),
        "pix_rpm": pix.mean(),
        "diff": diff,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_recovery": p_recovery,
        "p_payment": p_payment
    }


# ============================================================
# PRIOR PAYMENT = YES
# ============================================================

r_yes = discount_vs_pix_test(
    base,
    True,
    "DPD30–45 | ENGAGED | PRIOR PAYMENT = YES"
)


# ============================================================
# PRIOR PAYMENT = NO
# ============================================================

r_no = discount_vs_pix_test(
    base,
    False,
    "DPD30–45 | ENGAGED | PRIOR PAYMENT = NO"
)


# ============================================================
# FINAL SUMMARY
# ============================================================

results_discount_pix = pd.DataFrame([
    r_yes,
    r_no
])

print("\n")
print("=" * 90)
print("FINAL DECISION TABLE")
print("=" * 90)

display(
    results_discount_pix.round({
        "discount_rpm": 2,
        "pix_rpm": 2,
        "diff": 2,
        "ci_low": 2,
        "ci_high": 2,
        "p_recovery": 4,
        "p_payment": 4
    })
)



DPD30–45 | ENGAGED | PRIOR PAYMENT = YES

DISCOUNT
n                 : 553
Recovery / msg    : R$ 42.33
Payment rate      : 17.36%

PIX
n                 : 274
Recovery / msg    : R$ 35.47
Payment rate      : 13.50%

DIFFERENCE
Discount - Pix    : R$ 6.87
Relative lift     : +19.4%

RECOVERY / MESSAGE
Permutation p     : 0.2244
Bootstrap 95% CI  : R$ [-11.20, 23.70]

PAYMENT RATE
Fisher p          : 0.0923

✕ Insufficient evidence to conclude DISCOUNT > PIX.


DPD30–45 | ENGAGED | PRIOR PAYMENT = NO

DISCOUNT
n                 : 2,737
Recovery / msg    : R$ 54.42
Payment rate      : 8.22%

PIX
n                 : 1,290
Recovery / msg    : R$ 45.43
Payment rate      : 6.51%

DIFFERENCE
Discount - Pix    : R$ 8.99
Relative lift     : +19.8%

RECOVERY / MESSAGE
Permutation p     : 0.1100
Bootstrap 95% CI  : R$ [-5.67, 23.01]

PAYMENT RATE
Fisher p          : 0.0318

✕ Insufficient evidence to conclude DISCOUNT > PIX.


FINAL DECISION TABLE


,segment,discount_n,pix_n,discount_rpm,pix_rpm,diff,ci_low,ci_high,p_recovery,p_payment
0,DPD30–45 | ENGAGED | PRIOR PAYMENT = YES,553,274,42.33,35.47,6.87,-11.20,23.70,0.22,0.09
1,DPD30–45 | ENGAGED | PRIOR PAYMENT = NO,2737,1290,54.42,45.43,8.99,-5.67,23.01,0.11,0.03


In [57]:
# ============================================================
# DPD 30–45 — CAN ENGAGEMENT ALONE DEFINE SEND / NO SEND?
#
# Rule candidate:
#
# PRIOR ENGAGEMENT = YES  -> SEND
# PRIOR ENGAGEMENT = NO   -> NO SEND
#
# No prior_payment
# No contact pressure
# No template segmentation
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact


# ============================================================
# 1. BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)

base = df.loc[
    df["days_past_due"].between(30, 45)
].copy()


# ============================================================
# 2. DESCRIPTIVE — ENGAGED vs NO ENGAGEMENT
# ============================================================

summary = (
    base
    .groupby("has_prior_engagement")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payments=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

summary["segment"] = np.where(
    summary["has_prior_engagement"],
    "ENGAGED",
    "NO ENGAGEMENT"
)

summary["payment_rate"] = (
    summary["payments"]
    / summary["messages"]
)


print("=" * 100)
print("DPD 30–45 | ENGAGEMENT AS THE ONLY OPENING RULE")
print("=" * 100)

print(
    summary[
        [
            "segment",
            "messages",
            "customers",
            "payments",
            "recovery_brl",
            "recovery_per_message",
            "payment_rate",
            "avg_balance"
        ]
    ].to_string(
        index=False,
        formatters={
            "recovery_brl":
                lambda x: f"R$ {x:,.2f}",

            "recovery_per_message":
                lambda x: f"R$ {x:,.2f}",

            "payment_rate":
                lambda x: f"{x:.2%}",

            "avg_balance":
                lambda x: f"R$ {x:,.2f}"
        }
    )
)


# ============================================================
# 3. ARRAYS
# ============================================================

engaged = base.loc[
    base["has_prior_engagement"].eq(True),
    "recovery_72h"
].to_numpy()

no_engagement = base.loc[
    base["has_prior_engagement"].eq(False),
    "recovery_72h"
].to_numpy()


# ============================================================
# 4. OBSERVED DIFFERENCE
# ============================================================

diff = (
    engaged.mean()
    - no_engagement.mean()
)

relative_lift = (
    engaged.mean()
    / no_engagement.mean()
    - 1
)


# ============================================================
# 5. PERMUTATION TEST
#
# H1:
# Engaged > No Engagement
# ============================================================

rng = np.random.default_rng(42)

values = np.concatenate([
    engaged,
    no_engagement
])

n_engaged = len(engaged)

n_perm = 50_000

perm_diffs = np.empty(n_perm)

for i in range(n_perm):

    perm = rng.permutation(values)

    perm_diffs[i] = (
        perm[:n_engaged].mean()
        -
        perm[n_engaged:].mean()
    )

p_recovery = (
    np.sum(perm_diffs >= diff) + 1
) / (
    n_perm + 1
)


# ============================================================
# 6. BOOTSTRAP CI
# ============================================================

n_boot = 50_000

boot_diff = np.empty(n_boot)

for i in range(n_boot):

    e_boot = rng.choice(
        engaged,
        len(engaged),
        replace=True
    )

    n_boot_sample = rng.choice(
        no_engagement,
        len(no_engagement),
        replace=True
    )

    boot_diff[i] = (
        e_boot.mean()
        -
        n_boot_sample.mean()
    )

ci_low, ci_high = np.percentile(
    boot_diff,
    [2.5, 97.5]
)


# ============================================================
# 7. PAYMENT RATE — FISHER
# ============================================================

engaged_paid = int(
    base.loc[
        base["has_prior_engagement"].eq(True),
        "payment_event_72h"
    ].sum()
)

no_engagement_paid = int(
    base.loc[
        base["has_prior_engagement"].eq(False),
        "payment_event_72h"
    ].sum()
)

table = [
    [
        engaged_paid,
        len(engaged) - engaged_paid
    ],
    [
        no_engagement_paid,
        len(no_engagement) - no_engagement_paid
    ]
]

odds_ratio, p_payment = fisher_exact(
    table,
    alternative="greater"
)


# ============================================================
# 8. RESULT
# ============================================================

print("\n")
print("=" * 100)
print("ENGAGED vs NO ENGAGEMENT")
print("=" * 100)

print(
    f"""
ENGAGED
Messages             : {len(engaged):,}
Recovery / msg       : R$ {engaged.mean():,.2f}
Payment rate         : {engaged_paid / len(engaged):.2%}

NO ENGAGEMENT
Messages             : {len(no_engagement):,}
Recovery / msg       : R$ {no_engagement.mean():,.2f}
Payment rate         : {no_engagement_paid / len(no_engagement):.2%}

ECONOMIC DIFFERENCE
Engaged - No Eng.    : R$ {diff:,.2f}/msg
Relative lift        : {relative_lift:+.1%}

RECOVERY / MESSAGE
Permutation p-value  : {p_recovery:.6f}
Bootstrap 95% CI     : R$ [{ci_low:,.2f}, {ci_high:,.2f}]

PAYMENT RATE
Fisher p-value       : {p_payment:.6f}
Odds ratio           : {odds_ratio:.2f}
"""
)


# ============================================================
# 9. SIMPLE RULE
# ============================================================

print("=" * 100)
print("CANDIDATE OPENING RULE")
print("=" * 100)

if (
    p_recovery < 0.05
    and ci_low > 0
):

    print(
        """
✓ ENGAGEMENT alone strongly separates the historical economics.

Candidate rule:

DPD 30–45
    PRIOR ENGAGEMENT = YES -> SEND
    PRIOR ENGAGEMENT = NO  -> NO SEND

Prior payment and 14d contact pressure are not required
for the opening decision.
"""
    )

else:

    print(
        """
△ Engagement shows an observed difference, but the
uncertainty does not support using it alone as a strong
opening rule.
"""
    )

DPD 30–45 | ENGAGEMENT AS THE ONLY OPENING RULE
      segment  messages  customers  payments  recovery_brl recovery_per_message payment_rate avg_balance
NO ENGAGEMENT      1093        617        28  R$ 16,824.07             R$ 15.39        2.56%   R$ 789.08
      ENGAGED      7459       4026       581 R$ 328,661.00             R$ 44.06        7.79%   R$ 788.94


ENGAGED vs NO ENGAGEMENT

ENGAGED
Messages             : 7,459
Recovery / msg       : R$ 44.06
Payment rate         : 7.79%

NO ENGAGEMENT
Messages             : 1,093
Recovery / msg       : R$ 15.39
Payment rate         : 2.56%

ECONOMIC DIFFERENCE
Engaged - No Eng.    : R$ 28.67/msg
Relative lift        : +186.3%

RECOVERY / MESSAGE
Permutation p-value  : 0.000020
Bootstrap 95% CI     : R$ [19.74, 36.78]

PAYMENT RATE
Fisher p-value       : 0.000000
Odds ratio           : 3.21

CANDIDATE OPENING RULE

✓ ENGAGEMENT alone strongly separates the historical economics.

Candidate rule:

DPD 30–45
    PRIOR ENGAGEMENT = YES -> SEND

In [58]:
# ============================================================
# FINAL OPENING RULE — DPD 30–60
#
# ONLY VARIABLE:
# PRIOR ENGAGEMENT
#
# Candidate rule:
#
# DPD 30–60
#     ENGAGED     -> SEND
#     NO ENGAGEMENT -> NO SEND
#
# No prior payment
# No contact pressure
# No template segmentation
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact


# ============================================================
# 1. PREPARE BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)


# ============================================================
# 2. DPD 30–60
# ============================================================

base = df.loc[
    df["days_past_due"].between(30, 60)
].copy()


# ============================================================
# 3. DESCRIPTIVE
# ============================================================

summary = (
    base
    .groupby("has_prior_engagement")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payments=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

summary["segment"] = np.where(
    summary["has_prior_engagement"],
    "ENGAGED",
    "NO ENGAGEMENT"
)

summary["payment_rate"] = (
    summary["payments"]
    / summary["messages"]
)


print("=" * 100)
print("DPD 30–60 | ENGAGEMENT AS THE ONLY OPENING RULE")
print("=" * 100)

print(
    summary[
        [
            "segment",
            "messages",
            "customers",
            "payments",
            "recovery_brl",
            "recovery_per_message",
            "payment_rate",
            "avg_balance"
        ]
    ].to_string(
        index=False,
        formatters={
            "recovery_brl":
                lambda x: f"R$ {x:,.2f}",

            "recovery_per_message":
                lambda x: f"R$ {x:,.2f}",

            "payment_rate":
                lambda x: f"{x:.2%}",

            "avg_balance":
                lambda x: f"R$ {x:,.2f}"
        }
    )
)


# ============================================================
# 4. ARRAYS
# ============================================================

engaged = base.loc[
    base["has_prior_engagement"].eq(True),
    "recovery_72h"
].to_numpy()

no_engagement = base.loc[
    base["has_prior_engagement"].eq(False),
    "recovery_72h"
].to_numpy()


# ============================================================
# 5. OBSERVED ECONOMIC DIFFERENCE
# ============================================================

diff = (
    engaged.mean()
    - no_engagement.mean()
)

relative_lift = (
    engaged.mean()
    / no_engagement.mean()
    - 1
)


# ============================================================
# 6. PERMUTATION TEST
#
# H0:
# Engaged and No Engagement have same recovery/msg
#
# H1:
# Engaged > No Engagement
# ============================================================

rng = np.random.default_rng(42)

values = np.concatenate([
    engaged,
    no_engagement
])

n_engaged = len(engaged)

n_perm = 50_000

perm_diffs = np.empty(n_perm)

for i in range(n_perm):

    perm = rng.permutation(values)

    perm_diffs[i] = (
        perm[:n_engaged].mean()
        -
        perm[n_engaged:].mean()
    )

p_recovery = (
    np.sum(
        perm_diffs >= diff
    ) + 1
) / (
    n_perm + 1
)


# ============================================================
# 7. BOOTSTRAP 95% CI
# ============================================================

n_boot = 50_000

boot_diff = np.empty(n_boot)

for i in range(n_boot):

    engaged_boot = rng.choice(
        engaged,
        len(engaged),
        replace=True
    )

    no_engagement_boot = rng.choice(
        no_engagement,
        len(no_engagement),
        replace=True
    )

    boot_diff[i] = (
        engaged_boot.mean()
        -
        no_engagement_boot.mean()
    )

ci_low, ci_high = np.percentile(
    boot_diff,
    [2.5, 97.5]
)


# ============================================================
# 8. PAYMENT RATE
# ============================================================

engaged_paid = int(
    base.loc[
        base["has_prior_engagement"].eq(True),
        "payment_event_72h"
    ].sum()
)

no_engagement_paid = int(
    base.loc[
        base["has_prior_engagement"].eq(False),
        "payment_event_72h"
    ].sum()
)

table = [
    [
        engaged_paid,
        len(engaged) - engaged_paid
    ],
    [
        no_engagement_paid,
        len(no_engagement) - no_engagement_paid
    ]
]

odds_ratio, p_payment = fisher_exact(
    table,
    alternative="greater"
)


# ============================================================
# 9. RESULTS
# ============================================================

print("\n")
print("=" * 100)
print("ENGAGED vs NO ENGAGEMENT — DPD30–60")
print("=" * 100)

print(
    f"""
ENGAGED
Messages             : {len(engaged):,}
Recovery / msg       : R$ {engaged.mean():,.2f}
Payment rate         : {engaged_paid / len(engaged):.2%}

NO ENGAGEMENT
Messages             : {len(no_engagement):,}
Recovery / msg       : R$ {no_engagement.mean():,.2f}
Payment rate         : {no_engagement_paid / len(no_engagement):.2%}

ECONOMIC DIFFERENCE
Engaged - No Eng.    : R$ {diff:,.2f}/msg
Relative lift        : {relative_lift:+.1%}

RECOVERY / MESSAGE
Permutation p-value  : {p_recovery:.6f}
Bootstrap 95% CI     : R$ [{ci_low:,.2f}, {ci_high:,.2f}]

PAYMENT RATE
Fisher p-value       : {p_payment:.6f}
Odds ratio           : {odds_ratio:.2f}
"""
)


# ============================================================
# 10. CHECK ROBUSTNESS BY DPD BAND
#
# Important:
# The operational rule is DPD30–60,
# but we verify that the signal exists in BOTH regions.
# ============================================================

print("\n")
print("=" * 100)
print("ROBUSTNESS CHECK BY DPD BAND")
print("=" * 100)

for dpd_min, dpd_max in [
    (30, 45),
    (46, 60)
]:

    temp = df.loc[
        df["days_past_due"].between(
            dpd_min,
            dpd_max
        )
    ].copy()

    check = (
        temp
        .groupby("has_prior_engagement")
        .agg(
            messages=("customer_id", "size"),
            customers=("customer_id", "nunique"),
            payment_rate=(
                "payment_event_72h",
                "mean"
            ),
            recovery_per_message=(
                "recovery_72h",
                "mean"
            )
        )
        .reset_index()
    )

    check["segment"] = np.where(
        check["has_prior_engagement"],
        "ENGAGED",
        "NO ENGAGEMENT"
    )

    print(
        f"\nDPD {dpd_min}–{dpd_max}"
    )

    print("-" * 70)

    print(
        check[
            [
                "segment",
                "messages",
                "customers",
                "payment_rate",
                "recovery_per_message"
            ]
        ].to_string(
            index=False,
            formatters={
                "payment_rate":
                    lambda x: f"{x:.2%}",

                "recovery_per_message":
                    lambda x: f"R$ {x:,.2f}"
            }
        )
    )


# ============================================================
# 11. FINAL CANDIDATE RULE
# ============================================================

print("\n")
print("=" * 100)
print("FINAL CANDIDATE RULE — DPD30–60")
print("=" * 100)

if (
    p_recovery < 0.05
    and ci_low > 0
):

    print(
        """
✓ Engagement alone strongly separates historical economics.

SIMPLIFIED OPENING RULE

DPD 30–60
    |
    +-- PRIOR ENGAGEMENT = YES --> SEND
    |
    +-- PRIOR ENGAGEMENT = NO  --> NO SEND


REMOVED FROM OPENING DECISION
    - Prior payment
    - Number of contacts in last 14d
    - Template

Template becomes a SECOND decision,
after deciding whether to contact.
"""
    )

else:

    print(
        """
△ Consolidated DPD30–60 evidence is not sufficient
to support engagement alone as the opening rule.
"""
    )

DPD 30–60 | ENGAGEMENT AS THE ONLY OPENING RULE
      segment  messages  customers  payments  recovery_brl recovery_per_message payment_rate avg_balance
NO ENGAGEMENT      1654        679        31  R$ 18,915.16             R$ 11.44        1.87%   R$ 783.54
      ENGAGED     11861       4488       807 R$ 451,823.22             R$ 38.09        6.80%   R$ 788.62


ENGAGED vs NO ENGAGEMENT — DPD30–60

ENGAGED
Messages             : 11,861
Recovery / msg       : R$ 38.09
Payment rate         : 6.80%

NO ENGAGEMENT
Messages             : 1,654
Recovery / msg       : R$ 11.44
Payment rate         : 1.87%

ECONOMIC DIFFERENCE
Engaged - No Eng.    : R$ 26.66/msg
Relative lift        : +233.1%

RECOVERY / MESSAGE
Permutation p-value  : 0.000020
Bootstrap 95% CI     : R$ [20.37, 32.41]

PAYMENT RATE
Fisher p-value       : 0.000000
Odds ratio           : 3.82



ROBUSTNESS CHECK BY DPD BAND

DPD 30–45
----------------------------------------------------------------------
      segment  messages  

In [59]:
# ============================================================
# ENGAGED CUSTOMERS
# TEMPLATE PERFORMANCE BY DPD BUCKET
#
# Population:
#   DPD 30–60
#   PRIOR ENGAGEMENT = YES
#
# Compare:
#   PIX
#   URGENT
#   DISCOUNT
#
# Metrics:
#   messages
#   customers
#   payment rate
#   recovery
#   recovery / message
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. PREPARE DATA
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)


# ============================================================
# 2. ENGAGED ONLY — DPD30–60
# ============================================================

base = df.loc[
    df["days_past_due"].between(30, 60)
    & df["has_prior_engagement"].eq(True)
    & df["template"].isin([
        "friendly_reminder",
        "pix_link",
        "urgent_reminder",
        "discount_offer"
    ])
].copy()


# ============================================================
# 3. DPD BUCKET
# ============================================================

base["dpd_bucket"] = pd.cut(
    base["days_past_due"],
    bins=[
        29,
        34,
        39,
        45,
        50,
        55,
        60
    ],
    labels=[
        "30–34",
        "35–39",
        "40–45",
        "46–50",
        "51–55",
        "56–60"
    ]
)


# ============================================================
# 4. SUMMARY
# ============================================================

summary = (
    base
    .groupby(
        ["dpd_bucket", "template"],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payments=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

summary["payment_rate"] = (
    summary["payments"]
    / summary["messages"]
)

summary["recovery_per_message"] = (
    summary["recovery_brl"]
    / summary["messages"]
)


# ============================================================
# 5. RANK TEMPLATE WITHIN EACH DPD BUCKET
# ============================================================

summary["rank_rpm"] = (
    summary
    .groupby("dpd_bucket", observed=True)[
        "recovery_per_message"
    ]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)


# ============================================================
# 6. FULL TABLE
# ============================================================

print("=" * 115)
print("ENGAGED CUSTOMERS — TEMPLATE PERFORMANCE BY DPD")
print("=" * 115)

print(
    summary[
        [
            "dpd_bucket",
            "template",
            "messages",
            "customers",
            "payments",
            "payment_rate",
            "recovery_brl",
            "recovery_per_message",
            "avg_balance",
            "rank_rpm"
        ]
    ].to_string(
        index=False,
        formatters={
            "payment_rate":
                lambda x: f"{x:.2%}",

            "recovery_brl":
                lambda x: f"R$ {x:,.2f}",

            "recovery_per_message":
                lambda x: f"R$ {x:,.2f}",

            "avg_balance":
                lambda x: f"R$ {x:,.2f}"
        }
    )
)


# ============================================================
# 7. MATRIX — RECOVERY / MESSAGE
# ============================================================

rpm_matrix = (
    summary
    .pivot(
        index="dpd_bucket",
        columns="template",
        values="recovery_per_message"
    )
    .round(2)
)

print("\n")
print("=" * 100)
print("RECOVERY / MESSAGE MATRIX")
print("=" * 100)

display(rpm_matrix)


# ============================================================
# 8. MATRIX — PAYMENT RATE
# ============================================================

payment_matrix = (
    summary
    .pivot(
        index="dpd_bucket",
        columns="template",
        values="payment_rate"
    )
)

print("\n")
print("=" * 100)
print("PAYMENT RATE MATRIX")
print("=" * 100)

display(
    payment_matrix.style.format("{:.2%}")
)


# ============================================================
# 9. SAMPLE SIZE MATRIX
# ============================================================

n_matrix = (
    summary
    .pivot(
        index="dpd_bucket",
        columns="template",
        values="messages"
    )
)

print("\n")
print("=" * 100)
print("SAMPLE SIZE — MESSAGES")
print("=" * 100)

display(n_matrix)


# ============================================================
# 10. OBSERVED WINNER BY DPD
# ============================================================

winner = (
    summary.loc[
        summary["rank_rpm"].eq(1)
    ]
    [
        [
            "dpd_bucket",
            "template",
            "messages",
            "payment_rate",
            "recovery_per_message"
        ]
    ]
    .sort_values("dpd_bucket")
)

print("\n")
print("=" * 100)
print("HIGHEST OBSERVED RECOVERY / MESSAGE BY DPD")
print("=" * 100)

print(
    winner.to_string(
        index=False,
        formatters={
            "payment_rate":
                lambda x: f"{x:.2%}",

            "recovery_per_message":
                lambda x: f"R$ {x:,.2f}"
        }
    )
)

ENGAGED CUSTOMERS — TEMPLATE PERFORMANCE BY DPD
dpd_bucket          template  messages  customers  payments payment_rate recovery_brl recovery_per_message avg_balance  rank_rpm
     30–34    discount_offer       942        887        99       10.51% R$ 55,199.35             R$ 58.60   R$ 830.06         1
     30–34 friendly_reminder       196        196        14        7.14%  R$ 8,613.23             R$ 43.95   R$ 764.86         3
     30–34          pix_link       646        618        55        8.51% R$ 33,207.22             R$ 51.40   R$ 787.24         2
     30–34   urgent_reminder      1086       1007        55        5.06% R$ 40,238.08             R$ 37.05   R$ 807.37         4
     35–39    discount_offer      1148       1044        98        8.54% R$ 52,783.52             R$ 45.98   R$ 797.04         1
     35–39          pix_link       455        441        30        6.59% R$ 19,216.36             R$ 42.23   R$ 770.40         2
     35–39   urgent_reminder       654        617

template,discount_offer,friendly_reminder,pix_link,urgent_reminder
dpd_bucket,,,,
30–34,58.60,43.95,51.40,37.05
35–39,45.98,NaN,42.23,30.07
40–45,53.64,NaN,34.33,29.12
46–50,26.02,NaN,21.69,26.91
51–55,24.04,NaN,24.36,29.65
56–60,35.73,NaN,33.41,32.98




PAYMENT RATE MATRIX


template,discount_offer,friendly_reminder,pix_link,urgent_reminder
dpd_bucket,,,,
30–34,10.51%,7.14%,8.51%,5.06%
35–39,8.54%,nan%,6.59%,5.50%
40–45,10.33%,nan%,7.78%,5.08%
46–50,4.92%,nan%,4.76%,3.95%
51–55,6.75%,nan%,3.93%,4.30%
56–60,6.32%,nan%,4.42%,5.06%




SAMPLE SIZE — MESSAGES


template,discount_offer,friendly_reminder,pix_link,urgent_reminder
dpd_bucket,,,,
30–34,942.00,196.00,646.00,"1,086.00"
35–39,"1,148.00",NaN,455.00,654.00
40–45,"1,200.00",NaN,463.00,669.00
46–50,834.00,NaN,336.00,532.00
51–55,741.00,NaN,331.00,442.00
56–60,601.00,NaN,249.00,336.00




HIGHEST OBSERVED RECOVERY / MESSAGE BY DPD
dpd_bucket        template  messages payment_rate recovery_per_message
     30–34  discount_offer       942       10.51%             R$ 58.60
     35–39  discount_offer      1148        8.54%             R$ 45.98
     40–45  discount_offer      1200       10.33%             R$ 53.64
     46–50 urgent_reminder       532        3.95%             R$ 26.91
     51–55 urgent_reminder       442        4.30%             R$ 29.65
     56–60  discount_offer       601        6.32%             R$ 35.73


In [60]:
# ============================================================
# ENGAGED CUSTOMERS — TEMPLATE BY 2 DPD BUCKETS
#
# Buckets:
#   DPD 30–45
#   DPD 46–60
#
# Population:
#   PRIOR ENGAGEMENT = YES
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. PREPARE
# ------------------------------------------------------------

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)


# ------------------------------------------------------------
# 2. ENGAGED DPD30–60
# ------------------------------------------------------------

base = df.loc[
    df["days_past_due"].between(30, 60)
    & df["has_prior_engagement"].eq(True)
    & df["template"].isin([
        "discount_offer",
        "pix_link",
        "urgent_reminder"
    ])
].copy()


# ------------------------------------------------------------
# 3. TWO BUCKETS
# ------------------------------------------------------------

base["dpd_bucket"] = np.select(
    [
        base["days_past_due"].between(30, 45),
        base["days_past_due"].between(46, 60)
    ],
    [
        "30–45",
        "46–60"
    ],
    default=None
)


# ------------------------------------------------------------
# 4. TEMPLATE PERFORMANCE
# ------------------------------------------------------------

summary = (
    base
    .groupby(
        ["dpd_bucket", "template"],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        payments=("payment_event_72h", "sum"),
        recovery_brl=("recovery_72h", "sum"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

summary["payment_rate"] = (
    summary["payments"]
    / summary["messages"]
)

summary["recovery_per_message"] = (
    summary["recovery_brl"]
    / summary["messages"]
)


# ------------------------------------------------------------
# 5. RANK WITHIN DPD BUCKET
# ------------------------------------------------------------

summary["rank"] = (
    summary
    .groupby("dpd_bucket")["recovery_per_message"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)


# ------------------------------------------------------------
# 6. PRINT FULL RESULTS
# ------------------------------------------------------------

print("=" * 110)
print("ENGAGED CUSTOMERS — TEMPLATE PERFORMANCE")
print("=" * 110)

print(
    summary.sort_values(
        ["dpd_bucket", "rank"]
    ).to_string(
        index=False,
        formatters={
            "payment_rate":
                lambda x: f"{x:.2%}",

            "recovery_brl":
                lambda x: f"R$ {x:,.2f}",

            "recovery_per_message":
                lambda x: f"R$ {x:,.2f}",

            "avg_balance":
                lambda x: f"R$ {x:,.2f}"
        }
    )
)


# ------------------------------------------------------------
# 7. RECOVERY / MESSAGE MATRIX
# ------------------------------------------------------------

rpm = (
    summary
    .pivot(
        index="dpd_bucket",
        columns="template",
        values="recovery_per_message"
    )
)

print("\n")
print("=" * 80)
print("RECOVERY / MESSAGE")
print("=" * 80)

display(
    rpm.style.format("R$ {:.2f}")
)


# ------------------------------------------------------------
# 8. PAYMENT RATE MATRIX
# ------------------------------------------------------------

payment = (
    summary
    .pivot(
        index="dpd_bucket",
        columns="template",
        values="payment_rate"
    )
)

print("\n")
print("=" * 80)
print("PAYMENT RATE")
print("=" * 80)

display(
    payment.style.format("{:.2%}")
)


# ------------------------------------------------------------
# 9. SAMPLE SIZE
# ------------------------------------------------------------

sample = (
    summary
    .pivot(
        index="dpd_bucket",
        columns="template",
        values="messages"
    )
)

print("\n")
print("=" * 80)
print("SAMPLE SIZE — MESSAGES")
print("=" * 80)

display(sample)


# ------------------------------------------------------------
# 10. OBSERVED WINNER
# ------------------------------------------------------------

winner = (
    summary.loc[
        summary["rank"].eq(1)
    ]
    .sort_values("dpd_bucket")
)

print("\n")
print("=" * 80)
print("HIGHEST OBSERVED R$/MESSAGE")
print("=" * 80)

print(
    winner[
        [
            "dpd_bucket",
            "template",
            "messages",
            "payment_rate",
            "recovery_per_message"
        ]
    ].to_string(
        index=False,
        formatters={
            "payment_rate":
                lambda x: f"{x:.2%}",

            "recovery_per_message":
                lambda x: f"R$ {x:,.2f}"
        }
    )
)

ENGAGED CUSTOMERS — TEMPLATE PERFORMANCE
dpd_bucket        template  messages  customers  payments  recovery_brl avg_balance payment_rate recovery_per_message  rank
     30–45  discount_offer      3290       2469       321 R$ 172,348.53   R$ 801.89        9.76%             R$ 52.39     1
     30–45        pix_link      1564       1387       121  R$ 68,317.14   R$ 771.62        7.74%             R$ 43.68     2
     30–45 urgent_reminder      2409       1947       125  R$ 79,382.10   R$ 784.46        5.19%             R$ 32.95     3
     46–60 urgent_reminder      1310       1108        57  R$ 38,504.80   R$ 790.78        4.35%             R$ 29.39     1
     46–60  discount_offer      2176       1635       129  R$ 60,990.68   R$ 779.08        5.93%             R$ 28.03     2
     46–60        pix_link       916        821        40  R$ 23,666.74   R$ 805.62        4.37%             R$ 25.84     3


RECOVERY / MESSAGE


template,discount_offer,pix_link,urgent_reminder
dpd_bucket,,,
30–45,R$ 52.39,R$ 43.68,R$ 32.95
46–60,R$ 28.03,R$ 25.84,R$ 29.39




PAYMENT RATE


template,discount_offer,pix_link,urgent_reminder
dpd_bucket,,,
30–45,9.76%,7.74%,5.19%
46–60,5.93%,4.37%,4.35%




SAMPLE SIZE — MESSAGES


template,discount_offer,pix_link,urgent_reminder
dpd_bucket,,,
30–45,3290,1564,2409
46–60,2176,916,1310




HIGHEST OBSERVED R$/MESSAGE
dpd_bucket        template  messages payment_rate recovery_per_message
     30–45  discount_offer      3290        9.76%             R$ 52.39
     46–60 urgent_reminder      1310        4.35%             R$ 29.39


In [62]:
# ============================================================
# PAIRWISE TEMPLATE TESTS — ENGAGED ONLY
#
# DPD buckets:
#   30–45
#   46–60
#
# Metric:
#   Recovery / message
#
# Method:
#   Two-sided permutation test
#   + Bootstrap 95% CI
#   + Holm correction for multiple comparisons
#
# IMPORTANT:
#   Tests historical association, NOT causal superiority.
# ============================================================

import numpy as np
import pandas as pd
from itertools import combinations
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)

TEMPLATES = [
    "discount_offer",
    "pix_link",
    "urgent_reminder"
]


# ============================================================
# 2. PAIRWISE FUNCTION
# ============================================================

def pairwise_template_tests(
    data,
    dpd_min,
    dpd_max,
    n_perm=50_000,
    n_boot=20_000,
    seed=42
):

    rng = np.random.default_rng(seed)

    d = data.loc[
        data["days_past_due"].between(dpd_min, dpd_max)
        & data["has_prior_engagement"].eq(True)
        & data["template"].isin(TEMPLATES)
    ].copy()

    results = []


    # --------------------------------------------------------
    # TEST EVERY TEMPLATE PAIR
    # --------------------------------------------------------

    for template_a, template_b in combinations(
        TEMPLATES,
        2
    ):

        a = d.loc[
            d["template"].eq(template_a),
            "recovery_72h"
        ].to_numpy()

        b = d.loc[
            d["template"].eq(template_b),
            "recovery_72h"
        ].to_numpy()


        # ----------------------------------------------------
        # OBSERVED DIFFERENCE
        # ----------------------------------------------------

        mean_a = a.mean()
        mean_b = b.mean()

        diff = mean_a - mean_b


        # ----------------------------------------------------
        # TWO-SIDED PERMUTATION TEST
        # ----------------------------------------------------

        pooled = np.concatenate([a, b])

        n_a = len(a)

        perm_diffs = np.empty(n_perm)

        for i in range(n_perm):

            shuffled = rng.permutation(pooled)

            perm_diffs[i] = (
                shuffled[:n_a].mean()
                -
                shuffled[n_a:].mean()
            )

        p_value = (
            np.sum(
                np.abs(perm_diffs)
                >=
                abs(diff)
            ) + 1
        ) / (
            n_perm + 1
        )


        # ----------------------------------------------------
        # BOOTSTRAP CI
        # ----------------------------------------------------

        boot_diff = np.empty(n_boot)

        for i in range(n_boot):

            a_boot = rng.choice(
                a,
                size=len(a),
                replace=True
            )

            b_boot = rng.choice(
                b,
                size=len(b),
                replace=True
            )

            boot_diff[i] = (
                a_boot.mean()
                -
                b_boot.mean()
            )

        ci_low, ci_high = np.percentile(
            boot_diff,
            [2.5, 97.5]
        )


        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        results.append({

            "comparison":
                f"{template_a} vs {template_b}",

            "template_A":
                template_a,

            "template_B":
                template_b,

            "n_A":
                len(a),

            "n_B":
                len(b),

            "rpm_A":
                mean_a,

            "rpm_B":
                mean_b,

            "difference_A_minus_B":
                diff,

            "relative_difference":
                (
                    mean_a / mean_b - 1
                    if mean_b != 0
                    else np.nan
                ),

            "ci_low":
                ci_low,

            "ci_high":
                ci_high,

            "p_raw":
                p_value
        })


    results = pd.DataFrame(results)


    # ========================================================
    # 3. HOLM CORRECTION
    # ========================================================

    reject, p_holm, _, _ = multipletests(
        results["p_raw"],
        alpha=0.05,
        method="holm"
    )

    results["p_holm"] = p_holm

    results["significant_5pct"] = reject


    # ========================================================
    # 4. PRINT
    # ========================================================

    print("\n")
    print("=" * 110)
    print(
        f"DPD {dpd_min}–{dpd_max} | ENGAGED | "
        "PAIRWISE TEMPLATE TESTS"
    )
    print("=" * 110)

    for _, row in results.iterrows():

        print(
            f"""
{row['template_A']}  vs  {row['template_B']}
----------------------------------------------------------------------
R$/msg A             : R$ {row['rpm_A']:,.2f}
R$/msg B             : R$ {row['rpm_B']:,.2f}

Difference A - B     : R$ {row['difference_A_minus_B']:,.2f}
Relative difference : {row['relative_difference']:+.1%}

95% bootstrap CI     : R$ [{row['ci_low']:,.2f}, {row['ci_high']:,.2f}]

Raw p-value          : {row['p_raw']:.4f}
Holm-adjusted p      : {row['p_holm']:.4f}

Significant @ 5%     : {"YES" if row['significant_5pct'] else "NO"}
"""
        )

    return results


# ============================================================
# 5. DPD 30–45
# ============================================================

results_30_45 = pairwise_template_tests(
    df,
    30,
    45
)


# ============================================================
# 6. DPD 46–60
# ============================================================

results_46_60 = pairwise_template_tests(
    df,
    46,
    60
)


# ============================================================
# 7. FINAL TABLE
# ============================================================

results_30_45["dpd_bucket"] = "30–45"
results_46_60["dpd_bucket"] = "46–60"

final_tests = pd.concat(
    [
        results_30_45,
        results_46_60
    ],
    ignore_index=True
)

print("\n")
print("=" * 110)
print("FINAL PAIRWISE RESULTS")
print("=" * 110)

display(
    final_tests[
        [
            "dpd_bucket",
            "comparison",
            "n_A",
            "n_B",
            "rpm_A",
            "rpm_B",
            "difference_A_minus_B",
            "ci_low",
            "ci_high",
            "p_raw",
            "p_holm",
            "significant_5pct"
        ]
    ].style.format({

        "rpm_A":
            "R$ {:.2f}",

        "rpm_B":
            "R$ {:.2f}",

        "difference_A_minus_B":
            "R$ {:.2f}",

        "ci_low":
            "R$ {:.2f}",

        "ci_high":
            "R$ {:.2f}",

        "p_raw":
            "{:.4f}",

        "p_holm":
            "{:.4f}"
    })
)



DPD 30–45 | ENGAGED | PAIRWISE TEMPLATE TESTS

discount_offer  vs  pix_link
----------------------------------------------------------------------
R$/msg A             : R$ 52.39
R$/msg B             : R$ 43.68

Difference A - B     : R$ 8.70
Relative difference : +19.9%

95% bootstrap CI     : R$ [-3.89, 20.81]

Raw p-value          : 0.1683
Holm-adjusted p      : 0.1683

Significant @ 5%     : NO


discount_offer  vs  urgent_reminder
----------------------------------------------------------------------
R$/msg A             : R$ 52.39
R$/msg B             : R$ 32.95

Difference A - B     : R$ 19.43
Relative difference : +59.0%

95% bootstrap CI     : R$ [9.36, 29.39]

Raw p-value          : 0.0001
Holm-adjusted p      : 0.0003

Significant @ 5%     : YES


pix_link  vs  urgent_reminder
----------------------------------------------------------------------
R$/msg A             : R$ 43.68
R$/msg B             : R$ 32.95

Difference A - B     : R$ 10.73
Relative difference : +32.6%

9

,dpd_bucket,comparison,n_A,n_B,rpm_A,rpm_B,difference_A_minus_B,ci_low,ci_high,p_raw,p_holm,significant_5pct
0,30–45,discount_offer vs pix_link,3290,1564,R$ 52.39,R$ 43.68,R$ 8.70,R$ -3.89,R$ 20.81,0.1683,0.1683,False
1,30–45,discount_offer vs urgent_reminder,3290,2409,R$ 52.39,R$ 32.95,R$ 19.43,R$ 9.36,R$ 29.39,0.0001,0.0003,True
2,30–45,pix_link vs urgent_reminder,1564,2409,R$ 43.68,R$ 32.95,R$ 10.73,R$ -1.20,R$ 23.15,0.0759,0.1518,False
3,46–60,discount_offer vs pix_link,2176,916,R$ 28.03,R$ 25.84,R$ 2.19,R$ -10.51,R$ 13.76,0.7161,1.0000,False
4,46–60,discount_offer vs urgent_reminder,2176,1310,R$ 28.03,R$ 29.39,R$ -1.36,R$ -12.66,R$ 9.44,0.8006,1.0000,False
5,46–60,pix_link vs urgent_reminder,916,1310,R$ 25.84,R$ 29.39,R$ -3.56,R$ -17.45,R$ 10.77,0.6261,1.0000,False


In [63]:
# ============================================================
# QA — POPULAÇÃO DOS TESTES
# ============================================================

for lo, hi in [(30, 45), (46, 60)]:

    x = df.loc[
        df["days_past_due"].between(lo, hi)
        & df["template"].isin(TEMPLATES)
    ].copy()

    print("=" * 80)
    print(f"DPD {lo}–{hi}")
    print("=" * 80)

    print(f"All messages       : {len(x):,}")
    print(
        f"Prior engaged      : "
        f"{x['has_prior_engagement'].eq(True).sum():,}"
    )
    print(
        f"No prior engagement: "
        f"{x['has_prior_engagement'].eq(False).sum():,}"
    )

    print("\nTemplates — PRIOR ENGAGED:")

    print(
        x.loc[x["has_prior_engagement"].eq(True)]
        ["template"]
        .value_counts()
    )

    print()

DPD 30–45
All messages       : 8,335
Prior engaged      : 7,263
No prior engagement: 1,072

Templates — PRIOR ENGAGED:
template
discount_offer     3290
urgent_reminder    2409
pix_link           1564
Name: count, dtype: int64

DPD 46–60
All messages       : 4,963
Prior engaged      : 4,402
No prior engagement: 561

Templates — PRIOR ENGAGED:
template
discount_offer     2176
urgent_reminder    1310
pix_link            916
Name: count, dtype: int64



In [64]:
# ============================================================
# PRIOR ENGAGEMENT vs NO PRIOR ENGAGEMENT
#
# Buckets:
#   DPD 31–45
#   DPD 46–60
#
# Metric:
#   Recovery / message
#
# Method:
#   Two-sided permutation test
#   Bootstrap 95% CI
#   Holm correction across the 2 DPD buckets
#
# IMPORTANT:
#   Historical association, NOT causality.
# ============================================================

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)


# ============================================================
# 2. FUNCTION
# ============================================================

def engagement_test(
    data,
    dpd_min,
    dpd_max,
    n_perm=50_000,
    n_boot=20_000,
    seed=42
):

    rng = np.random.default_rng(seed)

    d = data.loc[
        data["days_past_due"].between(dpd_min, dpd_max)
        & data["has_prior_engagement"].notna()
    ].copy()

    engaged = d.loc[
        d["has_prior_engagement"].eq(True),
        "recovery_72h"
    ].to_numpy()

    no_engaged = d.loc[
        d["has_prior_engagement"].eq(False),
        "recovery_72h"
    ].to_numpy()


    # ========================================================
    # OBSERVED
    # ========================================================

    mean_engaged = engaged.mean()
    mean_no_engaged = no_engaged.mean()

    diff = mean_engaged - mean_no_engaged

    relative_diff = (
        mean_engaged / mean_no_engaged - 1
        if mean_no_engaged != 0
        else np.nan
    )


    # ========================================================
    # PERMUTATION TEST
    # ========================================================

    pooled = np.concatenate([
        engaged,
        no_engaged
    ])

    n_engaged = len(engaged)

    perm_diffs = np.empty(n_perm)

    for i in range(n_perm):

        shuffled = rng.permutation(pooled)

        perm_diffs[i] = (
            shuffled[:n_engaged].mean()
            -
            shuffled[n_engaged:].mean()
        )

    p_value = (
        np.sum(
            np.abs(perm_diffs) >= abs(diff)
        ) + 1
    ) / (
        n_perm + 1
    )


    # ========================================================
    # BOOTSTRAP CI
    # ========================================================

    boot_diff = np.empty(n_boot)

    for i in range(n_boot):

        engaged_boot = rng.choice(
            engaged,
            size=len(engaged),
            replace=True
        )

        no_engaged_boot = rng.choice(
            no_engaged,
            size=len(no_engaged),
            replace=True
        )

        boot_diff[i] = (
            engaged_boot.mean()
            -
            no_engaged_boot.mean()
        )

    ci_low, ci_high = np.percentile(
        boot_diff,
        [2.5, 97.5]
    )


    return {
        "dpd_bucket": f"{dpd_min}–{dpd_max}",

        "n_engaged": len(engaged),
        "n_no_engagement": len(no_engaged),

        "rpm_engaged": mean_engaged,
        "rpm_no_engagement": mean_no_engaged,

        "difference": diff,
        "relative_difference": relative_diff,

        "ci_low": ci_low,
        "ci_high": ci_high,

        "p_raw": p_value
    }


# ============================================================
# 3. RUN BOTH BUCKETS
# ============================================================

results = pd.DataFrame([

    engagement_test(
        df,
        31,
        45,
        seed=42
    ),

    engagement_test(
        df,
        46,
        60,
        seed=43
    )

])


# ============================================================
# 4. HOLM CORRECTION — 2 TESTS
# ============================================================

reject, p_holm, _, _ = multipletests(
    results["p_raw"],
    alpha=0.05,
    method="holm"
)

results["p_holm"] = p_holm
results["significant_5pct"] = reject


# ============================================================
# 5. PRINT RESULTS
# ============================================================

for _, row in results.iterrows():

    print("\n")
    print("=" * 90)
    print(
        f"DPD {row['dpd_bucket']} | "
        "PRIOR ENGAGEMENT vs NO PRIOR ENGAGEMENT"
    )
    print("=" * 90)

    print(
        f"""
Engaged N               : {row['n_engaged']:,.0f}
No engagement N         : {row['n_no_engagement']:,.0f}

R$/msg — Engaged        : R$ {row['rpm_engaged']:,.2f}
R$/msg — No engagement  : R$ {row['rpm_no_engagement']:,.2f}

Difference              : R$ {row['difference']:,.2f}
Relative difference     : {row['relative_difference']:+.1%}

95% bootstrap CI        : R$ [{row['ci_low']:,.2f}, {row['ci_high']:,.2f}]

Raw p-value             : {row['p_raw']:.4f}
Holm-adjusted p         : {row['p_holm']:.4f}

Significant @ 5%        : {"YES" if row['significant_5pct'] else "NO"}
"""
    )


# ============================================================
# 6. FINAL TABLE
# ============================================================

display(
    results.style.format({

        "rpm_engaged": "R$ {:.2f}",
        "rpm_no_engagement": "R$ {:.2f}",

        "difference": "R$ {:.2f}",
        "relative_difference": "{:+.1%}",

        "ci_low": "R$ {:.2f}",
        "ci_high": "R$ {:.2f}",

        "p_raw": "{:.4f}",
        "p_holm": "{:.4f}"
    })
)



DPD 31–45 | PRIOR ENGAGEMENT vs NO PRIOR ENGAGEMENT

Engaged N               : 6,519
No engagement N         : 955

R$/msg — Engaged        : R$ 44.66
R$/msg — No engagement  : R$ 15.43

Difference              : R$ 29.23
Relative difference     : +189.5%

95% bootstrap CI        : R$ [19.30, 38.12]

Raw p-value             : 0.0000
Holm-adjusted p         : 0.0000

Significant @ 5%        : YES



DPD 46–60 | PRIOR ENGAGEMENT vs NO PRIOR ENGAGEMENT

Engaged N               : 4,402
No engagement N         : 561

R$/msg — Engaged        : R$ 27.98
R$/msg — No engagement  : R$ 3.73

Difference              : R$ 24.25
Relative difference     : +650.6%

95% bootstrap CI        : R$ [17.00, 30.60]

Raw p-value             : 0.0006
Holm-adjusted p         : 0.0006

Significant @ 5%        : YES



,dpd_bucket,n_engaged,n_no_engagement,rpm_engaged,rpm_no_engagement,difference,relative_difference,ci_low,ci_high,p_raw,p_holm,significant_5pct
0,31–45,6519,955,R$ 44.66,R$ 15.43,R$ 29.23,+189.5%,R$ 19.30,R$ 38.12,0.0000,0.0000,True
1,46–60,4402,561,R$ 27.98,R$ 3.73,R$ 24.25,+650.6%,R$ 17.00,R$ 30.60,0.0006,0.0006,True


In [65]:
# ============================================================
# DPD 31–45 | PRIOR ENGAGED
# PRIOR PAYMENT vs NO PRIOR PAYMENT
#
# Metric:
#   Recovery / message
#
# Method:
#   Two-sided permutation test
#   Bootstrap 95% CI
#
# IMPORTANT:
#   Historical association, NOT causality.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)


# ============================================================
# 2. FILTER
# DPD 31–45 + PRIOR ENGAGEMENT
# ============================================================

d = df.loc[
    df["days_past_due"].between(31, 45)
    & df["has_prior_engagement"].eq(True)
    & df["has_prior_payment"].notna()
].copy()


# ============================================================
# 3. SPLIT
# ============================================================

prior_payment = d.loc[
    d["has_prior_payment"].eq(True),
    "recovery_72h"
].to_numpy()

no_prior_payment = d.loc[
    d["has_prior_payment"].eq(False),
    "recovery_72h"
].to_numpy()


# ============================================================
# 4. DESCRIPTIVE
# ============================================================

mean_prior = prior_payment.mean()
mean_no_prior = no_prior_payment.mean()

diff = mean_prior - mean_no_prior

relative_diff = (
    mean_prior / mean_no_prior - 1
    if mean_no_prior != 0
    else np.nan
)

print("=" * 90)
print("DPD 31–45 | PRIOR ENGAGED")
print("PRIOR PAYMENT vs NO PRIOR PAYMENT")
print("=" * 90)

print(f"""
Total messages            : {len(d):,}

Prior payment N           : {len(prior_payment):,}
No prior payment N        : {len(no_prior_payment):,}

R$/msg — Prior payment    : R$ {mean_prior:,.2f}
R$/msg — No prior payment : R$ {mean_no_prior:,.2f}

Difference                : R$ {diff:,.2f}
Relative difference       : {relative_diff:+.1%}
""")


# ============================================================
# 5. TWO-SIDED PERMUTATION TEST
# ============================================================

N_PERM = 50_000
SEED = 42

rng = np.random.default_rng(SEED)

pooled = np.concatenate([
    prior_payment,
    no_prior_payment
])

n_prior = len(prior_payment)

perm_diffs = np.empty(N_PERM)

for i in range(N_PERM):

    shuffled = rng.permutation(pooled)

    perm_diffs[i] = (
        shuffled[:n_prior].mean()
        -
        shuffled[n_prior:].mean()
    )

p_value = (
    np.sum(
        np.abs(perm_diffs) >= abs(diff)
    ) + 1
) / (
    N_PERM + 1
)


# ============================================================
# 6. BOOTSTRAP 95% CI
# ============================================================

N_BOOT = 20_000

boot_diffs = np.empty(N_BOOT)

for i in range(N_BOOT):

    prior_boot = rng.choice(
        prior_payment,
        size=len(prior_payment),
        replace=True
    )

    no_prior_boot = rng.choice(
        no_prior_payment,
        size=len(no_prior_payment),
        replace=True
    )

    boot_diffs[i] = (
        prior_boot.mean()
        -
        no_prior_boot.mean()
    )

ci_low, ci_high = np.percentile(
    boot_diffs,
    [2.5, 97.5]
)


# ============================================================
# 7. FINAL RESULT
# ============================================================

print("-" * 90)

print(f"""
Difference:
Prior payment - No prior payment

Observed difference       : R$ {diff:,.2f}
95% bootstrap CI          : R$ [{ci_low:,.2f}, {ci_high:,.2f}]

Permutation p-value       : {p_value:.4f}

Significant @ 5%          : {"YES" if p_value < 0.05 else "NO"}
""")


# ============================================================
# 8. PAYMENT RATE AS SECONDARY QA
# ============================================================

payment_summary = (
    d.groupby("has_prior_payment")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        recovery_72h=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        payment_rate_72h=("payment_event_72h", "mean")
    )
    .reset_index()
)

payment_summary["group"] = np.where(
    payment_summary["has_prior_payment"],
    "Prior payment",
    "No prior payment"
)

print("=" * 90)
print("DESCRIPTIVE SUMMARY")
print("=" * 90)

display(
    payment_summary[
        [
            "group",
            "messages",
            "customers",
            "recovery_72h",
            "recovery_per_message",
            "payment_rate_72h"
        ]
    ].style.format({
        "recovery_72h": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:.2f}",
        "payment_rate_72h": "{:.2%}"
    })
)

DPD 31–45 | PRIOR ENGAGED
PRIOR PAYMENT vs NO PRIOR PAYMENT

Total messages            : 6,519

Prior payment N           : 1,118
No prior payment N        : 5,401

R$/msg — Prior payment    : R$ 38.13
R$/msg — No prior payment : R$ 46.01

Difference                : R$ -7.88
Relative difference       : -17.1%

------------------------------------------------------------------------------------------

Difference:
Prior payment - No prior payment

Observed difference       : R$ -7.88
95% bootstrap CI          : R$ [-16.89, 1.28]

Permutation p-value       : 0.2211

Significant @ 5%          : NO

DESCRIPTIVE SUMMARY


,group,messages,customers,recovery_72h,recovery_per_message,payment_rate_72h
0,No prior payment,5401,3125,"R$ 248,510.11",R$ 46.01,6.74%
1,Prior payment,1118,654,"R$ 42,632.18",R$ 38.13,14.40%


In [66]:
# ============================================================
# DPD 31–45
# ENGAGEMENT × PRIOR PAYMENT × TEMPLATE
#
# Metric:
#   Recovery / message
#
# Output:
#   Table used in decision slide
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)

TEMPLATES = [
    "pix_link",
    "urgent_reminder",
    "discount_offer"
]


# ============================================================
# 2. DPD 31–45 ONLY
# ============================================================

d = df.loc[
    df["days_past_due"].between(31, 45)
].copy()

print("=" * 100)
print("DPD 31–45")
print("=" * 100)

print(f"Messages  : {len(d):,}")
print(f"Customers : {d['customer_id'].nunique():,}")


# ============================================================
# 3. A — ENGAGED vs NO ENGAGEMENT
# ============================================================

engagement_summary = (
    d.loc[d["has_prior_engagement"].notna()]
    .groupby("has_prior_engagement")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        payment_rate=("payment_event_72h", "mean")
    )
    .reset_index()
)

engagement_summary["segment"] = np.where(
    engagement_summary["has_prior_engagement"],
    "ENGAGED",
    "NO ENGAGEMENT"
)

print("\n" + "=" * 100)
print("A. ENGAGEMENT")
print("=" * 100)

display(
    engagement_summary[
        [
            "segment",
            "messages",
            "customers",
            "recovery",
            "recovery_per_message",
            "payment_rate"
        ]
    ].style.format({
        "recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:.2f}",
        "payment_rate": "{:.2%}"
    })
)


# ============================================================
# 4. B — ENGAGED ONLY
# ============================================================

engaged = d.loc[
    d["has_prior_engagement"].eq(True)
    & d["has_prior_payment"].notna()
    & d["template"].isin(TEMPLATES)
].copy()


# ============================================================
# 5. PRIOR PAYMENT × TEMPLATE
# ============================================================

template_summary = (
    engaged
    .groupby(
        ["has_prior_payment", "template"]
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        payment_rate=("payment_event_72h", "mean")
    )
    .reset_index()
)

template_summary["prior_payment_group"] = np.where(
    template_summary["has_prior_payment"],
    "Prior payment",
    "No prior payment"
)

print("\n" + "=" * 100)
print("B. ENGAGED — PRIOR PAYMENT × TEMPLATE")
print("=" * 100)

display(
    template_summary[
        [
            "prior_payment_group",
            "template",
            "messages",
            "customers",
            "recovery",
            "recovery_per_message",
            "payment_rate"
        ]
    ].style.format({
        "recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:.2f}",
        "payment_rate": "{:.2%}"
    })
)


# ============================================================
# 6. TABLE — SAME STRUCTURE AS SLIDE
# ============================================================

slide_table = (
    template_summary
    .pivot(
        index="prior_payment_group",
        columns="template",
        values="recovery_per_message"
    )
    .reindex(
        ["Prior payment", "No prior payment"]
    )
    .rename(
        columns={
            "pix_link": "PIX",
            "urgent_reminder": "URGENT",
            "discount_offer": "DISCOUNT"
        }
    )
)

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE — SLIDE TABLE")
print("=" * 100)

display(
    slide_table.style.format(
        "R$ {:.2f}"
    )
)


# ============================================================
# 7. N — SAME STRUCTURE AS SLIDE
# ============================================================

n_table = (
    template_summary
    .pivot(
        index="prior_payment_group",
        columns="template",
        values="messages"
    )
    .reindex(
        ["Prior payment", "No prior payment"]
    )
    .rename(
        columns={
            "pix_link": "PIX",
            "urgent_reminder": "URGENT",
            "discount_offer": "DISCOUNT"
        }
    )
)

print("\n" + "=" * 100)
print("N MESSAGES — SLIDE TABLE")
print("=" * 100)

display(n_table)


# ============================================================
# 8. COMBINED TABLE — R$/MSG + N
# ============================================================

combined = slide_table.copy().astype(object)

for row in slide_table.index:
    for col in slide_table.columns:

        rpm = slide_table.loc[row, col]
        n = n_table.loc[row, col]

        combined.loc[row, col] = (
            f"R$ {rpm:,.2f}\n"
            f"n={int(n):,}"
        )

print("\n" + "=" * 100)
print("FINAL TABLE — READY FOR SLIDE")
print("=" * 100)

display(combined)


DPD 31–45
Messages  : 7,474
Customers : 4,269

A. ENGAGEMENT


,segment,messages,customers,recovery,recovery_per_message,payment_rate
0,NO ENGAGEMENT,955,565,"R$ 14,733.78",R$ 15.43,2.41%
1,ENGAGED,6519,3750,"R$ 291,142.29",R$ 44.66,8.05%



B. ENGAGED — PRIOR PAYMENT × TEMPLATE


,prior_payment_group,template,messages,customers,recovery,recovery_per_message,payment_rate
0,No prior payment,discount_offer,2737,2069,"R$ 148,938.36",R$ 54.42,8.22%
1,No prior payment,pix_link,1079,975,"R$ 48,568.32",R$ 45.01,6.58%
2,No prior payment,urgent_reminder,1585,1329,"R$ 51,003.43",R$ 32.18,4.29%
3,Prior payment,discount_offer,553,413,"R$ 23,410.17",R$ 42.33,17.36%
4,Prior payment,pix_link,238,215,"R$ 8,550.00",R$ 35.92,14.29%
5,Prior payment,urgent_reminder,327,270,"R$ 10,672.01",R$ 32.64,9.48%



RECOVERY / MESSAGE — SLIDE TABLE


template,DISCOUNT,PIX,URGENT
prior_payment_group,,,
Prior payment,R$ 42.33,R$ 35.92,R$ 32.64
No prior payment,R$ 54.42,R$ 45.01,R$ 32.18



N MESSAGES — SLIDE TABLE


template,DISCOUNT,PIX,URGENT
prior_payment_group,,,
Prior payment,553,238,327
No prior payment,2737,1079,1585



FINAL TABLE — READY FOR SLIDE


template,DISCOUNT,PIX,URGENT
prior_payment_group,,,
Prior payment,R$ 42.33\nn=553,R$ 35.92\nn=238,R$ 32.64\nn=327
No prior payment,"R$ 54.42\nn=2,737","R$ 45.01\nn=1,079","R$ 32.18\nn=1,585"


In [67]:
# ============================================================
# DPD 46–60
# ENGAGEMENT × PRIOR PAYMENT × TEMPLATE
#
# Metric:
#   Recovery / message
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)

TEMPLATES = [
    "pix_link",
    "urgent_reminder",
    "discount_offer"
]


# ============================================================
# 2. DPD 46–60 ONLY
# ============================================================

d = df.loc[
    df["days_past_due"].between(46, 60)
].copy()

print("=" * 100)
print("DPD 46–60")
print("=" * 100)

print(f"Messages  : {len(d):,}")
print(f"Customers : {d['customer_id'].nunique():,}")


# ============================================================
# 3. A — ENGAGED vs NO ENGAGEMENT
# ============================================================

engagement_summary = (
    d.loc[d["has_prior_engagement"].notna()]
    .groupby("has_prior_engagement")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        payment_rate=("payment_event_72h", "mean")
    )
    .reset_index()
)

engagement_summary["segment"] = np.where(
    engagement_summary["has_prior_engagement"],
    "ENGAGED",
    "NO ENGAGEMENT"
)

print("\n" + "=" * 100)
print("A. ENGAGEMENT")
print("=" * 100)

display(
    engagement_summary[
        [
            "segment",
            "messages",
            "customers",
            "recovery",
            "recovery_per_message",
            "payment_rate"
        ]
    ].style.format({
        "recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:.2f}",
        "payment_rate": "{:.2%}"
    })
)


# ============================================================
# 4. B — ENGAGED ONLY
# ============================================================

engaged = d.loc[
    d["has_prior_engagement"].eq(True)
    & d["has_prior_payment"].notna()
    & d["template"].isin(TEMPLATES)
].copy()


# ============================================================
# 5. PRIOR PAYMENT × TEMPLATE
# ============================================================

template_summary = (
    engaged
    .groupby(
        ["has_prior_payment", "template"]
    )
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        payment_rate=("payment_event_72h", "mean")
    )
    .reset_index()
)

template_summary["prior_payment_group"] = np.where(
    template_summary["has_prior_payment"],
    "Prior payment",
    "No prior payment"
)

print("\n" + "=" * 100)
print("B. ENGAGED — PRIOR PAYMENT × TEMPLATE")
print("=" * 100)

display(
    template_summary[
        [
            "prior_payment_group",
            "template",
            "messages",
            "customers",
            "recovery",
            "recovery_per_message",
            "payment_rate"
        ]
    ].style.format({
        "recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:.2f}",
        "payment_rate": "{:.2%}"
    })
)


# ============================================================
# 6. RECOVERY / MESSAGE TABLE
# ============================================================

slide_table_46_60 = (
    template_summary
    .pivot(
        index="prior_payment_group",
        columns="template",
        values="recovery_per_message"
    )
    .reindex([
        "Prior payment",
        "No prior payment"
    ])
    .rename(
        columns={
            "pix_link": "PIX",
            "urgent_reminder": "URGENT",
            "discount_offer": "DISCOUNT"
        }
    )
)

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE — DPD 46–60")
print("=" * 100)

display(
    slide_table_46_60.style.format(
        "R$ {:.2f}"
    )
)


# ============================================================
# 7. N MESSAGES TABLE
# ============================================================

n_table_46_60 = (
    template_summary
    .pivot(
        index="prior_payment_group",
        columns="template",
        values="messages"
    )
    .reindex([
        "Prior payment",
        "No prior payment"
    ])
    .rename(
        columns={
            "pix_link": "PIX",
            "urgent_reminder": "URGENT",
            "discount_offer": "DISCOUNT"
        }
    )
)

print("\n" + "=" * 100)
print("N MESSAGES — DPD 46–60")
print("=" * 100)

display(n_table_46_60)


# ============================================================
# 8. FINAL TABLE — R$/MSG + N
# ============================================================

combined_46_60 = slide_table_46_60.copy().astype(object)

for row in slide_table_46_60.index:

    for col in slide_table_46_60.columns:

        rpm = slide_table_46_60.loc[row, col]
        n = n_table_46_60.loc[row, col]

        combined_46_60.loc[row, col] = (
            f"R$ {rpm:,.2f}\n"
            f"n={int(n):,}"
        )

print("\n" + "=" * 100)
print("FINAL TABLE — DPD 46–60")
print("=" * 100)

display(combined_46_60)

DPD 46–60
Messages  : 4,963
Customers : 2,868

A. ENGAGEMENT


,segment,messages,customers,recovery,recovery_per_message,payment_rate
0,NO ENGAGEMENT,561,321,"R$ 2,091.09",R$ 3.73,0.53%
1,ENGAGED,4402,2551,"R$ 123,162.22",R$ 27.98,5.13%



B. ENGAGED — PRIOR PAYMENT × TEMPLATE


,prior_payment_group,template,messages,customers,recovery,recovery_per_message,payment_rate
0,No prior payment,discount_offer,1792,1341,"R$ 49,580.23",R$ 27.67,4.69%
1,No prior payment,pix_link,760,678,"R$ 20,229.32",R$ 26.62,3.55%
2,No prior payment,urgent_reminder,1044,877,"R$ 32,945.94",R$ 31.56,3.83%
3,Prior payment,discount_offer,384,299,"R$ 11,410.45",R$ 29.71,11.72%
4,Prior payment,pix_link,156,143,"R$ 3,437.42",R$ 22.03,8.33%
5,Prior payment,urgent_reminder,266,231,"R$ 5,558.86",R$ 20.90,6.39%



RECOVERY / MESSAGE — DPD 46–60


template,DISCOUNT,PIX,URGENT
prior_payment_group,,,
Prior payment,R$ 29.71,R$ 22.03,R$ 20.90
No prior payment,R$ 27.67,R$ 26.62,R$ 31.56



N MESSAGES — DPD 46–60


template,DISCOUNT,PIX,URGENT
prior_payment_group,,,
Prior payment,384,156,266
No prior payment,1792,760,1044



FINAL TABLE — DPD 46–60


template,DISCOUNT,PIX,URGENT
prior_payment_group,,,
Prior payment,R$ 29.71\nn=384,R$ 22.03\nn=156,R$ 20.90\nn=266
No prior payment,"R$ 27.67\nn=1,792",R$ 26.62\nn=760,"R$ 31.56\nn=1,044"


In [68]:
# ============================================================
# DPD 46–60 | PRIOR ENGAGED
# PRIOR PAYMENT vs NO PRIOR PAYMENT
#
# Metric:
#   Recovery / message
#
# Method:
#   Two-sided permutation test
#   Bootstrap 95% CI
#
# IMPORTANT:
#   Historical association, NOT causality.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)


# ============================================================
# 2. FILTER
# DPD 46–60 + PRIOR ENGAGEMENT
# ============================================================

d = df.loc[
    df["days_past_due"].between(46, 60)
    & df["has_prior_engagement"].eq(True)
    & df["has_prior_payment"].notna()
].copy()


# ============================================================
# 3. SPLIT
# ============================================================

prior_payment = d.loc[
    d["has_prior_payment"].eq(True),
    "recovery_72h"
].to_numpy()

no_prior_payment = d.loc[
    d["has_prior_payment"].eq(False),
    "recovery_72h"
].to_numpy()


# ============================================================
# 4. DESCRIPTIVE
# ============================================================

mean_prior = prior_payment.mean()
mean_no_prior = no_prior_payment.mean()

diff = mean_prior - mean_no_prior

relative_diff = (
    mean_prior / mean_no_prior - 1
    if mean_no_prior != 0
    else np.nan
)

print("=" * 90)
print("DPD 46–60 | PRIOR ENGAGED")
print("PRIOR PAYMENT vs NO PRIOR PAYMENT")
print("=" * 90)

print(f"""
Total messages            : {len(d):,}

Prior payment N           : {len(prior_payment):,}
No prior payment N        : {len(no_prior_payment):,}

R$/msg — Prior payment    : R$ {mean_prior:,.2f}
R$/msg — No prior payment : R$ {mean_no_prior:,.2f}

Difference                : R$ {diff:,.2f}
Relative difference       : {relative_diff:+.1%}
""")


# ============================================================
# 5. TWO-SIDED PERMUTATION TEST
# ============================================================

N_PERM = 50_000
N_BOOT = 20_000
SEED = 42

rng = np.random.default_rng(SEED)

pooled = np.concatenate([
    prior_payment,
    no_prior_payment
])

n_prior = len(prior_payment)

perm_diffs = np.empty(N_PERM)

for i in range(N_PERM):

    shuffled = rng.permutation(pooled)

    perm_diffs[i] = (
        shuffled[:n_prior].mean()
        -
        shuffled[n_prior:].mean()
    )

p_value = (
    np.sum(
        np.abs(perm_diffs) >= abs(diff)
    ) + 1
) / (
    N_PERM + 1
)


# ============================================================
# 6. BOOTSTRAP 95% CI
# ============================================================

boot_diffs = np.empty(N_BOOT)

for i in range(N_BOOT):

    prior_boot = rng.choice(
        prior_payment,
        size=len(prior_payment),
        replace=True
    )

    no_prior_boot = rng.choice(
        no_prior_payment,
        size=len(no_prior_payment),
        replace=True
    )

    boot_diffs[i] = (
        prior_boot.mean()
        -
        no_prior_boot.mean()
    )

ci_low, ci_high = np.percentile(
    boot_diffs,
    [2.5, 97.5]
)


# ============================================================
# 7. FINAL RESULT
# ============================================================

print("-" * 90)

print(f"""
Difference:
Prior payment - No prior payment

Observed difference       : R$ {diff:,.2f}
95% bootstrap CI          : R$ [{ci_low:,.2f}, {ci_high:,.2f}]

Permutation p-value       : {p_value:.4f}

Significant @ 5%          : {"YES" if p_value < 0.05 else "NO"}
""")


# ============================================================
# 8. DESCRIPTIVE SUMMARY
# ============================================================

payment_summary = (
    d.groupby("has_prior_payment")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        recovery_72h=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        payment_rate_72h=("payment_event_72h", "mean")
    )
    .reset_index()
)

payment_summary["group"] = np.where(
    payment_summary["has_prior_payment"],
    "Prior payment",
    "No prior payment"
)

print("=" * 90)
print("DESCRIPTIVE SUMMARY")
print("=" * 90)

display(
    payment_summary[
        [
            "group",
            "messages",
            "customers",
            "recovery_72h",
            "recovery_per_message",
            "payment_rate_72h"
        ]
    ].style.format({
        "recovery_72h": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:.2f}",
        "payment_rate_72h": "{:.2%}"
    })
)

DPD 46–60 | PRIOR ENGAGED
PRIOR PAYMENT vs NO PRIOR PAYMENT

Total messages            : 4,402

Prior payment N           : 806
No prior payment N        : 3,596

R$/msg — Prior payment    : R$ 25.32
R$/msg — No prior payment : R$ 28.57

Difference                : R$ -3.26
Relative difference       : -11.4%

------------------------------------------------------------------------------------------

Difference:
Prior payment - No prior payment

Observed difference       : R$ -3.26
95% bootstrap CI          : R$ [-12.33, 6.36]

Permutation p-value       : 0.5942

Significant @ 5%          : NO

DESCRIPTIVE SUMMARY


,group,messages,customers,recovery_72h,recovery_per_message,payment_rate_72h
0,No prior payment,3596,2071,"R$ 102,755.49",R$ 28.57,4.20%
1,Prior payment,806,497,"R$ 20,406.73",R$ 25.32,9.31%


In [69]:
# ============================================================
# DPD 31–45 | PRIOR ENGAGED
# PAIRWISE TEMPLATE TESTS
#
# Metric:
#   Recovery / message
#
# Templates:
#   Pix
#   Urgent
#   Discount
#
# Method:
#   Two-sided permutation test
#   Bootstrap 95% CI
#   Holm correction across pairwise comparisons
#
# IMPORTANT:
#   Historical association, NOT causal superiority.
# ============================================================

import numpy as np
import pandas as pd

from itertools import combinations
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. BASE
# ============================================================

df = wa_pit.copy()

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)

df["payment_event_72h"] = (
    df["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

df["recovery_72h"] = np.where(
    df["payment_event_72h"],
    df["amount_paid_brl"],
    0
)

TEMPLATES = [
    "discount_offer",
    "pix_link",
    "urgent_reminder"
]


# ============================================================
# 2. SEGMENT
# DPD 31–45 + PRIOR ENGAGED
# ============================================================

d = df.loc[
    df["days_past_due"].between(31, 45)
    & df["has_prior_engagement"].eq(True)
    & df["template"].isin(TEMPLATES)
].copy()


print("=" * 100)
print("DPD 31–45 | PRIOR ENGAGED")
print("=" * 100)

print(f"Messages  : {len(d):,}")
print(f"Customers : {d['customer_id'].nunique():,}")


# ============================================================
# 3. DESCRIPTIVE BY TEMPLATE
# ============================================================

template_summary = (
    d.groupby("template")
    .agg(
        messages=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        total_recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        payment_rate=("payment_event_72h", "mean")
    )
    .reset_index()
    .sort_values(
        "recovery_per_message",
        ascending=False
    )
)

print("\n" + "=" * 100)
print("DESCRIPTIVE — TEMPLATE PERFORMANCE")
print("=" * 100)

display(
    template_summary.style.format({
        "total_recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:.2f}",
        "payment_rate": "{:.2%}"
    })
)


# ============================================================
# 4. PAIRWISE TESTS
# ============================================================

N_PERM = 50_000
N_BOOT = 20_000
SEED = 42

rng = np.random.default_rng(SEED)

results = []


for template_a, template_b in combinations(
    TEMPLATES,
    2
):

    # --------------------------------------------------------
    # Samples
    # --------------------------------------------------------

    a = d.loc[
        d["template"].eq(template_a),
        "recovery_72h"
    ].to_numpy()

    b = d.loc[
        d["template"].eq(template_b),
        "recovery_72h"
    ].to_numpy()


    # --------------------------------------------------------
    # Observed means
    # --------------------------------------------------------

    mean_a = a.mean()
    mean_b = b.mean()

    diff = mean_a - mean_b

    relative_diff = (
        mean_a / mean_b - 1
        if mean_b != 0
        else np.nan
    )


    # ========================================================
    # 5. TWO-SIDED PERMUTATION TEST
    # ========================================================

    pooled = np.concatenate([a, b])

    n_a = len(a)

    perm_diffs = np.empty(N_PERM)

    for i in range(N_PERM):

        shuffled = rng.permutation(pooled)

        perm_diffs[i] = (
            shuffled[:n_a].mean()
            -
            shuffled[n_a:].mean()
        )

    p_raw = (
        np.sum(
            np.abs(perm_diffs)
            >= abs(diff)
        ) + 1
    ) / (
        N_PERM + 1
    )


    # ========================================================
    # 6. BOOTSTRAP 95% CI
    # ========================================================

    boot_diffs = np.empty(N_BOOT)

    for i in range(N_BOOT):

        a_boot = rng.choice(
            a,
            size=len(a),
            replace=True
        )

        b_boot = rng.choice(
            b,
            size=len(b),
            replace=True
        )

        boot_diffs[i] = (
            a_boot.mean()
            -
            b_boot.mean()
        )

    ci_low, ci_high = np.percentile(
        boot_diffs,
        [2.5, 97.5]
    )


    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    results.append({

        "comparison":
            f"{template_a} vs {template_b}",

        "template_A":
            template_a,

        "template_B":
            template_b,

        "n_A":
            len(a),

        "n_B":
            len(b),

        "rpm_A":
            mean_a,

        "rpm_B":
            mean_b,

        "difference_A_minus_B":
            diff,

        "relative_difference":
            relative_diff,

        "ci_low":
            ci_low,

        "ci_high":
            ci_high,

        "p_raw":
            p_raw
    })


results = pd.DataFrame(results)


# ============================================================
# 7. HOLM CORRECTION
# ============================================================

reject, p_holm, _, _ = multipletests(
    results["p_raw"],
    alpha=0.05,
    method="holm"
)

results["p_holm"] = p_holm
results["significant_5pct"] = reject


# ============================================================
# 8. PRINT EACH COMPARISON
# ============================================================

print("\n")
print("=" * 100)
print("PAIRWISE TEMPLATE TESTS")
print("=" * 100)


for _, row in results.iterrows():

    print(
f"""
{row['template_A']}  vs  {row['template_B']}
----------------------------------------------------------------------
N A                   : {row['n_A']:,.0f}
N B                   : {row['n_B']:,.0f}

R$/msg A              : R$ {row['rpm_A']:,.2f}
R$/msg B              : R$ {row['rpm_B']:,.2f}

Difference A - B      : R$ {row['difference_A_minus_B']:,.2f}
Relative difference  : {row['relative_difference']:+.1%}

95% bootstrap CI      : R$ [{row['ci_low']:,.2f}, {row['ci_high']:,.2f}]

Raw p-value           : {row['p_raw']:.4f}
Holm-adjusted p       : {row['p_holm']:.4f}

Significant @ 5%      : {"YES" if row['significant_5pct'] else "NO"}
"""
    )


# ============================================================
# 9. FINAL TABLE
# ============================================================

print("=" * 100)
print("FINAL RESULTS — DPD 31–45 | PRIOR ENGAGED")
print("=" * 100)

display(
    results[
        [
            "comparison",
            "n_A",
            "n_B",
            "rpm_A",
            "rpm_B",
            "difference_A_minus_B",
            "relative_difference",
            "ci_low",
            "ci_high",
            "p_raw",
            "p_holm",
            "significant_5pct"
        ]
    ].style.format({

        "rpm_A": "R$ {:.2f}",
        "rpm_B": "R$ {:.2f}",

        "difference_A_minus_B": "R$ {:.2f}",
        "relative_difference": "{:+.1%}",

        "ci_low": "R$ {:.2f}",
        "ci_high": "R$ {:.2f}",

        "p_raw": "{:.4f}",
        "p_holm": "{:.4f}"
    })
)


DPD 31–45 | PRIOR ENGAGED
Messages  : 6,519
Customers : 3,750

DESCRIPTIVE — TEMPLATE PERFORMANCE


,template,messages,customers,total_recovery,recovery_per_message,payment_rate
0,discount_offer,3290,2469,"R$ 172,348.53",R$ 52.39,9.76%
1,pix_link,1317,1189,"R$ 57,118.32",R$ 43.37,7.97%
2,urgent_reminder,1912,1595,"R$ 61,675.44",R$ 32.26,5.18%




PAIRWISE TEMPLATE TESTS

discount_offer  vs  pix_link
----------------------------------------------------------------------
N A                   : 3,290
N B                   : 1,317

R$/msg A              : R$ 52.39
R$/msg B              : R$ 43.37

Difference A - B      : R$ 9.02
Relative difference  : +20.8%

95% bootstrap CI      : R$ [-4.30, 21.80]

Raw p-value           : 0.1779
Holm-adjusted p       : 0.1796

Significant @ 5%      : NO


discount_offer  vs  urgent_reminder
----------------------------------------------------------------------
N A                   : 3,290
N B                   : 1,912

R$/msg A              : R$ 52.39
R$/msg B              : R$ 32.26

Difference A - B      : R$ 20.13
Relative difference  : +62.4%

95% bootstrap CI      : R$ [9.74, 30.59]

Raw p-value           : 0.0002
Holm-adjusted p       : 0.0007

Significant @ 5%      : YES


pix_link  vs  urgent_reminder
----------------------------------------------------------------------
N A         

,comparison,n_A,n_B,rpm_A,rpm_B,difference_A_minus_B,relative_difference,ci_low,ci_high,p_raw,p_holm,significant_5pct
0,discount_offer vs pix_link,3290,1317,R$ 52.39,R$ 43.37,R$ 9.02,+20.8%,R$ -4.30,R$ 21.80,0.1779,0.1796,False
1,discount_offer vs urgent_reminder,3290,1912,R$ 52.39,R$ 32.26,R$ 20.13,+62.4%,R$ 9.74,R$ 30.59,0.0002,0.0007,True
2,pix_link vs urgent_reminder,1317,1912,R$ 43.37,R$ 32.26,R$ 11.11,+34.5%,R$ -2.12,R$ 24.48,0.0898,0.1796,False
